In [ ]:
/**
 * Startup Interview Pro-Tip:
 * - In sabme Edge Cases (Sparse Arrays, new Keyword) ko skip kar do.
 * - Bas itna dhyan rakho:
 *   - Callback function hai ya nhi check karo  (typeof callback !== 'function').
 *   - this aur arguments pass karo (func.apply(context, args)).
 * - Agar ye 10 polyfills (Tier 1 + Tier 2) smoothly likh sakte ho, toh kisi v
 *   startup ka JS round clear ho jaega. 
 * 
 * Tier 1 — Pakka Pucha Jayega (100% Chance)
 * 1. Array.prototype.map (Basic loop + new array)
 * 2. Array.prototype.filter (Condition check + new array)
 * 3. Array.prototype.reduce (Accumulator + sum/object)
 * 4. Function.prototype.bind (Basic this binding)
 * 5. Debounce (Search box use-case ke saath)
 * 6. Throttle (Scroll/Click use-case ke saath)
 * 
 * Tier 2 — Aksar Pucha Jayega (60% Chance)
 * 7. Array.prototype.forEach (Simple loop)
 * 8. Array.prototype.find (Pehla match return karna)
 * 9. Promise (Basic) (States + .then() + chaining)
 * 10. Promise.all (Parallel API calls ka use-case)
*/

In [ ]:
/**
 * Array Polyfills:
 * 1. Array.prototype.map 
 * 2. Array.prototype.forEach
 * 3. Array.prototype.filter
 * 4. Array.prototype.reduce
 * 5. Array.prototype.reduceRight
 * 6. Array.prototype.concat
 * 7. Array.prototype.fill
 * 8. Array.prototype.find
 * 9. Array.prototype.flat
 * 10. Array.prototype.flatMap
 * 11. Array.prototype.includes
 * 12. Array.prototype.join
 * 13. Array.prototype.reverse
 * 14. Array.prototype.split
*/

In [ ]:
/**
 * Map Polyfill:
 */

/**
 * Interviewer: Can you write a polyfill for Array.prototype.map?
 *
 * You: Sure. Let me explain how map() works first.
 * - map() creates a NEW array.
 * - It runs a callback function on every element.
 * - Whatever the callback returns gets pushed into the new array.
 * - The original array is NOT changed.
 *
 * The callback receives 3 arguments:
 * 1. currentValue      : the current element
 * 2. index  (optional) : position of the current element
 * 3. array  (optional) : the original array map() was called on
 *
 * map() can also take a second argument called 'thisArg',
 * which becomes 'this' inside the callback function.
 *
 * Syntax: array.map(callback, thisArg)
 */

/* Step 1: Basic version — just the happy path, no edge cases yet */
Array.prototype.myMap = function (callback) {
    const result = [];

    for (let i = 0; i < this.length; i++) {
        result.push(callback(this[i], i, this));
    }

    return result;
};

/* Test 1: Basic doubling */
const numbers = [1, 2, 3, 4, 5];
const doubled = numbers.myMap((num) => num * 2);
console.log(doubled); // [2, 4, 6, 8, 10] ✅

/* Test 2: Using the index argument */
const withIndex = numbers.myMap((num, idx) => num + idx);
console.log(withIndex); // [1, 3, 5, 7, 9] ✅

In [ ]:
/**
 * What's Failing, and Why?
 */

/**
 * The basic polyfill works for simple cases, but it breaks in 3 situations. 
 * Let's look at each problem one by one.
 */

/**
 * FAILURE 1 — Callback nahi diya gaya
 * - Agar koi [1, 2, 3].myMap() bina argument ke call kare, toh 'callback' undefined
 *   ban jata hai.
 * - undefined() ko call karna allowed nahi — JS throw karta hai:
 *   TypeError: callback is not a function
 */

const failed1 = [1, 2, 3].myMap();  /* TypeError: callback is not a function */

/**
 * FAILURE 2 — thisArg ignore ho raha hai
 *
 * Pehle samajhte hain thisArg kya hai:
 * - thisArg ek second argument hai jo map() ko pass hota hai.
 * - Iska kaam hai: callback ke ANDAR 'this' ka value set karna.
 * - Matlab callback ke andar jab hum 'this' likhte hain, toh woh thisArg object ko
 *   point karega.
 *
 * Example:
 *   const obj = { multiplier: 10 };
 *   [1, 2, 3].myMap(function (num) {
 *       return num * this.multiplier;  // yahan 'this' = obj hona chahiye
 *   }, obj);
 *
 * - Hamara polyfill thisArg ko accept hi nahi karta!
 * - Toh 'this' ka default value lagta hai (undefined ya global).
 * - Iska matlab this.multiplier undefined hai.
 * - Aur num * undefined = NaN aata hai.
 * - Result: [NaN, NaN, NaN] instead of [10, 20, 30]
 */

const obj = { multiplier: 10 };

const failed2 = [1, 2, 3].myMap(function (num) {
    return num * this.multiplier;
}, obj);

console.log(failed2); // [NaN, NaN, NaN]

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays can have "holes":
 *     const sparse = [1, , 3];
 * - Index 1 here is a HOLE — it doesn't exist, it's not just undefined.
 *
 * - Native map() SKIPS holes — it never calls the callback for them.
 *   Result: [2, empty, 6]
 *
 * - Our polyfill blindly loops from 0 to length-1, with no hole check.
 * - At i = 1, sparse[1] reads as undefined, the callback still runs,
 *   and undefined * 2 = NaN gets pushed in.
 *
 * Result: [2, NaN, 6] instead of the expected [2, empty, 6]
 */

/* Native map() gives [2, empty, 6] — it skips holes */
const sparse = [1, , 3];

const failed3 = sparse.myMap(function (x) { return x * 2; });
console.log(failed3); // [2, NaN, 6]


In [ ]:
/**
 * Edge Cases 1 & 2 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar callback function nahi hai toh?
 * - Agar koi myMap() bina argument ke call kare, ya null ya number de,
 *   toh code unexpectedly crash ho jayega.
 * - Native map() clear TypeError throw karta hai, humein bhi karna chahiye.
 *
 * Interviewer: thisArg ka issue?
 *
 * You: Haan, basic version second argument ignore karta hai.
 * - Mujhe thisArg ko callback ke andar 'this' banana hai.
 * - Iske liye callback.call(thisArg, ...) use karunga,
 *   direct call ki jagah.
 *
 * callback.call(thisArg, arg1, arg2, arg3) do kaam ek saath karta hai:
 * 1. Callback ke andar 'this' ko thisArg set karta hai.
 * 2. arg1, arg2, arg3 ko callback ke arguments banata hai.
 */

/* Step 2: Type check + thisArg binding fix */
Array.prototype.myMap = function (callback, thisArg) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    const result = [];
    for (let i = 0; i < this.length; i++) {
        /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
        result.push(callback.call(thisArg, this[i], i, this));
    }

    return result;
};

/* Test 1: No callback — TypeError aana chahiye */
try {
    [1, 2, 3].myMap();
} catch (e) {
    console.log(e.message); // ✅ "undefined is not a function"
}

/* Test 2: null as callback — TypeError aana chahiye */
try {
    [1, 2, 3].myMap(null);
} catch (e) {
    console.log(e.message); // ✅ "null is not a function"
}

/* Test 3: thisArg ab sahi kaam karta hai */
const obj2 = { multiplier: 10 };
const fixedResult = [1, 2, 3].myMap(function (num) {
    return num * this.multiplier;
}, obj2);
console.log(fixedResult); // ✅ [10, 20, 30]


In [ ]:
/**
 * What's Still Failing, and Why?
 */

/**
 * We've fixed 2 edge cases so far:
 * a. Type check — non-function callbacks ab TypeError dete hain.
 * b. thisArg — 'this' ab sahi se callback ke andar bind hota hai.
 *               Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays are still broken
 * - The loop still blindly runs from 0 to length-1, with no hole check.
 *
 * [1, , 3] ke liye:
 *   i = 0 → this[0] = 1     → callback chalta → 2 push          ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → callback undefined ke saath phir bhi chalta hai
 *           → undefined * 2 = NaN → NaN push                   ❌
 *   i = 2 → this[2] = 3     → callback chalta → 6 push          ✅
 *
 * Result:   [2, NaN, 6]
 * Expected: [2, empty, 6]
 *
 * Problem: 
 * - Hum index 1 pe callback call kar rahe hain, jabki woh hole hai. 
 * - Native map() kabhi nahi karta.
 *
 * Fix: 
 * - 'i in this' check use karo — yeh batata hai ki index i array mein ACTUALLY exist
 *   karta hai ya nahi.
 * - Holes ke liye false return karta hai.
 */

const sparse2 = [1, , 3];
const stillBroken = sparse2.myMap((x) => x * 2);
console.log(stillBroken); // [2, NaN, 6] — native map() gives [2, empty, 6]

In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native map() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myMap = function (callback, thisArg) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    const result = [];

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
            result.push(callback.call(thisArg, this[i], i, this));
        }
    }

    return result;
};

/* Test 1: Sparse array — callback hole ko skip karega */
const sparse3 = [1, , 3];
const sparseResult = sparse3.myMap((x) => x * 2);
console.log(sparseResult); // ✅ [2, empty, 6]

/* Test 2: Normal (dense) array abhi bhi theek chalega */
const dense = [1, 2, 3];
const denseResult = dense.myMap((x) => x * 2);
console.log(denseResult); // ✅ [2, 4, 6]

/* Test 3: thisArg abhi bhi kaam karta hai */
const obj3 = { multiplier: 10 };
const thisArgResult = [1, 2, 3].myMap(function (num) {
    return num * this.multiplier;
}, obj3);
console.log(thisArgResult); // ✅ [10, 20, 30]

/* Test 4: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myMap("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 5: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.myMap((x) => x * 100);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * Final Code:
 * 1. Type check — non-function callback pe TypeError throw hota hai
 * 2. thisArg — 'this' sahi se callback ke andar bind hota hai
 * 3. Sparse arrays — holes skip hote hain, callback call nahi hota
 */

Array.prototype.myMap = function (callback, thisArg) {
  if (typeof callback !== 'function') {
    throw new TypeError(callback + ' is not a function');
  }

  const result = [];

  for (let i = 0; i < this.length; i++) {
    if (i in this) {
      result.push(callback.call(thisArg, this[i], i, this));
    }
  }

  return result;
};

In [ ]:
/**
 * forEach Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.forEach ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain forEach() kaise kaam karta hai:
 * - forEach() har element pe ek function chalata hai.
 * - Yeh NAYA array return NAHI karta (map se yahi difference hai).
 * - Iska kaam hai side effects ke liye — jaise:
 *   - console.log karna
 *   - kisi variable ko update karna
 *   - API call karna
 * - Original array automatically change nahi hota.
 *
 * Callback function ko 3 cheezein milti hain:
 * 1. currentValue  : current element
 * 2. index (optional) : element ka position
 * 3. array (optional) : original array jis pe forEach() call hua
 *
 * forEach() ka second argument hota hai 'thisArg' —
 * yeh callback ke andar 'this' ko control karta hai.
 *
 * Syntax: array.forEach(callback, thisArg)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.myForEach = function (callback) {
    for (let i = 0; i < this.length; i++) {
        callback(this[i], i, this);
    }
};

/* Test 1: Named function se use karna */
const numbers = [1, 2, 3, 4, 5];

const printNums = function (number, index) {
    console.log(`Element ${number} is at index ${index}`);
};

numbers.myForEach(printNums);

// Output:
// Element 1 is at index 0
// Element 2 is at index 1
// Element 3 is at index 2
// Element 4 is at index 3
// Element 5 is at index 4

/* Test 2: Arrow function se use karna */
const sum = { total: 0 };
[1, 2, 3, 4, 5].myForEach((num) => {
    sum.total += num;
});
console.log(sum.total); // 15 ✅

In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 3 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Callback nahi diya gaya
 * - Agar koi [1, 2, 3].myForEach() bina argument ke call kare,
 *   toh 'callback' undefined ban jata hai.
 * - undefined() ko call karna allowed nahi — JS throw karta hai:
 *   TypeError: callback is not a function
 */

const failed1 = [1, 2, 3].myForEach();  /* TypeError: callback is not a function */

/**
 * FAILURE 2 — thisArg ignore ho raha hai
 *
 * Pehle samajhte hain thisArg kya hai:
 * - thisArg ek second argument hai jo forEach() ko pass hota hai.
 * - Iska kaam: callback ke ANDAR 'this' ka value set karna.
 * - Matlab callback ke andar jab hum 'this' likhte hain,
 *   toh woh thisArg object ko point karega.
 *
 * Example:
 *   const obj = { multiplier: 10, result: [] };
 *   [1, 2, 3].myForEach(function (num) {
 *       this.result.push(num * this.multiplier);  // yahan 'this' = obj hona chahiye
 *   }, obj);
 *
 * - Hamara polyfill thisArg ko accept hi nahi karta!
 * - Toh 'this' ka default value lagta hai (undefined ya global).
 * - Iska matlab this.multiplier aur this.result dono undefined hain.
 * - .push() undefined pe call karna → TypeError.
 * - Result: crash ho jata hai.
 */

const obj = { multiplier: 10, result: [] };

try {
    [1, 2, 3].myForEach(function (num) {
        this.result.push(num * this.multiplier);
    }, obj);
} catch (e) {
    console.log(e.message); // ❌ crash — this.result is undefined
}

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta,
 *   undefined value nahi hai.
 *
 * - Native forEach() holes ko SKIP karta hai — callback unke liye nahi chalta.
 *
 * - Hamara polyfill blindly 0 se length-1 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, sparse[1] undefined padta hai, callback phir bhi chalta hai.
 * - Yeh galat behaviour hai — native forEach() kabhi hole pe callback nahi chalata.
 *
 * Result: callback hole ke liye bhi chal gaya — galat!
 */

/* Native forEach() skips holes — callback call nahi karta */
const sparse = [1, , 3];

sparse.myForEach((x) => console.log(x));
// Humein sirf 1 aur 3 print hone chahiye,
// lekin hamara polyfill 1, undefined, 3 print karega. ❌


In [ ]:
/**
 * Edge Cases 1 & 2 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar callback function nahi hai toh?
 * - Agar koi myForEach() bina argument ke call kare, ya null ya number de,
 *   toh code unexpectedly crash ho jayega.
 * - Native forEach() clear TypeError throw karta hai, humein bhi karna chahiye.
 *
 * Interviewer: thisArg ka issue?
 *
 * You: Haan, basic version second argument ignore karta hai.
 * - Mujhe thisArg ko callback ke andar 'this' banana hai.
 * - Iske liye callback.call(thisArg, ...) use karunga,
 *   direct call ki jagah.
 *
 * callback.call(thisArg, arg1, arg2, arg3) do kaam ek saath karta hai:
 * 1. Callback ke andar 'this' ko thisArg set karta hai.
 * 2. arg1, arg2, arg3 ko callback ke arguments banata hai.
 */

/* Step 2: Type check + thisArg binding fix */
Array.prototype.myForEach = function (callback, thisArg) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
        callback.call(thisArg, this[i], i, this);
    }
};

/* Test 1: No callback — TypeError aana chahiye */
try {
    [1, 2, 3].myForEach();
} catch (e) {
    console.log(e.message); // ✅ "undefined is not a function"
}

/* Test 2: null as callback — TypeError aana chahiye */
try {
    [1, 2, 3].myForEach(null);
} catch (e) {
    console.log(e.message); // ✅ "null is not a function"
}

/* Test 3: thisArg ab sahi kaam karta hai */
const obj2 = { multiplier: 10, result: [] };
[1, 2, 3].myForEach(function (num) {
    this.result.push(num * this.multiplier);  // 'this' = obj2
}, obj2);
console.log(obj2.result); // ✅ [10, 20, 30]


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 2 edge cases fix ho gaye:
 * a. Type check — non-function callbacks ab TypeError dete hain.
 * b. thisArg — 'this' ab sahi se callback ke andar bind hota hai.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays abhi bhi broken
 * - Loop abhi bhi blindly 0 se length-1 tak chalta hai, koi hole check nahi.
 *
 * [1, , 3] ke liye:
 *   i = 0 → this[0] = 1     → callback chalta → 1 print hota   ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → callback undefined ke saath phir bhi chalta hai  ❌
 *   i = 2 → this[2] = 3     → callback chalta → 3 print hota    ✅
 *
 * Result:   1, undefined, 3 print hota hai
 * Expected: sirf 1, 3 print hona chahiye (hole skip hona chahiye)
 *
 * Problem: Hum index 1 pe callback call kar rahe hain,
 * jabki woh hole hai. Native forEach() kabhi nahi karta.
 *
 * Fix: 'i in this' check use karo — yeh batata hai ki
 * index i array mein ACTUALLY exist karta hai ya nahi.
 * Holes ke liye false return karta hai.
 */

const sparse2 = [1, , 3];
sparse2.myForEach((x) => console.log(x));
// 1, undefined, 3 — native forEach() sirf 1, 3 deta ❌


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native forEach() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myForEach = function (callback, thisArg) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
            callback.call(thisArg, this[i], i, this);
        }
    }
};

/* Test 1: Sparse array — callback hole ko skip karega */
const sparse3 = [1, , 3];
const printed = [];
sparse3.myForEach((x) => printed.push(x));
console.log(printed); // ✅ [1, 3] — hole skip ho gaya

/* Test 2: Normal (dense) array abhi bhi theek chalega */
const dense = [1, 2, 3];
dense.myForEach((x) => console.log(x)); // ✅ 1, 2, 3

/* Test 3: thisArg abhi bhi kaam karta hai */
const obj3 = { multiplier: 10, result: [] };
[1, 2, 3].myForEach(function (num) {
    this.result.push(num * this.multiplier);
}, obj3);
console.log(obj3.result); // ✅ [10, 20, 30]

/* Test 4: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myForEach("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 5: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.myForEach((x) => x * 100);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native forEach() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myForEach = function (callback, thisArg) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
            callback.call(thisArg, this[i], i, this);
        }
    }
};

/* Test 1: Sparse array — callback hole ko skip karega */
const sparse3 = [1, , 3];
const printed = [];
sparse3.myForEach((x) => printed.push(x));
console.log(printed); // ✅ [1, 3] — hole skip ho gaya

/* Test 2: Normal (dense) array abhi bhi theek chalega */
const dense = [1, 2, 3];
dense.myForEach((x) => console.log(x)); // ✅ 1, 2, 3

/* Test 3: thisArg abhi bhi kaam karta hai */
const obj3 = { multiplier: 10, result: [] };
[1, 2, 3].myForEach(function (num) {
    this.result.push(num * this.multiplier);
}, obj3);
console.log(obj3.result); // ✅ [10, 20, 30]

/* Test 4: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myForEach("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 5: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.myForEach((x) => x * 100);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * Final Code:
 * 1. Type check — non-function callback pe TypeError throw hota hai
 * 2. thisArg — 'this' sahi se callback ke andar bind hota hai
 * 3. Sparse arrays — holes skip hote hain, callback call nahi hota
 *
 * forEach map jaisa hi hai, bas yeh new array NAHI banata.
 * Yeh sirf callback chalata hai aur undefined return karta hai.
 */

Array.prototype.myForEach = function (callback, thisArg) {
  if (typeof callback !== 'function') {
    throw new TypeError(callback + ' is not a function');
  }

  for (let i = 0; i < this.length; i++) {
    if (i in this) {
      callback.call(thisArg, this[i], i, this);
    }
  }
};

In [ ]:
/**
 * Filter Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.filter ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain filter() kaise kaam karta hai:
 * - filter() ek NAYA array banata hai (original array change nahi hota).
 * - Har element pe callback function chalata hai.
 * - Agar callback TRUE return karta hai → element naye array mein aata hai.
 * - Agar callback FALSE return karta hai → element skip ho jata hai.
 *
 * Use cases:
 * - unwanted data hatana
 * - specific elements nikalna (even, odd, active users, etc.)
 *
 * Callback function ko 3 cheezein milti hain:
 * 1. currentValue  : current element
 * 2. index (optional) : element ka position
 * 3. array (optional) : original array jis pe filter() call hua
 *
 * filter() ka second argument hota hai 'thisArg' —
 * yeh callback ke andar 'this' ko control karta hai.
 *
 * Syntax: array.filter(callback, thisArg)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.myFilter = function (callback) {
    const result = [];

    for (let i = 0; i < this.length; i++) {
        /* Agar callback true return kare, toh element ko naye array mein daalo */
        if (callback(this[i], i, this)) {
            result.push(this[i]);
        }
    }

    return result;
};

/* Test 1: Even numbers nikalna */
const numbers = [1, 2, 3, 4, 5];

const isEven = function (number) {
    return number % 2 === 0;
};

const evenNumbers = numbers.myFilter(isEven);
console.log(evenNumbers); // ✅ [2, 4]

/* Test 2: Index argument ka use */
const greaterThanIndex = numbers.myFilter((num, idx) => num > idx);
console.log(greaterThanIndex); // ✅ [1, 2, 3, 4, 5]

/**
 * forEach vs map vs filter ka difference:
 *
 * forEach() → sirf loop chalata hai, kuch return nahi karta
 * map()     → har element ko transform karta hai, naya array deta hai
 * filter()  → unwanted elements hatata hai, naya array deta hai
 *
 * filter() = "jo chahiye sirf wahi rakh lo"
 */

In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 3 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Callback nahi diya gaya
 * - Agar koi [1, 2, 3].myFilter() bina argument ke call kare,
 *   toh 'callback' undefined ban jata hai.
 * - undefined() ko call karna allowed nahi — JS throw karta hai:
 *   TypeError: callback is not a function
 */

const failed1 = [1, 2, 3].myFilter();  /* TypeError: callback is not a function */

/**
 * FAILURE 2 — thisArg ignore ho raha hai
 *
 * Pehle samajhte hain thisArg kya hai:
 * - thisArg ek second argument hai jo filter() ko pass hota hai.
 * - Iska kaam: callback ke ANDAR 'this' ka value set karna.
 * - Matlab callback ke andar jab hum 'this' likhte hain,
 *   toh woh thisArg object ko point karega.
 *
 * Example:
 *   const obj = { min: 3 };
 *   [1, 2, 3, 4, 5].myFilter(function (num) {
 *       return num > this.min;  // yahan 'this' = obj hona chahiye
 *   }, obj);
 *
 * - Hamara polyfill thisArg ko accept hi nahi karta!
 * - Toh 'this' ka default value lagta hai (undefined ya global).
 * - Iska matlab this.min undefined hai.
 * - num > undefined → hamesha false aata hai.
 * - Result: [] (empty array) instead of [4, 5]
 */

const obj = { min: 3 };

const failed2 = [1, 2, 3, 4, 5].myFilter(function (num) {
    return num > this.min;
}, obj);

console.log(failed2); // ❌ [] — should be [4, 5]

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta,
 *   undefined value nahi hai.
 *
 * - Native filter() holes ko SKIP karta hai — callback unke liye nahi chalta.
 *   Result: [2, 6] (hole copy nahi hota naye array mein)
 *
 * - Hamara polyfill blindly 0 se length-1 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, sparse[1] undefined padta hai, callback phir bhi chalta hai.
 * - Agar callback undefined ke liye true return kare, toh undefined
 *   naye array mein aa jata hai — yeh galat hai.
 *
 * Result: [1, undefined, 3] instead of expected [1, 3]
 */

/* Native filter() gives [2, 6] — hole skip karta hai */
const sparse = [1, , 3];

const failed3 = sparse.myFilter((x) => true);  /* sab keep karo */
console.log(failed3); // ❌ [1, undefined, 3] — hole bhi aa gaya

In [ ]:
/**
 * Edge Cases 1 & 2 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar callback function nahi hai toh?
 * - Agar koi myFilter() bina argument ke call kare, ya null ya number de,
 *   toh code unexpectedly crash ho jayega.
 * - Native filter() clear TypeError throw karta hai, humein bhi karna chahiye.
 *
 * Interviewer: thisArg ka issue?
 *
 * You: Haan, basic version second argument ignore karta hai.
 * - Mujhe thisArg ko callback ke andar 'this' banana hai.
 * - Iske liye callback.call(thisArg, ...) use karunga,
 *   direct call ki jagah.
 *
 * callback.call(thisArg, arg1, arg2, arg3) do kaam ek saath karta hai:
 * 1. Callback ke andar 'this' ko thisArg set karta hai.
 * 2. arg1, arg2, arg3 ko callback ke arguments banata hai.
 */

/* Step 2: Type check + thisArg binding fix */
Array.prototype.myFilter = function (callback, thisArg) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    const result = [];

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
        if (callback.call(thisArg, this[i], i, this)) {
            result.push(this[i]);
        }
    }

    return result;
};

/* Test 1: No callback — TypeError aana chahiye */
try {
    [1, 2, 3].myFilter();
} catch (e) {
    console.log(e.message); // ✅ "undefined is not a function"
}

/* Test 2: null as callback — TypeError aana chahiye */
try {
    [1, 2, 3].myFilter(null);
} catch (e) {
    console.log(e.message); // ✅ "null is not a function"
}

/* Test 3: thisArg ab sahi kaam karta hai */
const obj2 = { min: 3 };
const fixedResult = [1, 2, 3, 4, 5].myFilter(function (num) {
    return num > this.min;  // 'this' = obj2, this.min = 3
}, obj2);
console.log(fixedResult); // ✅ [4, 5]


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 2 edge cases fix ho gaye:
 * a. Type check — non-function callbacks ab TypeError dete hain.
 * b. thisArg — 'this' ab sahi se callback ke andar bind hota hai.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays abhi bhi broken
 * - Loop abhi bhi blindly 0 se length-1 tak chalta hai, koi hole check nahi.
 *
 * [1, , 3] ke liye filter(x => true):
 *   i = 0 → this[0] = 1     → callback chalta → true → 1 push          ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → callback undefined ke saath phir bhi chalta hai
 *           → true return karta hai → undefined push                   ❌
 *   i = 2 → this[2] = 3     → callback chalta → true → 3 push          ✅
 *
 * Result:   [1, undefined, 3]
 * Expected: [1, 3] — hole skip hona chahiye
 *
 * Problem: Hum index 1 pe callback call kar rahe hain,
 * jabki woh hole hai. Native filter() kabhi nahi karta.
 *
 * Fix: 'i in this' check use karo — yeh batata hai ki
 * index i array mein ACTUALLY exist karta hai ya nahi.
 * Holes ke liye false return karta hai.
 */

const sparse2 = [1, , 3];
const stillBroken = sparse2.myFilter((x) => true);
console.log(stillBroken); // ❌ [1, undefined, 3] — native filter() gives [1, 3]

In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native filter() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta, aur holes naye array mein copy bhi nahi hote.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myFilter = function (callback, thisArg) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    const result = [];

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
            if (callback.call(thisArg, this[i], i, this)) {
                result.push(this[i]);
            }
        }
    }

    return result;
};

/* Test 1: Sparse array — hole skip hoga, naye array mein nahi aayega */
const sparse3 = [1, , 3];
const sparseResult = sparse3.myFilter((x) => true);  /* sab keep karo */
console.log(sparseResult); // ✅ [1, 3] — hole skip ho gaya

/* Test 2: Normal (dense) array abhi bhi theek chalega */
const dense = [1, 2, 3, 4, 5];
const denseResult = dense.myFilter((x) => x % 2 === 0);
console.log(denseResult); // ✅ [2, 4]

/* Test 3: thisArg abhi bhi kaam karta hai */
const obj3 = { min: 3 };
const thisArgResult = [1, 2, 3, 4, 5].myFilter(function (num) {
    return num > this.min;
}, obj3);
console.log(thisArgResult); // ✅ [4, 5]

/* Test 4: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myFilter("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 5: Original array MUTATE nahi hota */
const original = [1, 2, 3, 4, 5];
original.myFilter((x) => x > 2);
console.log(original); // ✅ [1, 2, 3, 4, 5] — unchanged


In [ ]:
/**
 * Final Code:
 * 1. Type check — non-function callback pe TypeError throw hota hai
 * 2. thisArg — 'this' sahi se callback ke andar bind hota hai
 * 3. Sparse arrays — holes skip hote hain, callback call nahi hota
 *
 * filter() map jaisa hi hai, bas map transform karta hai,
 * filter condition check karke sirf matching elements rakhtha hai.
 */

Array.prototype.myFilter = function (callback, thisArg) {
  if (typeof callback !== 'function') {
    throw new TypeError(callback + ' is not a function');
  }

  const result = [];

  for (let i = 0; i < this.length; i++) {
    if (i in this) {
      if (callback.call(thisArg, this[i], i, this)) {
        result.push(this[i]);
      }
    }
  }

  return result;
};

In [ ]:
/**
 * Reduce Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.reduce ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain reduce() kaise kaam karta hai:
 * - reduce() array ke saare elements ko combine karke EK single value deta hai.
 * - Yeh left se right chalta hai (pehla element se last tak).
 * - Simple words mein: "many → one"
 *
 * Use cases:
 * - numbers ka sum nikalna
 * - total price calculate karna
 * - items count karna
 * - objects ya arrays build karna
 *
 * Callback function ko 4 cheezein milti hain:
 * 1. accumulator   : pichle step ka result (yeh value step-by-step grow karti hai)
 * 2. currentValue  : current element
 * 3. index (optional) : element ka position
 * 4. array (optional) : original array jis pe reduce() call hua
 *
 * Very important rule:
 * - Callback jo bhi return karta hai, woh NEXT accumulator ban jata hai.
 *
 * reduce() ka second argument hota hai 'initialValue' —
 * yeh accumulator ka starting value set karta hai.
 *
 * Syntax: array.reduce(callback, initialValue)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.myReduce = function (callback, initialValue) {
    let accumulator = initialValue;

    for (let i = 0; i < this.length; i++) {
        /* Agar initialValue nahi diya gaya, toh first element ko accumulator banao */
        if (accumulator === undefined) {
            accumulator = this[i];
        } else {
            accumulator = callback(accumulator, this[i], i, this);
        }
    }

    return accumulator;
};

/* Test 1: Sum of numbers */
const numbers = [1, 2, 3, 4, 5];

const sumFn = function (accumulator, currentValue) {
    return accumulator + currentValue;
};

const total = numbers.myReduce(sumFn, 0);
console.log(total); // ✅ 15

/**
 * Step-by-step flow:
 *
 * Initial accumulator = 0 (initialValue)
 *
 * Step 1: 0 + 1 = 1
 * Step 2: 1 + 2 = 3
 * Step 3: 3 + 3 = 6
 * Step 4: 6 + 4 = 10
 * Step 5: 10 + 5 = 15
 *
 * Final result → 15
 */

/* Test 2: Bina initialValue ke */
const total2 = numbers.myReduce((acc, curr) => acc + curr);
console.log(total2); // ✅ 15

/**
 * Bina initialValue ke flow:
 * - accumulator = this[0] = 1 (first element)
 * - Loop index 1 se start hota hai (second element se)
 *
 * Step 1: 1 + 2 = 3
 * Step 2: 3 + 3 = 6
 * Step 3: 6 + 4 = 10
 * Step 4: 10 + 5 = 15
 */

/**
 * forEach vs map vs filter vs reduce ka difference:
 *
 * forEach() → sirf loop chalata hai, kuch return nahi karta
 * map()     → har element ko transform karta hai, naya array deta hai
 * filter()  → unwanted elements hatata hai, naya array deta hai
 * reduce()  → saare elements ko combine karke EK value deta hai
 *
 * reduce() = "many → one"
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 4 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Callback nahi diya gaya
 * - Agar koi [1, 2, 3].myReduce() bina argument ke call kare,
 *   toh 'callback' undefined ban jata hai.
 * - undefined() ko call karna allowed nahi — JS throw karta hai:
 *   TypeError: callback is not a function
 */

const failed1 = [1, 2, 3].myReduce();  /* TypeError: callback is not a function */

/**
 * FAILURE 2 — Empty array + no initialValue
 * - Agar array EMPTY hai aur initialValue NAHI di gayi:
 *     [].myReduce((acc, curr) => acc + curr)
 * - Hamara polyfill:
 *   → accumulator = undefined (initialValue nahi hai)
 *   → Loop nahi chalega (length = 0)
 *   → return undefined ❌
 *
 * - Native reduce() is case mein TypeError throw karta hai:
 *   "Reduce of empty array with no initial value"
 * - Yeh important hai kyunki undefined silently return karna
 *   galat hai — user ko pata hi nahi chalega ki kuch galat hua.
 */

const failed2 = [].myReduce((acc, curr) => acc + curr);
console.log(failed2); // ❌ undefined — native reduce() throws TypeError

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta.
 *
 * - Native reduce() holes ko SKIP karta hai — callback unke liye nahi chalta.
 *
 * - Hamara polyfill blindly 0 se length-1 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, this[1] undefined padta hai, callback phir bhi chalta hai.
 * - Isse accumulator undefined ke saath combine ho jata hai → NaN.
 *
 * Result: NaN instead of expected 4 (1 + 3)
 */

const sparse = [1, , 3];
const failed3 = sparse.myReduce((acc, curr) => acc + curr, 0);
console.log(failed3); // ❌ NaN — native reduce() gives 4

/**
 * FAILURE 4 — initialValue ka check ghalat hai
 * - Humne `accumulator === undefined` check kiya hai.
 * - Lekin agar user JAAN BOOJHKE undefined as initialValue pass kare:
 *     [1, 2, 3].myReduce((acc, curr) => acc + curr, undefined)
 *   Toh hamara code sochega initialValue nahi di gayi,
 *   aur first element ko accumulator bana dega.
 * - Native reduce() mein undefined ek VALID initialValue hai.
 *
 * - Fix: humein check karna hoga ki initialValue argument
 *   ACTUALLY pass hua ya nahi — `arguments.length` se pata chalta hai.
 */


In [ ]:
/**
 * Edge Cases 1 & 2 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar callback function nahi hai toh?
 * - Agar koi myReduce() bina argument ke call kare, ya null ya number de,
 *   toh code unexpectedly crash ho jayega.
 * - Native reduce() clear TypeError throw karta hai, humein bhi karna chahiye.
 *
 * Interviewer: Empty array + no initialValue ka issue?
 *
 * You: Haan, agar array empty hai aur initialValue nahi di gayi,
 *   toh humein TypeError throw karna chahiye — undefined silently
 *   return karna galat hai.
 * - Check: agar array ki length 0 hai AUR initialValue nahi di gayi,
 *   toh throw TypeError.
 *
 * initialValue actually pass hua ya nahi — yeh check karne ke liye
 * `arguments.length` use karenge.
 * - arguments.length < 2 ka matlab: sirf callback mila, initialValue nahi.
 * - arguments.length >= 2 ka matlab: dono mile, chahe initialValue undefined ho.
 */

/* Step 2: Type check + empty array check + proper initialValue check */
Array.prototype.myReduce = function (callback, initialValue) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    /* Edge Case 2: empty array + no initialValue → TypeError */
    if (this.length === 0 && arguments.length < 2) {
        throw new TypeError('Reduce of empty array with no initial value');
    }

    /* Edge Case 4: check karo ki initialValue actually pass hua ya nahi */
    let accumulator;
    let startIndex = 0;

    if (arguments.length >= 2) {
        /* initialValue diya gaya — chahe woh undefined ho */
        accumulator = initialValue;
    } else {
        /* initialValue nahi diya — first element ko accumulator banao */
        accumulator = this[0];
        startIndex = 1;
    }

    for (let i = startIndex; i < this.length; i++) {
        accumulator = callback(accumulator, this[i], i, this);
    }

    return accumulator;
};

/* Test 1: No callback — TypeError aana chahiye */
try {
    [1, 2, 3].myReduce();
} catch (e) {
    console.log(e.message); // ✅ "undefined is not a function"
}

/* Test 2: null as callback — TypeError aana chahiye */
try {
    [1, 2, 3].myReduce(null);
} catch (e) {
    console.log(e.message); // ✅ "null is not a function"
}

/* Test 3: Empty array + no initialValue — TypeError aana chahiye */
try {
    [].myReduce((acc, curr) => acc + curr);
} catch (e) {
    console.log(e.message); // ✅ "Reduce of empty array with no initial value"
}

/* Test 4: Empty array + initialValue — kaam karega */
const emptyResult = [].myReduce((acc, curr) => acc + curr, 0);
console.log(emptyResult); // ✅ 0

/* Test 5: undefined as initialValue — ab sahi kaam karega */
const undefinedInit = [1, 2, 3].myReduce((acc, curr) => {
    if (acc === undefined) return curr;
    return acc + curr;
}, undefined);
console.log(undefinedInit); // ✅ 6 (1 + 2 + 3)


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 3 edge cases fix ho gaye:
 * a. Type check — non-function callbacks ab TypeError dete hain.
 * b. Empty array + no initialValue — ab clear TypeError deta hai.
 * c. Proper initialValue check — undefined as initialValue ab sahi handle hota hai.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays abhi bhi broken
 * - Loop abhi bhi blindly startIndex se length-1 tak chalta hai,
 *   koi hole check nahi karta.
 *
 * [1, , 3] ke liye reduce((acc, curr) => acc + curr, 0):
 *   i = 0 → this[0] = 1     → callback chalta → 0 + 1 = 1   ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → callback undefined ke saath phir bhi chalta hai
 *           → 1 + undefined = NaN                            ❌
 *   i = 2 → this[2] = 3     → callback chalta → NaN + 3 = NaN ✅(broken)
 *
 * Result:   NaN
 * Expected: 4 (0 + 1 + 3 — hole skip hona chahiye)
 *
 * Problem: Hum index 1 pe callback call kar rahe hain,
 * jabki woh hole hai. Native reduce() kabhi nahi karta.
 *
 * Fix: 'i in this' check use karo — yeh batata hai ki
 * index i array mein ACTUALLY exist karta hai ya nahi.
 * Holes ke liye false return karta hai.
 */

const sparse2 = [1, , 3];
const stillBroken = sparse2.myReduce((acc, curr) => acc + curr, 0);
console.log(stillBroken); // ❌ NaN — native reduce() gives 4


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native reduce() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myReduce = function (callback, initialValue) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    /* Edge Case 2: empty array + no initialValue → TypeError */
    if (this.length === 0 && arguments.length < 2) {
        throw new TypeError('Reduce of empty array with no initial value');
    }

    let accumulator;
    let startIndex = 0;

    if (arguments.length >= 2) {
        accumulator = initialValue;
    } else {
        /* initialValue nahi diya — first NON-HOLE element ko accumulator banao */
        accumulator = this[0];
        startIndex = 1;
    }

    for (let i = startIndex; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            accumulator = callback(accumulator, this[i], i, this);
        }
    }

    return accumulator;
};

/* Test 1: Sparse array + initialValue — hole skip hoga */
const sparse3 = [1, , 3];
const sparseResult = sparse3.myReduce((acc, curr) => acc + curr, 0);
console.log(sparseResult); // ✅ 4 (0 + 1 + 3)

/* Test 2: Sparse array + no initialValue — hole skip hoga */
const sparse4 = [1, , 3];
const sparseResult2 = sparse4.myReduce((acc, curr) => acc + curr);
console.log(sparseResult2); // ✅ 4 (1 + 3)

/* Test 3: Normal (dense) array abhi bhi theek chalega */
const dense = [1, 2, 3, 4, 5];
const denseResult = dense.myReduce((acc, curr) => acc + curr, 0);
console.log(denseResult); // ✅ 15

/* Test 4: Empty array + initialValue — kaam karega */
const emptyResult = [].myReduce((acc, curr) => acc + curr, 100);
console.log(emptyResult); // ✅ 100

/* Test 5: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myReduce("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 6: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.myReduce((acc, curr) => acc + curr, 0);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * initialValue Concept
 */

/**
 * initialValue kya hai?
 * - initialValue reduce() ka second argument hai.
 * - Yeh accumulator ka STARTING value set karta hai.
 *
 * Do cases hain:
 *
 * CASE 1: initialValue DIYA gaya
 * - accumulator = initialValue se start hoga
 * - Loop index 0 se start hoga (first element se)
 * - Callback har element ke liye chalega
 *
 *   Example: [1, 2, 3].reduce((acc, curr) => acc + curr, 10)
 *   Flow:
 *     acc = 10 (initialValue)
 *     Step 1: 10 + 1 = 11
 *     Step 2: 11 + 2 = 13
 *     Step 3: 13 + 3 = 16
 *   Result: 16
 *
 * CASE 2: initialValue NAHI diya gaya
 * - accumulator = this[0] (first element) se start hoga
 * - Loop index 1 se start hoga (second element se)
 * - Callback first element ke liye nahi chalega
 *
 *   Example: [1, 2, 3].reduce((acc, curr) => acc + curr)
 *   Flow:
 *     acc = 1 (first element)
 *     Step 1: 1 + 2 = 3
 *     Step 2: 3 + 3 = 6
 *   Result: 6
 *
 * Important: undefined as initialValue
 * - Agar user JAAN BOOJHKE undefined pass kare:
 *     [1, 2, 3].reduce((acc, curr) => acc + curr, undefined)
 *   Toh yeh CASE 1 mana jayega — initialValue diya gaya hai.
 *   accumulator = undefined se start hoga.
 *   Loop index 0 se start hoga.
 *
 * - Isliye hum `arguments.length >= 2` check karte hain,
 *   na ki `initialValue !== undefined`.
 * - arguments.length batata hai ki argument ACTUALLY pass hua ya nahi.
 */

/* Demo: initialValue diya gaya */
const withInit = [1, 2, 3, 4, 5].myReduce((acc, curr) => acc + curr, 10);
console.log(withInit); // ✅ 25 (10 + 1 + 2 + 3 + 4 + 5)

/* Demo: initialValue nahi diya gaya */
const withoutInit = [1, 2, 3, 4, 5].myReduce((acc, curr) => acc + curr);
console.log(withoutInit); // ✅ 15 (1 + 2 + 3 + 4 + 5)

/* Demo: undefined as initialValue — CASE 1 mana jayega */
const undefinedInit = [1, 2, 3].myReduce((acc, curr) => {
    if (acc === undefined) return curr;
    return acc + curr;
}, undefined);
console.log(undefinedInit); // ✅ 6

/* Demo: object build karna reduce se */
const people = [
    { name: 'Aslam', age: 25 },
    { name: 'Rahul', age: 30 },
    { name: 'Priya', age: 28 }
];

const nameToAge = people.myReduce((acc, person) => {
    acc[person.name] = person.age;
    return acc;
}, {});

console.log(nameToAge);
// ✅ { Aslam: 25, Rahul: 30, Priya: 28 }

/**
 * reduce vs map vs filter vs forEach — quick comparison:
 *
 * forEach:
 * - Kuch return NAHI karta (undefined return karta hai).
 * - Side effects ke liye use karo.
 *
 * map:
 * - NAYA array return karta hai (same length as original).
 * - Transformation ke liye use karo.
 *
 * filter:
 * - NAYA array return karta hai (length <= original).
 * - Filtering ke liye use karo.
 *
 * reduce:
 * - EK single value return karta hai (number, string, object, array — kuch bhi).
 * - Combine/aggregate karne ke liye use karo.
 * - Sabse powerful hai — map, filter, forEach sab reduce se banaye jaa sakte hain.
 */


In [ ]:
/**
 * Final Code:
 * 1. Type check — non-function callback pe TypeError throw hota hai
 * 2. Empty array + no initialValue — TypeError throw hota hai
 * 3. Sparse arrays — holes skip hote hain, callback call nahi hota
 * 4. Proper initialValue check — undefined as initialValue sahi handle hota hai
 *
 * reduce() sabse powerful hai — "many → one"
 */

Array.prototype.myReduce = function (callback, initialValue) {
  if (typeof callback !== 'function') {
    throw new TypeError(callback + ' is not a function');
  }

  if (this.length === 0 && arguments.length < 2) {
    throw new TypeError('Reduce of empty array with no initial value');
  }

  let accumulator;
  let startIndex = 0;

  if (arguments.length >= 2) {
    accumulator = initialValue;
  } else {
    accumulator = this[0];
    startIndex = 1;
  }

  for (let i = startIndex; i < this.length; i++) {
    if (i in this) {
      accumulator = callback(accumulator, this[i], i, this);
    }
  }

  return accumulator;
};


In [ ]:
/**
 * ReduceRight Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.reduceRight ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain reduceRight() kaise kaam karta hai:
 * - reduceRight() reduce() jaisa hi hai, bas yeh RIGHT se LEFT chalta hai.
 * - Yeh array ke saare elements ko combine karke EK single value deta hai.
 * - Simple words mein: "many → one (from right side)"
 *
 * reduce() vs reduceRight():
 * - reduce()      → left to right (index 0 se length-1 tak)
 * - reduceRight() → right to left (length-1 se index 0 tak)
 *
 * Callback function ko 4 cheezein milti hain:
 * 1. accumulator   : pichle step ka result (yeh value step-by-step grow karti hai)
 * 2. currentValue  : current element (right side se aana shuru hota hai)
 * 3. index (optional) : element ka position
 * 4. array (optional) : original array jis pe reduceRight() call hua
 *
 * Very important rule:
 * - Callback jo bhi return karta hai, woh NEXT accumulator ban jata hai.
 *
 * reduceRight() ka second argument hota hai 'initialValue' —
 * yeh accumulator ka starting value set karta hai.
 *
 * Syntax: array.reduceRight(callback, initialValue)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.myReduceRight = function (callback, initialValue) {
    let accumulator = initialValue;

    /* Loop RIGHT to LEFT — last index se 0 tak */
    for (let i = this.length - 1; i >= 0; i--) {
        /* Agar initialValue nahi diya gaya, toh last element ko accumulator banao */
        if (accumulator === undefined) {
            accumulator = this[i];
        } else {
            accumulator = callback(accumulator, this[i], i, this);
        }
    }

    return accumulator;
};

/* Test 1: Sum of numbers */
const numbers = [1, 2, 3, 4, 5];

const sumFn = function (accumulator, currentValue) {
    return accumulator + currentValue;
};

const total = numbers.myReduceRight(sumFn, 0);
console.log(total); // ✅ 15

/**
 * Step-by-step flow (Right to Left):
 *
 * Initial accumulator = 0 (initialValue)
 *
 * Step 1: 0 + 5 = 5
 * Step 2:  reduceRight pe alag result aata hai.
 */

/* Test 3: Object build karna reduceRight se */
const people = [
    { name: 'Aslam', age: 25 },
    { name: 'Rahul', age: 30 },
    { name: 'Priya', age: 28 }
];

/* Last (rightmost) element ka name pehle aayega */
const nameToAge = people.myReduceRight((acc, person) => {
    acc[person.name] = person.age;
    return acc;
}, {});

console.log(nameToAge);
// ✅ { Priya: 28, Rahul: 30, Aslam: 25 }

/**
 * forEach vs map vs filter vs reduce vs reduceRight ka difference:
 *
 * forEach()     → sirf loop chalata hai, kuch return nahi karta
 * map()         → har element ko transform karta hai, naya array deta hai
 * filter()      → unwanted elements hatata hai, naya array deta hai
 * reduce()      → combines LEFT to RIGHT, EK value deta hai
 * reduceRight() → combines RIGHT to LEFT, EK value deta hai
 *
 * reduceRight() = "many → one (from right side)"
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 4 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Callback nahi diya gaya
 * - Agar koi [1, 2, 3].myReduceRight() bina argument ke call kare,
 *   toh 'callback' undefined ban jata hai.
 * - undefined() ko call karna allowed nahi — JS throw karta hai:
 *   TypeError: callback is not a function
 */

const failed1 = [1, 2, 3].myReduceRight();  /* TypeError: callback is not a function */

/**
 * FAILURE 2 — Empty array + no initialValue
 * - Agar array EMPTY hai aur initialValue NAHI di gayi:
 *     [].myReduceRight((acc, curr) => acc + curr)
 * - Hamara polyfill:
 *   → accumulator = undefined (initialValue nahi hai)
 *   → Loop nahi chalega (length = 0)
 *   → return undefined ❌
 *
 * - Native reduceRight() is case mein TypeError throw karta hai:
 *   "Reduce of empty array with no initial value"
 * - Yeh important hai kyunki undefined silently return karna
 *   galat hai — user ko pata hi nahi chalega ki kuch galat hua.
 */

const failed2 = [].myReduceRight((acc, curr) => acc + curr);
console.log(failed2); // ❌ undefined — native reduceRight() throws TypeError

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta.
 *
 * - Native reduceRight() holes ko SKIP karta hai — callback unke liye nahi chalta.
 *
 * - Hamara polyfill blindly length-1 se 0 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, this[1] undefined padta hai, callback phir bhi chalta hai.
 * - Isse accumulator undefined ke saath combine ho jata hai → NaN.
 *
 * Result: NaN instead of expected 4 (3 + 1)
 */

const sparse = [1, , 3];
const failed3 = sparse.myReduceRight((acc, curr) => acc + curr, 0);
console.log(failed3); // ❌ NaN — native reduceRight() gives 4

/**
 * FAILURE 4 — initialValue ka check ghalat hai
 * - Humne `accumulator === undefined` check kiya hai.
 * - Lekin agar user JAAN BOOJHKE undefined as initialValue pass kare:
 *     [1, 2, 3].myReduceRight((acc, curr) => acc + curr, undefined)
 *   Toh hamara code sochega initialValue nahi di gayi,
 *   aur last element ko accumulator bana dega.
 * - Native reduceRight() mein undefined ek VALID initialValue hai.
 *
 * - Fix: humein check karna hoga ki initialValue argument
 *   ACTUALLY pass hua ya nahi — `arguments.length` se pata chalta hai.
 */


In [ ]:
/**
 * Edge Cases 1 & 2 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar callback function nahi hai toh?
 * - Agar koi myReduceRight() bina argument ke call kare, ya null ya number de,
 *   toh code unexpectedly crash ho jayega.
 * - Native reduceRight() clear TypeError throw karta hai, humein bhi karna chahiye.
 *
 * Interviewer: Empty array + no initialValue ka issue?
 *
 * You: Haan, agar array empty hai aur initialValue nahi di gayi,
 *   toh humein TypeError throw karna chahiye — undefined silently
 *   return karna galat hai.
 * - Check: agar array ki length 0 hai AUR initialValue nahi di gayi,
 *   toh throw TypeError.
 *
 * initialValue actually pass hua ya nahi — yeh check karne ke liye
 * `arguments.length` use karenge.
 * - arguments.length < 2 ka matlab: sirf callback mila, initialValue nahi.
 * - arguments.length >= 2 ka matlab: dono mile, chahe initialValue undefined ho.
 */

/* Step 2: Type check + empty array check + proper initialValue check */
Array.prototype.myReduceRight = function (callback, initialValue) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    /* Edge Case 2: empty array + no initialValue → TypeError */
    if (this.length === 0 && arguments.length < 2) {
        throw new TypeError('Reduce of empty array with no initial value');
    }

    /* Edge Case 4: check karo ki initialValue actually pass hua ya nahi */
    let accumulator;
    let startIndex = this.length - 1;  /* Right side se start */

    if (arguments.length >= 2) {
        /* initialValue diya gaya — chahe woh undefined ho */
        accumulator = initialValue;
    } else {
        /* initialValue nahi diya — last element ko accumulator banao */
        accumulator = this[this.length - 1];
        startIndex = this.length - 2;  /* Second last element se loop start */
    }

    /* Loop RIGHT to LEFT — startIndex se 0 tak */
    for (let i = startIndex; i >= 0; i--) {
        accumulator = callback(accumulator, this[i], i, this);
    }

    return accumulator;
};

/* Test 1: No callback — TypeError aana chahiye */
try {
    [1, 2, 3].myReduceRight();
} catch (e) {
    console.log(e.message); // ✅ "undefined is not a function"
}

/* Test 2: null as callback — TypeError aana chahiye */
try {
    [1, 2, 3].myReduceRight(null);
} catch (e) {
    console.log(e.message); // ✅ "null is not a function"
}

/* Test 3: Empty array + no initialValue — TypeError aana chahiye */
try {
    [].myReduceRight((acc, curr) => acc + curr);
} catch (e) {
    console.log(e.message); // ✅ "Reduce of empty array with no initial value"
}

/* Test 4: Empty array + initialValue — kaam karega */
const emptyResult = [].myReduceRight((acc, curr) => acc + curr, 0);
console.log(emptyResult); // ✅ 0

/* Test 5: undefined as initialValue — ab sahi handle hoga */
const undefinedInit = [1, 2, 3].myReduceRight((acc, curr) => {
    if (acc === undefined) return curr;
    return acc + curr;
}, undefined);
console.log(undefinedInit); // ✅ 6 (3 + 2 + 1)


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 3 edge cases fix ho gaye:
 * a. Type check — non-function callbacks ab TypeError dete hain.
 * b. Empty array + no initialValue — ab clear TypeError deta hai.
 * c. Proper initialValue check — undefined as initialValue ab sahi handle hota hai.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays abhi bhi broken
 * - Loop abhi bhi blindly startIndex se 0 tak chalta hai, koi hole check nahi.
 *
 * [1, , 3] ke liye reduceRight((acc, curr) => acc + curr, 0):
 *   i = 2 → this[2] = 3     → callback chalta → 0 + 3 = 3   ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → callback undefined ke saath phir bhi chalta hai
 *           → 3 + undefined = NaN                            ❌
 *   i = 0 → this[0] = 1     → callback chalta → NaN + 1 = NaN ✅(broken)
 *
 * Result:   NaN
 * Expected: 4 (0 + 3 + 1 — hole skip hona chahiye)
 *
 * Problem: Hum index 1 pe callback call kar rahe hain,
 * jabki woh hole hai. Native reduceRight() kabhi nahi karta.
 *
 * Fix: 'i in this' check use karo — yeh batata hai ki
 * index i array mein ACTUALLY exist karta hai ya nahi.
 * Holes ke liye false return karta hai.
 */

const sparse2 = [1, , 3];
const stillBroken = sparse2.myReduceRight((acc, curr) => acc + curr, 0);
console.log(stillBroken); // ❌ NaN — native reduceRight() gives 4


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native reduceRight() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myReduceRight = function (callback, initialValue) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    /* Edge Case 2: empty array + no initialValue → TypeError */
    if (this.length === 0 && arguments.length < 2) {
        throw new TypeError('Reduce of empty array with no initial value');
    }

    let accumulator;
    let startIndex = this.length - 1;  /* Right side se start */

    if (arguments.length >= 2) {
        accumulator = initialValue;
    } else {
        /* initialValue nahi diya — last element ko accumulator banao */
        accumulator = this[this.length - 1];
        startIndex = this.length - 2;  /* Second last element se loop start */
    }

    /* Loop RIGHT to LEFT — startIndex se 0 tak */
    for (let i = startIndex; i >= 0; i--) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            accumulator = callback(accumulator, this[i], i, this);
        }
    }

    return accumulator;
};

/* Test 1: Sparse array + initialValue — hole skip hoga */
const sparse3 = [1, , 3];
const sparseResult = sparse3.myReduceRight((acc, curr) => acc + curr, 0);
console.log(sparseResult); // ✅ 4 (0 + 3 + 1)

/* Test 2: Sparse array + no initialValue — hole skip hoga */
const sparse4 = [1, , 3];
const sparseResult2 = sparse4.myReduceRight((acc, curr) => acc + curr);
console.log(sparseResult2); // ✅ 4 (3 + 1)

/* Test 3: Normal (dense) array abhi bhi theek chalega */
const dense = [1, 2, 3, 4, 5];
const denseResult = dense.myReduceRight((acc, curr) => acc + curr, 0);
console.log(denseResult); // ✅ 15

/* Test 4: Empty array + initialValue — kaam karega */
const emptyResult = [].myReduceRight((acc, curr) => acc + curr, 100);
console.log(emptyResult); // ✅ 100

/* Test 5: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myReduceRight("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 6: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.myReduceRight((acc, curr) => acc + curr, 0);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * initialValue Concept — Detail Mein (Hinglish)
 */

/**
 * initialValue kya hai?
 * - initialValue reduceRight() ka second argument hai.
 * - Yeh accumulator ka STARTING value set karta hai.
 *
 * Do cases hain:
 *
 * CASE 1: initialValue DIYA gaya
 * - accumulator = initialValue se start hoga
 * - Loop last index (length-1) se start hoga
 * - Callback har element ke liye chalega
 *
 *   Example: [1, 2, 3].reduceRight((acc, curr) => acc + curr, 10)
 *   Flow:
 *     acc = 10 (initialValue)
 *     Step 1: 10 + 3 = 13
 *     Step 2: 13 + 2 = 15
 *     Step 3: 15 + 1 = 16
 *   Result: 16
 *
 * CASE 2: initialValue NAHI diya gaya
 * - accumulator = this[length-1] (last element) se start hoga
 * - Loop second-last index (length-2) se start hoga
 * - Callback last element ke liye nahi chalega
 *
 *   Example: [1, 2, 3].reduceRight((acc, curr) => acc + curr)
 *   Flow:
 *     acc = 3 (last element)
 *     Step 1: 3 + 2 = 5
 *     Step 2: 5 + 1 = 6
 *   Result: 6
 *
 * Important: undefined as initialValue
 * - Agar user JAAN BOOJHKE undefined pass kare:
 *     [1, 2, 3].reduceRight((acc, curr) => acc + curr, undefined)
 *   Toh yeh CASE 1 mana jayega — initialValue diya gaya hai.
 *   accumulator = undefined se start hoga.
 *   Loop last index se start hoga.
 *
 * - Isliye hum `arguments.length >= 2` check karte hain,
 *   na ki `initialValue !== undefined`.
 * - arguments.length batata hai ki argument ACTUALLY pass hua ya nahi.
 */

/* Demo: initialValue diya gaya */
const withInit = [1, 2, 3, 4, 5].myReduceRight((acc, curr) => acc + curr, 10);
console.log(withInit); // ✅ 25 (10 + 5 + 4 + 3 + 2 + 1)

/* Demo: initialValue nahi diya gaya */
const withoutInit = [1, 2, 3, 4, 5].myReduceRight((acc, curr) => acc + curr);
console.log(withoutInit); // ✅ 15 (5 + 4 + 3 + 2 + 1)

/* Demo: undefined as initialValue — CASE 1 mana jayega */
const undefinedInit = [1, 2, 3].myReduceRight((acc, curr) => {
    if (acc === undefined) return curr;
    return acc + curr;
}, undefined);
console.log(undefinedInit); // ✅ 6

/* Demo: Object build karna reduceRight se */
const people = [
    { name: 'Aslam', age: 25 },
    { name: 'Rahul', age: 30 },
    { name: 'Priya', age: 28 }
];

/* Last (rightmost) element ka name pehle aayega */
const nameToAge = people.myReduceRight((acc, person) => {
    acc[person.name] = person.age;
    return acc;
}, {});

console.log(nameToAge);
// ✅ { Priya: 28, Rahul: 30, Aslam: 25 }

/**
 * reduce vs reduceRight — quick comparison:
 *
 * reduce:
 * - Starts from index 0
 * - Moves left → right
 * - First element becomes accumulator (if no initialValue)
 *
 * reduceRight:
 * - Starts from last index
 * - Moves right → left
 * - Last element becomes accumulator (if no initialValue)
 *
 * Output can be same for addition (1+2+3 = 3+2+1),
 * but DIFFERENT for strings, arrays, and objects.
 */


In [ ]:
/**
 * Final Code:
 * 1. Type check — non-function callback pe TypeError throw hota hai
 * 2. Empty array + no initialValue — TypeError throw hota hai
 * 3. Sparse arrays — holes skip hote hain, callback call nahi hota
 * 4. Proper initialValue check — undefined as initialValue sahi handle hota hai
 *
 * reduceRight() reduce() jaisa hi hai, bas direction ultee hai.
 * "many → one (from right side)"
 */

Array.prototype.myReduceRight = function (callback, initialValue) {
  if (typeof callback !== 'function') {
    throw new TypeError(callback + ' is not a function');
  }

  if (this.length === 0 && arguments.length < 2) {
    throw new TypeError('Reduce of empty array with no initial value');
  }

  let accumulator;
  let startIndex = this.length - 1;

  if (arguments.length >= 2) {
    accumulator = initialValue;
  } else {
    accumulator = this[this.length - 1];
    startIndex = this.length - 2;
  }

  for (let i = startIndex; i >= 0; i--) {
    if (i in this) {
      accumulator = callback(accumulator, this[i], i, this);
    }
  }

  return accumulator;
};


In [ ]:
/**
 * Find Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.find ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain find() kaise kaam karta hai:
 * - find() array mein condition ke basis pe element search karta hai.
 * - Jo pehla element condition satisfy kare, woh return kar deta hai.
 * - Match milte hi loop STOP ho jata hai — aage check nahi karta.
 *
 * Important points:
 * 1. find() FIRST matching element return karta hai.
 * 2. Agar kuch match nahi hua, toh undefined return karta hai.
 * 3. Yeh NAYA array NAHI banata (filter se yahi difference hai).
 * 4. Original array change nahi hota.
 *
 * Callback function ko 3 cheezein milti hain:
 * 1. currentValue  : current element
 * 2. index (optional) : element ka position
 * 3. array (optional) : original array jis pe find() call hua
 *
 * Very important rule:
 * - Callback ko true ya false return karna hota hai.
 * - true  → element mil gaya, return karo aur loop roko
 * - false → aage badho, next element check karo
 *
 * find() ka second argument hota hai 'thisArg' —
 * yeh callback ke andar 'this' ko control karta hai.
 *
 * Syntax: array.find(callback, thisArg)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.myFind = function (callback) {
    for (let i = 0; i < this.length; i++) {
        /* Agar callback true return kare, toh element return karo aur loop roko */
        if (callback(this[i], i, this)) {
            return this[i];
        }
    }

    /* Agar kuch match nahi hua, toh undefined return karo */
    return undefined;
};

/* Test 1: Simple search */
const numbers = [1, 2, 3, 4, 5];

const found = numbers.myFind(function (number) {
    return number > 3;
});

console.log(found); // ✅ 4

/**
 * Step-by-step flow:
 *
 * numbers = [1, 2, 3, 4, 5]
 *
 * Check 1 → 1 > 3 ❌
 * Check 2 → 2 > 3 ❌
 * Check 3 → 3 > 3 ❌
 * Check 4 → 4 > 3 ✅ → match mil gaya, loop stop, return 4
 */

/* Test 2: Object array mein search by id */
const users = [
    { id: 1, name: 'Aslam' },
    { id: 2, name: 'Rahul' },
    { id: 3, name: 'Priya' }
];

const user = users.myFind((u) => u.id === 2);
console.log(user); // ✅ { id: 2, name: 'Rahul' }

/* Test 3: Kuch match na ho toh undefined */
const notFound = numbers.myFind((num) => num > 100);
console.log(notFound); // ✅ undefined

/**
 * find() vs filter() ka difference:
 *
 * find():
 * - Returns ONE element (pehla match)
 * - Loop pehle match pe stop ho jata hai
 *
 * filter():
 * - Returns an ARRAY (saare matches)
 * - Loop end tak chalta hai
 *
 * find() = "search and stop"
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 4 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Callback nahi diya gaya
 * - Agar koi [1, 2, 3].myFind() bina argument ke call kare,
 *   toh 'callback' undefined ban jata hai.
 * - undefined() ko call karna allowed nahi — JS throw karta hai:
 *   TypeError: callback is not a function
 */

const failed1 = [1, 2, 3].myFind();  /* TypeError: callback is not a function */

/**
 * FAILURE 2 — thisArg ignore ho raha hai
 *
 * Pehle samajhte hain thisArg kya hai:
 * - thisArg ek second argument hai jo find() ko pass hota hai.
 * - Iska kaam: callback ke ANDAR 'this' ka value set karna.
 * - Matlab callback ke andar jab hum 'this' likhte hain,
 *   toh woh thisArg object ko point karega.
 *
 * Example:
 *   const obj = { min: 3 };
 *   [1, 2, 3, 4, 5].myFind(function (num) {
 *       return num > this.min;  // yahan 'this' = obj hona chahiye
 *   }, obj);
 *
 * - Hamara polyfill thisArg ko accept hi nahi karta!
 * - Toh 'this' ka default value lagta hai (undefined ya global).
 * - Iska matlab this.min undefined hai.
 * - num > undefined → hamesha false aata hai.
 * - Result: undefined (kuch nahi mila) instead of expected 4
 */

const obj = { min: 3 };

const failed2 = [1, 2, 3, 4, 5].myFind(function (num) {
    return num > this.min;
}, obj);

console.log(failed2); // ❌ undefined — should be 4

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta,
 *   undefined value nahi hai.
 *
 * - Native find() holes ko SKIP karta hai — callback unke liye nahi chalta.
 *
 * - Hamara polyfill blindly 0 se length-1 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, sparse[1] undefined padta hai, callback phir bhi chalta hai.
 * - Agar callback undefined ke liye true return kare, toh undefined
 *   return ho jata hai — yeh galat hai.
 *
 * Result: undefined (wrong match) instead of expected 3
 */

/* Native find() skips holes — callback call nahi karta */
const sparse = [1, , 3];

const failed3 = sparse.myFind((x) => x === undefined);
console.log(failed3); // ❌ undefined — native find() also skips, but logic is wrong

/**
 * FAILURE 4 — null/undefined pe call karna
 * - Agar koi null ya undefined pe find() call kare:
 *     null.myFind(...)
 *     undefined.myFind(...)
 * - Hamara polyfill crash ho jayega — TypeError.
 * - Native find() bhi TypeError throw karta hai, par clear message ke saath.
 * - Humien bhi check karna chahiye ki 'this' null ya undefined na ho.
 */


In [ ]:
/**
 * Edge Cases 1, 2 & 4 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar callback function nahi hai toh?
 * - Agar koi myFind() bina argument ke call kare, ya null ya number de,
 *   toh code unexpectedly crash ho jayega.
 * - Native find() clear TypeError throw karta hai, humein bhi karna chahiye.
 *
 * Interviewer: thisArg ka issue?
 *
 * You: Haan, basic version second argument ignore karta hai.
 * - Mujhe thisArg ko callback ke andar 'this' banana hai.
 * - Iske liye callback.call(thisArg, ...) use karunga,
 *   direct call ki jagah.
 *
 * Interviewer: null/undefined pe call karna?
 *
 * You: Haan, agar 'this' null ya undefined hai, toh TypeError throw karna chahiye.
 */

/* Step 2: Type check + thisArg binding + null check */
Array.prototype.myFind = function (callback, thisArg) {
    /* Edge Case 4: this null ya undefined nahi hona chahiye */
    if (this == null) {
        throw new TypeError('Array.prototype.find called on null or undefined');
    }

    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
        if (callback.call(thisArg, this[i], i, this)) {
            return this[i];
        }
    }

    return undefined;
};

/* Test 1: No callback — TypeError aana chahiye */
try {
    [1, 2, 3].myFind();
} catch (e) {
    console.log(e.message); // ✅ "undefined is not a function"
}

/* Test 2: null as callback — TypeError aana chahiye */
try {
    [1, 2, 3].myFind(null);
} catch (e) {
    console.log(e.message); // ✅ "null is not a function"
}

/* Test 3: thisArg ab sahi kaam karta hai */
const obj2 = { min: 3 };
const fixedResult = [1, 2, 3, 4, 5].myFind(function (num) {
    return num > this.min;  // 'this' = obj2, this.min = 3
}, obj2);
console.log(fixedResult); // ✅ 4

/* Test 4: null pe call — TypeError aana chahiye */
try {
    Array.prototype.myFind.call(null, (x) => x > 0);
} catch (e) {
    console.log(e.message); // ✅ "Array.prototype.find called on null or undefined"
}


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 3 edge cases fix ho gaye:
 * a. Type check — non-function callbacks ab TypeError dete hain.
 * b. thisArg — 'this' ab sahi se callback ke andar bind hota hai.
 * c. null/undefined check — ab clear TypeError deta hai.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays abhi bhi broken
 * - Loop abhi bhi blindly 0 se length-1 tak chalta hai, koi hole check nahi.
 *
 * [1, , 3] ke liye find(x => x === undefined):
 *   i = 0 → this[0] = 1     → callback chalta → 1 === undefined → false ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → callback undefined ke saath phir bhi chalta hai
 *           → undefined === undefined → true → returns undefined ❌
 *
 * Result:   undefined
 * Expected: 3 (hole skip hona chahiye, uske baad 3 match nahi karta is condition mein)
 *
 * Problem: Hum index 1 pe callback call kar rahe hain,
 * jabki woh hole hai. Native find() kabhi nahi karta.
 *
 * Fix: 'i in this' check use karo — yeh batata hai ki
 * index i array mein ACTUALLY exist karta hai ya nahi.
 * Holes ke liye false return karta hai.
 */

const sparse2 = [1, , 3];
const stillBroken = sparse2.myFind((x) => x === undefined);
console.log(stillBroken); // ❌ undefined — native find() skips hole


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native find() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myFind = function (callback, thisArg) {
    /* Edge Case 4: this null ya undefined nahi hona chahiye */
    if (this == null) {
        throw new TypeError('Array.prototype.find called on null or undefined');
    }

    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
            if (callback.call(thisArg, this[i], i, this)) {
                return this[i];
            }
        }
    }

    return undefined;
};

/* Test 1: Sparse array — hole skip hoga */
const sparse3 = [1, , 3];
const sparseResult = sparse3.myFind((x) => x === undefined);
console.log(sparseResult); // ✅ undefined — kyunki hole skip ho gaya, koi explicit undefined nahi hai

/* Test 2: Normal (dense) array abhi bhi theek chalega */
const dense = [1, 2, 3, 4, 5];
const denseResult = dense.myFind((x) => x > 3);
console.log(denseResult); // ✅ 4

/* Test 3: thisArg abhi bhi kaam karta hai */
const obj3 = { min: 3 };
const thisArgResult = [1, 2, 3, 4, 5].myFind(function (num) {
    return num > this.min;
}, obj3);
console.log(thisArgResult); // ✅ 4

/* Test 4: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myFind("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 5: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.myFind((x) => x > 100);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * thisArg Concept — Detail Mein (Hinglish)
 */

/**
 * thisArg kya hai?
 * - thisArg find() ka second argument hai.
 * - Iska kaam: callback ke andar 'this' keyword ka value decide karna.
 *
 * 'this' JavaScript mein tricky hai:
 * - Normal function mein 'this' depends karta hai HOW it was called.
 * - Arrow function mein 'this' apne surrounding scope se aata hai.
 *
 * Agar hum thisArg NAHI dete, toh:
 * - Non-strict mode mein: 'this' global object (window) ban jata hai.
 * - Strict mode mein: 'this' undefined ho jata hai.
 * - Dono mein this.min undefined milega → comparison false hoga → undefined.
 *
 * Agar hum thisArg DETE hain, toh:
 * - callback.call(thisArg, ...) 'this' ko thisArg pe set karta hai.
 * - Toh this.min = obj.min = 3 milta hai.
 *
 * Real-world example:
 *   const config = { min: 3 };
 *   [1, 2, 3, 4, 5].find(function (n) {
 *       return n > this.min;  // 'this' = config
 *   }, config);
 *   // Result: 4 — kyunki 'this' = config
 *
 * Yaad rakho: thisArg sirf normal (non-arrow) functions pe asar karta hai.
 * - Arrow functions apna 'this' nahi lete, woh surrounding scope se lete hain.
 * - So arrow function ke saath thisArg pass karna koi farak nahi padta.
 */

/* Demo: thisArg normal function ke saath */
const config1 = { min: 3 };
const normalFnResult = [1, 2, 3, 4, 5].myFind(function (n) {
    return n > this.min;  // 'this' = config1
}, config1);
console.log(normalFnResult); // ✅ 4

/* Demo: thisArg arrow function ke saath — farak nahi padta */
const config2 = { min: 3 };
const arrowFnResult = [1, 2, 3, 4, 5].myFind((n) => {
    return n > config2.min;  // arrow function 'this' ignore karta hai
}, config2);
console.log(arrowFnResult); // ✅ 4 — lekin config2 directly use kiya

/**
 * find() vs filter() vs forEach() — quick comparison:
 *
 * find():
 * - Returns the FIRST matching ELEMENT (not array).
 * - Loop stops at first match — efficient for searching.
 * - Returns undefined if nothing matches.
 *
 * filter():
 * - Returns an ARRAY of ALL matching elements.
 * - Loop runs till end — checks every element.
 * - Returns empty array [] if nothing matches.
 *
 * forEach():
 * - Returns nothing (undefined).
 * - Loop runs till end — no early stop.
 * - Used for side effects (logging, updating, etc.).
 */


In [ ]:
/**
 * Final Code:
 * 1. Type check — non-function callback pe TypeError throw hota hai
 * 2. thisArg — 'this' sahi se callback ke andar bind hota hai
 * 3. Sparse arrays — holes skip hote hain, callback call nahi hota
 * 4. null/undefined check — this null ya undefined na ho
 *
 * find() = "search and stop" — pehla match milte hi return karo
 */

Array.prototype.myFind = function (callback, thisArg) {
  if (this == null) {
    throw new TypeError('Array.prototype.find called on null or undefined');
  }

  if (typeof callback !== 'function') {
    throw new TypeError(callback + ' is not a function');
  }

  for (let i = 0; i < this.length; i++) {
    if (i in this) {
      if (callback.call(thisArg, this[i], i, this)) {
        return this[i];
      }
    }
  }

  return undefined;
};


In [ ]:
/**
 * FindIndex Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.findIndex ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain findIndex() kaise kaam karta hai:
 * - findIndex() array mein condition ke basis pe element search karta hai.
 * - Jo pehla element condition satisfy kare, uska INDEX return kar deta hai.
 * - Match milte hi loop STOP ho jata hai — aage check nahi karta.
 *
 * find() vs findIndex():
 * - find()        → element return karta hai
 * - findIndex()   → element ka index return karta hai
 *
 * Important points:
 * 1. findIndex() FIRST matching element ka INDEX return karta hai.
 * 2. Agar kuch match nahi hua, toh -1 return karta hai.
 * 3. Yeh NAYA array NAHI banata.
 * 4. Original array change nahi hota.
 *
 * Callback function ko 3 cheezein milti hain:
 * 1. currentValue  : current element
 * 2. index (optional) : element ka position
 * 3. array (optional) : original array jis pe findIndex() call hua
 *
 * Very important rule:
 * - Callback ko true ya false return karna hota hai.
 * - true  → index return karo aur loop roko
 * - false → aage badho, next element check karo
 *
 * findIndex() ka second argument hota hai 'thisArg' —
 * yeh callback ke andar 'this' ko control karta hai.
 *
 * Syntax: array.findIndex(callback, thisArg)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.myFindIndex = function (callback) {
    for (let i = 0; i < this.length; i++) {
        /* Agar callback true return kare, toh index return karo aur loop roko */
        if (callback(this[i], i, this)) {
            return i;
        }
    }

    /* Agar kuch match nahi hua, toh -1 return karo */
    return -1;
};

/* Test 1: Simple search */
const numbers = [1, 2, 3, 4, 5];

const foundIndex = numbers.myFindIndex(function (number) {
    return number > 3;
});

console.log(foundIndex); // ✅ 3

/**
 * Step-by-step flow:
 *
 * numbers = [1, 2, 3, 4, 5]
 *
 * Check 0 → 1 > 3 ❌
 * Check 1 → 2 > 3 ❌
 * Check 2 → 3 > 3 ❌
 * Check 3 → 4 > 3 ✅ → match mil gaya, loop stop, return 3
 */

/* Test 2: Object array mein search by id */
const users = [
    { id: 1, name: 'Aslam' },
    { id: 2, name: 'Rahul' },
    { id: 3, name: 'Priya' }
];

const userIndex = users.myFindIndex((u) => u.id === 2);
console.log(userIndex); // ✅ 1

/* Test 3: Kuch match na ho toh -1 */
const notFound = numbers.myFindIndex((num) => num > 100);
console.log(notFound); // ✅ -1

/**
 * find() vs findIndex() vs indexOf() ka difference:
 *
 * find():
 * - Returns the FIRST matching ELEMENT.
 * - Condition ke basis pe search karta hai (callback use hota hai).
 *
 * findIndex():
 * - Returns the INDEX of first matching element.
 * - Condition ke basis pe search karta hai (callback use hota hai).
 *
 * indexOf():
 * - Returns the INDEX of first matching element.
 * - VALUE ke basis pe search karta hai (koi callback nahi).
 *
 * findIndex() = "search index and stop"
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 4 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Callback nahi diya gaya
 * - Agar koi [1, 2, 3].myFindIndex() bina argument ke call kare,
 *   toh 'callback' undefined ban jata hai.
 * - undefined() ko call karna allowed nahi — JS throw karta hai:
 *   TypeError: callback is not a function
 */

const failed1 = [1, 2, 3].myFindIndex();  /* TypeError: callback is not a function */

/**
 * FAILURE 2 — thisArg ignore ho raha hai
 *
 * Pehle samajhte hain thisArg kya hai:
 * - thisArg ek second argument hai jo findIndex() ko pass hota hai.
 * - Iska kaam: callback ke ANDAR 'this' ka value set karna.
 * - Matlab callback ke andar jab hum 'this' likhte hain,
 *   toh woh thisArg object ko point karega.
 *
 * Example:
 *   const obj = { min: 3 };
 *   [1, 2, 3, 4, 5].myFindIndex(function (num) {
 *       return num > this.min;  // yahan 'this' = obj hona chahiye
 *   }, obj);
 *
 * - Hamara polyfill thisArg ko accept hi nahi karta!
 * - Toh 'this' ka default value lagta hai (undefined ya global).
 * - Iska matlab this.min undefined hai.
 * - num > undefined → hamesha false aata hai.
 * - Result: -1 (kuch nahi mila) instead of expected 3
 */

const obj = { min: 3 };

const failed2 = [1, 2, 3, 4, 5].myFindIndex(function (num) {
    return num > this.min;
}, obj);

console.log(failed2); // ❌ -1 — should be 3

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta,
 *   undefined value nahi hai.
 *
 * - Native findIndex() holes ko SKIP karta hai — callback unke liye nahi chalta.
 *
 * - Hamara polyfill blindly 0 se length-1 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, sparse[1] undefined padta hai, callback phir bhi chalta hai.
 * - Agar callback undefined ke liye true return kare, toh index 1
 *   return ho jata hai — yeh galat hai.
 *
 * Result: 1 (wrong match on hole) instead of expected -1
 */

/* Native findIndex() skips holes — callback call nahi karta */
const sparse = [1, , 3];

const failed3 = sparse.myFindIndex((x) => x === undefined);
console.log(failed3); // ❌ 1 — native findIndex() skips hole, gives -1

/**
 * FAILURE 4 — null/undefined pe call karna
 * - Agar koi null ya undefined pe findIndex() call kare:
 *     null.myFindIndex(...)
 *     undefined.myFindIndex(...)
 * - Hamara polyfill crash ho jayega — TypeError.
 * - Native findIndex() bhi TypeError throw karta hai, par clear message ke saath.
 * - Humein bhi check karna chahiye ki 'this' null ya undefined na ho.
 */


In [ ]:
/**
 * Edge Cases 1, 2 & 4 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar callback function nahi hai toh?
 * - Agar koi myFindIndex() bina argument ke call kare, ya null ya number de,
 *   toh code unexpectedly crash ho jayega.
 * - Native findIndex() clear TypeError throw karta hai, humein bhi karna chahiye.
 *
 * Interviewer: thisArg ka issue?
 *
 * You: Haan, basic version second argument ignore karta hai.
 * - Mujhe thisArg ko callback ke andar 'this' banana hai.
 * - Iske liye callback.call(thisArg, ...) use karunga,
 *   direct call ki jagah.
 *
 * Interviewer: null/undefined pe call karna?
 *
 * You: Haan, agar 'this' null ya undefined hai, toh TypeError throw karna chahiye.
 */

/* Step 2: Type check + thisArg binding + null check */
Array.prototype.myFindIndex = function (callback, thisArg) {
    /* Edge Case 4: this null ya undefined nahi hona chahiye */
    if (this == null) {
        throw new TypeError('Array.prototype.findIndex called on null or undefined');
    }

    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
        if (callback.call(thisArg, this[i], i, this)) {
            return i;
        }
    }

    return -1;
};

/* Test 1: No callback — TypeError aana chahiye */
try {
    [1, 2, 3].myFindIndex();
} catch (e) {
    console.log(e.message); // ✅ "undefined is not a function"
}

/* Test 2: null as callback — TypeError aana chahiye */
try {
    [1, 2, 3].myFindIndex(null);
} catch (e) {
    console.log(e.message); // ✅ "null is not a function"
}

/* Test 3: thisArg ab sahi kaam karta hai */
const obj2 = { min: 3 };
const fixedResult = [1, 2, 3, 4, 5].myFindIndex(function (num) {
    return num > this.min;  // 'this' = obj2, this.min = 3
}, obj2);
console.log(fixedResult); // ✅ 3

/* Test 4: null pe call — TypeError aana chahiye */
try {
    Array.prototype.myFindIndex.call(null, (x) => x > 0);
} catch (e) {
    console.log(e.message); // ✅ "Array.prototype.findIndex called on null or undefined"
}


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 3 edge cases fix ho gaye:
 * a. Type check — non-function callbacks ab TypeError dete hain.
 * b. thisArg — 'this' ab sahi se callback ke andar bind hota hai.
 * c. null/undefined check — ab clear TypeError deta hai.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays abhi bhi broken
 * - Loop abhi bhi blindly 0 se length-1 tak chalta hai, koi hole check nahi.
 *
 * [1, , 3] ke liye findIndex(x => x === undefined):
 *   i = 0 → this[0] = 1     → callback chalta → 1 === undefined → false ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → callback undefined ke saath phir bhi chalta hai
 *           → undefined === undefined → true → returns 1    ❌
 *
 * Result:   1
 * Expected: -1 (hole skip hona chahiye, koi explicit undefined nahi hai)
 *
 * Problem: Hum index 1 pe callback call kar rahe hain,
 * jabki woh hole hai. Native findIndex() kabhi nahi karta.
 *
 * Fix: 'i in this' check use karo — yeh batata hai ki
 * index i array mein ACTUALLY exist karta hai ya nahi.
 * Holes ke liye false return karta hai.
 */

const sparse2 = [1, , 3];
const stillBroken = sparse2.myFindIndex((x) => x === undefined);
console.log(stillBroken); // ❌ 1 — native findIndex() skips hole, gives -1


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native findIndex() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myFindIndex = function (callback, thisArg) {
    /* Edge Case 4: this null ya undefined nahi hona chahiye */
    if (this == null) {
        throw new TypeError('Array.prototype.findIndex called on null or undefined');
    }

    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
            if (callback.call(thisArg, this[i], i, this)) {
                return i;
            }
        }
    }

    return -1;
};

/* Test 1: Sparse array — hole skip hoga */
const sparse3 = [1, , 3];
const sparseResult = sparse3.myFindIndex((x) => x === undefined);
console.log(sparseResult); // ✅ -1 — kyunki hole skip ho gaya, koi explicit undefined nahi hai

/* Test 2: Normal (dense) array abhi bhi theek chalega */
const dense = [1, 2, 3, 4, 5];
const denseResult = dense.myFindIndex((x) => x > 3);
console.log(denseResult); // ✅ 3

/* Test 3: thisArg abhi bhi kaam karta hai */
const obj3 = { min: 3 };
const thisArgResult = [1, 2, 3, 4, 5].myFindIndex(function (num) {
    return num > this.min;
}, obj3);
console.log(thisArgResult); // ✅ 3

/* Test 4: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myFindIndex("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 5: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.myFindIndex((x) => x > 100);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * thisArg Concept — Detail Mein (Hinglish)
 */

/**
 * thisArg kya hai?
 * - thisArg findIndex() ka second argument hai.
 * - Iska kaam: callback ke andar 'this' keyword ka value decide karna.
 *
 * 'this' JavaScript mein tricky hai:
 * - Normal function mein 'this' depends karta hai HOW it was called.
 * - Arrow function mein 'this' apne surrounding scope se aata hai.
 *
 * Agar hum thisArg NAHI dete, toh:
 * - Non-strict mode mein: 'this' global object (window) ban jata hai.
 * - Strict mode mein: 'this' undefined ho jata hai.
 * - Dono mein this.min undefined milega → comparison false hoga → -1.
 *
 * Agar hum thisArg DETE hain, toh:
 * - callback.call(thisArg, ...) 'this' ko thisArg pe set karta hai.
 * - Toh this.min = obj.min = 3 milta hai.
 *
 * Real-world example:
 *   const config = { min: 3 };
 *   [1, 2, 3, 4, 5].findIndex(function (n) {
 *       return n > this.min;  // 'this' = config
 *   }, config);
 *   // Result: 3 — kyunki 'this' = config, this.min = 3, pehla > 3 element index 3 pe hai
 *
 * Yaad rakho: thisArg sirf normal (non-arrow) functions pe asar karta hai.
 * - Arrow functions apna 'this' nahi lete, woh surrounding scope se lete hain.
 * - So arrow function ke saath thisArg pass karna koi farak nahi padta.
 */

/* Demo: thisArg normal function ke saath */
const config1 = { min: 3 };
const normalFnResult = [1, 2, 3, 4, 5].myFindIndex(function (n) {
    return n > this.min;  // 'this' = config1
}, config1);
console.log(normalFnResult); // ✅ 3

/* Demo: thisArg arrow function ke saath — farak nahi padta */
const config2 = { min: 3 };
const arrowFnResult = [1, 2, 3, 4, 5].myFindIndex((n) => {
    return n > config2.min;  // arrow function 'this' ignore karta hai
}, config2);
console.log(arrowFnResult); // ✅ 3 — lekin config2 directly use kiya

/**
 * find() vs findIndex() vs indexOf() — quick comparison:
 *
 * find():
 * - Returns the FIRST matching ELEMENT.
 * - Condition ke basis pe search (callback use hota hai).
 * - Returns undefined if nothing matches.
 *
 * findIndex():
 * - Returns the INDEX of first matching element.
 * - Condition ke basis pe search (callback use hota hai).
 * - Returns -1 if nothing matches.
 *
 * indexOf():
 * - Returns the INDEX of first matching element.
 * - VALUE ke basis pe search (koi callback nahi, direct value compare).
 * - Returns -1 if nothing matches.
 *
 * findIndex() = "search index and stop"
 */


In [ ]:
/**
 * Final Code:
 * 1. Type check — non-function callback pe TypeError throw hota hai
 * 2. thisArg — 'this' sahi se callback ke andar bind hota hai
 * 3. Sparse arrays — holes skip hote hain, callback call nahi hota
 * 4. null/undefined check — this null ya undefined na ho
 *
 * findIndex() = "search index and stop" — pehla match milte hi index return karo
 */

Array.prototype.myFindIndex = function (callback, thisArg) {
  if (this == null) {
    throw new TypeError('Array.prototype.findIndex called on null or undefined');
  }

  if (typeof callback !== 'function') {
    throw new TypeError(callback + ' is not a function');
  }

  for (let i = 0; i < this.length; i++) {
    if (i in this) {
      if (callback.call(thisArg, this[i], i, this)) {
        return i;
      }
    }
  }

  return -1;
};


In [ ]:
/**
 * Some Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.some ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain some() kaise kaam karta hai:
 * - some() check karta hai ki array mein KAM SE KAM EK element
 *   condition satisfy karta hai ya nahi.
 * - Agar ek bhi element match kare → true return karo aur loop stop.
 * - Agar koi bhi match na kare → false return karo.
 *
 * Simple words mein: "Kya koi ek bhi element condition pass karta hai?"
 *
 * Important points:
 * 1. some() boolean return karta hai (true ya false).
 * 2. Pehla match milte hi loop STOP ho jata hai.
 * 3. Yeh NAYA array NAHI banata.
 * 4. Original array change nahi hota.
 *
 * Callback function ko 3 cheezein milti hain:
 * 1. currentValue  : current element
 * 2. index (optional) : element ka position
 * 3. array (optional) : original array jis pe some() call hua
 *
 * Very important rule:
 * - Callback ko true ya false return karna hota hai.
 * - true  → mil gaya, loop stop, some() returns true
 * - false → aage badho, next element check karo
 *
 * some() ka second argument hota hai 'thisArg' —
 * yeh callback ke andar 'this' ko control karta hai.
 *
 * Syntax: array.some(callback, thisArg)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.mySome = function (callback) {
    for (let i = 0; i < this.length; i++) {
        /* Agar callback true return kare, toh true return karo aur loop roko */
        if (callback(this[i], i, this)) {
            return true;
        }
    }

    /* Agar kuch match nahi hua, toh false return karo */
    return false;
};

/* Test 1: Simple check — kya koi even number hai? */
const numbers = [1, 3, 5, 4, 7];

const hasEven = numbers.mySome(function (number) {
    return number % 2 === 0;
});

console.log(hasEven); // ✅ true

/**
 * Step-by-step flow:
 *
 * numbers = [1, 3, 5, 4, 7]
 *
 * Check 0 → 1 % 2 === 0 ❌
 * Check 1 → 3 % 2 === 0 ❌
 * Check 2 → 5 % 2 === 0 ❌
 * Check 3 → 4 % 2 === 0 ✅ → match mil gaya, loop stop, return true
 */

/* Test 2: Koi match na kare toh false */
const allOdd = [1, 3, 5, 7];

const hasEven2 = allOdd.mySome((num) => num % 2 === 0);
console.log(hasEven2); // ✅ false

/* Test 3: Empty array — hamesha false */
const empty = [].mySome((x) => x > 0);
console.log(empty); // ✅ false

/**
 * some() vs every() vs find() ka difference:
 *
 * some():
 * - Returns true agar KAM SE KAM EK element match kare.
 * - "Koi ek bhi?" — OR operation jaisa.
 * - Pehle match pe stop.
 *
 * every():
 * - Returns true agar SAARE elements match kare.
 * - "Sab ek ek?" — AND operation jaisa.
 * - Pehle false pe stop.
 *
 * find():
 * - Returns the matching ELEMENT (not boolean).
 * - Pehle match pe stop.
 *
 * some() = "koi ek bhi pass?"
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 4 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Callback nahi diya gaya
 * - Agar koi [1, 2, 3].mySome() bina argument ke call kare,
 *   toh 'callback' undefined ban jata hai.
 * - undefined() ko call karna allowed nahi — JS throw karta hai:
 *   TypeError: callback is not a function
 */

const failed1 = [1, 2, 3].mySome();  /* TypeError: callback is not a function */

/**
 * FAILURE 2 — thisArg ignore ho raha hai
 *
 * Pehle samajhte hain thisArg kya hai:
 * - thisArg ek second argument hai jo some() ko pass hota hai.
 * - Iska kaam: callback ke ANDAR 'this' ka value set karna.
 * - Matlab callback ke andar jab hum 'this' likhte hain,
 *   toh woh thisArg object ko point karega.
 *
 * Example:
 *   const obj = { min: 3 };
 *   [1, 2, 3, 4, 5].mySome(function (num) {
 *       return num > this.min;  // yahan 'this' = obj hona chahiye
 *   }, obj);
 *
 * - Hamara polyfill thisArg ko accept hi nahi karta!
 * - Toh 'this' ka default value lagta hai (undefined ya global).
 * - Iska matlab this.min undefined hai.
 * - num > undefined → hamesha false aata hai.
 * - Result: false (kuch nahi mila) instead of expected true
 */

const obj = { min: 3 };

const failed2 = [1, 2, 3, 4, 5].mySome(function (num) {
    return num > this.min;
}, obj);

console.log(failed2); // ❌ false — should be true

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta,
 *   undefined value nahi hai.
 *
 * - Native some() holes ko SKIP karta hai — callback unke liye nahi chalta.
 *
 * - Hamara polyfill blindly 0 se length-1 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, sparse[1] undefined padta hai, callback phir bhi chalta hai.
 * - Agar callback undefined ke liye true return kare, toh true
 *   return ho jata hai — yeh galat hai.
 *
 * Result: true (wrong match on hole) instead of expected false
 */

/* Native some() skips holes — callback call nahi karta */
const sparse = [1, , 3];

const failed3 = sparse.mySome((x) => x === undefined);
console.log(failed3); // ❌ true — native some() skips hole, gives false

/**
 * FAILURE 4 — null/undefined pe call karna
 * - Agar koi null ya undefined pe some() call kare:
 *     null.mySome(...)
 *     undefined.mySome(...)
 * - Hamara polyfill crash ho jayega — TypeError.
 * - Native some() bhi TypeError throw karta hai, par clear message ke saath.
 * - Humein bhi check karna chahiye ki 'this' null ya undefined na ho.
 */


In [ ]:
/**
 * Edge Cases 1, 2 & 4 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar callback function nahi hai toh?
 * - Agar koi mySome() bina argument ke call kare, ya null ya number de,
 *   toh code unexpectedly crash ho jayega.
 * - Native some() clear TypeError throw karta hai, humein bhi karna chahiye.
 *
 * Interviewer: thisArg ka issue?
 *
 * You: Haan, basic version second argument ignore karta hai.
 * - Mujhe thisArg ko callback ke andar 'this' banana hai.
 * - Iske liye callback.call(thisArg, ...) use karunga,
 *   direct call ki jagah.
 *
 * Interviewer: null/undefined pe call karna?
 *
 * You: Haan, agar 'this' null ya undefined hai, toh TypeError throw karna chahiye.
 */

/* Step 2: Type check + thisArg binding + null check */
Array.prototype.mySome = function (callback, thisArg) {
    /* Edge Case 4: this null ya undefined nahi hona chahiye */
    if (this == null) {
        throw new TypeError('Array.prototype.some called on null or undefined');
    }

    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
        if (callback.call(thisArg, this[i], i, this)) {
            return true;
        }
    }

    return false;
};

/* Test 1: No callback — TypeError aana chahiye */
try {
    [1, 2, 3].mySome();
} catch (e) {
    console.log(e.message); // ✅ "undefined is not a function"
}

/* Test 2: null as callback — TypeError aana chahiye */
try {
    [1, 2, 3].mySome(null);
} catch (e) {
    console.log(e.message); // ✅ "null is not a function"
}

/* Test 3: thisArg ab sahi kaam karta hai */
const obj2 = { min: 3 };
const fixedResult = [1, 2, 3, 4, 5].mySome(function (num) {
    return num > this.min;  // 'this' = obj2, this.min = 3
}, obj2);
console.log(fixedResult); // ✅ true

/* Test 4: null pe call — TypeError aana chahiye */
try {
    Array.prototype.mySome.call(null, (x) => x > 0);
} catch (e) {
    console.log(e.message); // ✅ "Array.prototype.some called on null or undefined"
}

/* Test 5: Empty array — hamesha false */
const emptyResult = [].mySome((x) => x > 0);
console.log(emptyResult); // ✅ false


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 3 edge cases fix ho gaye:
 * a. Type check — non-function callbacks ab TypeError dete hain.
 * b. thisArg — 'this' ab sahi se callback ke andar bind hota hai.
 * c. null/undefined check — ab clear TypeError deta hai.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays abhi bhi broken
 * - Loop abhi bhi blindly 0 se length-1 tak chalta hai, koi hole check nahi.
 *
 * [1, , 3] ke liye some(x => x === undefined):
 *   i = 0 → this[0] = 1     → callback chalta → 1 === undefined → false ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → callback undefined ke saath phir bhi chalta hai
 *           → undefined === undefined → true → returns true   ❌
 *
 * Result:   true
 * Expected: false (hole skip hona chahiye, koi explicit undefined nahi hai)
 *
 * Problem: Hum index 1 pe callback call kar rahe hain,
 * jabki woh hole hai. Native some() kabhi nahi karta.
 *
 * Fix: 'i in this' check use karo — yeh batata hai ki
 * index i array mein ACTUALLY exist karta hai ya nahi.
 * Holes ke liye false return karta hai.
 */

const sparse2 = [1, , 3];
const stillBroken = sparse2.mySome((x) => x === undefined);
console.log(stillBroken); // ❌ true — native some() skips hole, gives false


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native some() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.mySome = function (callback, thisArg) {
    /* Edge Case 4: this null ya undefined nahi hona chahiye */
    if (this == null) {
        throw new TypeError('Array.prototype.some called on null or undefined');
    }

    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
            if (callback.call(thisArg, this[i], i, this)) {
                return true;
            }
        }
    }

    return false;
};

/* Test 1: Sparse array — hole skip hoga */
const sparse3 = [1, , 3];
const sparseResult = sparse3.mySome((x) => x === undefined);
console.log(sparseResult); // ✅ false — kyunki hole skip ho gaya, koi explicit undefined nahi hai

/* Test 2: Normal (dense) array abhi bhi theek chalega */
const dense = [1, 3, 5, 4, 7];
const denseResult = dense.mySome((x) => x % 2 === 0);
console.log(denseResult); // ✅ true

/* Test 3: thisArg abhi bhi kaam karta hai */
const obj3 = { min: 3 };
const thisArgResult = [1, 2, 3, 4, 5].mySome(function (num) {
    return num > this.min;
}, obj3);
console.log(thisArgResult); // ✅ true

/* Test 4: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].mySome("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 5: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.mySome((x) => x > 100);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * thisArg Concept — Detail Mein (Hinglish)
 */

/**
 * thisArg kya hai?
 * - thisArg some() ka second argument hai.
 * - Iska kaam: callback ke andar 'this' keyword ka value decide karna.
 *
 * 'this' JavaScript mein tricky hai:
 * - Normal function mein 'this' depends karta hai HOW it was called.
 * - Arrow function mein 'this' apne surrounding scope se aata hai.
 *
 * Agar hum thisArg NAHI dete, toh:
 * - Non-strict mode mein: 'this' global object (window) ban jata hai.
 * - Strict mode mein: 'this' undefined ho jata hai.
 * - Dono mein this.min undefined milega → comparison false hoga → some() returns false.
 *
 * Agar hum thisArg DETE hain, toh:
 * - callback.call(thisArg, ...) 'this' ko thisArg pe set karta hai.
 * - Toh this.min = obj.min = 3 milta hai.
 *
 * Real-world example:
 *   const config = { min: 3 };
 *   [1, 2, 3, 4, 5].some(function (n) {
 *       return n > this.min;  // 'this' = config
 *   }, config);
 *   // Result: true — kyunki 4 aur 5 dono > 3 hain
 *
 * Yaad rakho: thisArg sirf normal (non-arrow) functions pe asar karta hai.
 * - Arrow functions apna 'this' nahi lete, woh surrounding scope se lete hain.
 * - So arrow function ke saath thisArg pass karna koi farak nahi padta.
 */

/* Demo: thisArg normal function ke saath */
const config1 = { min: 3 };
const normalFnResult = [1, 2, 3, 4, 5].mySome(function (n) {
    return n > this.min;  // 'this' = config1
}, config1);
console.log(normalFnResult); // ✅ true

/* Demo: thisArg arrow function ke saath — farak nahi padta */
const config2 = { min: 3 };
const arrowFnResult = [1, 2, 3, 4, 5].mySome((n) => {
    return n > config2.min;  // arrow function 'this' ignore karta hai
}, config2);
console.log(arrowFnResult); // ✅ true — lekin config2 directly use kiya

/**
 * some() vs every() — quick comparison:
 *
 * some():
 * - Returns true agar KAM SE KAM EK element match kare.
 * - "Koi ek bhi?" — OR jaisa.
 * - Pehle true match pe stop ho jata hai.
 * - Empty array → false.
 *
 * every():
 * - Returns true agar SAARE elements match kare.
 * - "Sab pass?" — AND jaisa.
 * - Pehle false match pe stop ho jata hai.
 * - Empty array → true (kyunki koi fail nahi hua).
 *
 * some() = "koi ek bhi pass?"
 * every() = "sab pass?"
 */


In [ ]:
/**
 * Final Code:
 * 1. Type check — non-function callback pe TypeError throw hota hai
 * 2. thisArg — 'this' sahi se callback ke andar bind hota hai
 * 3. Sparse arrays — holes skip hote hain, callback call nahi hota
 * 4. null/undefined check — this null ya undefined na ho
 *
 * some() = "koi ek bhi pass?" — ek match milte hi true return karo
 */

Array.prototype.mySome = function (callback, thisArg) {
  if (this == null) {
    throw new TypeError('Array.prototype.some called on null or undefined');
  }

  if (typeof callback !== 'function') {
    throw new TypeError(callback + ' is not a function');
  }

  for (let i = 0; i < this.length; i++) {
    if (i in this) {
      if (callback.call(thisArg, this[i], i, this)) {
        return true;
      }
    }
  }

  return false;
};

In [ ]:
/**
 * Every Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.every ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain every() kaise kaam karta hai:
 * - every() check karta hai ki array ke SAARE elements condition satisfy karte hain ya nahi.
 * - Agar saare elements match kare → true return karo.
 * - Agar ek bhi element match na kare → false return karo aur loop turant stop.
 *
 * Simple words mein: "Kya sabhi elements condition pass karte hain?"
 *
 * Important points:
 * 1. every() boolean return karta hai (true ya false).
 * 2. Pehla FALSE match milte hi loop STOP ho jata hai.
 * 3. Yeh NAYA array NAHI banata.
 * 4. Original array change nahi hota.
 *
 * Callback function ko 3 cheezein milti hain:
 * 1. currentValue  : current element
 * 2. index (optional) : element ka position
 * 3. array (optional) : original array jis pe every() call hua
 *
 * Very important rule:
 * - Callback ko true ya false return karna hota hai.
 * - true  → abhi tak sab pass, aage badho
 * - false → ek fail mil gaya, loop stop, every() returns false
 *
 * every() ka second argument hota hai 'thisArg' —
 * yeh callback ke andar 'this' ko control karta hai.
 *
 * Syntax: array.every(callback, thisArg)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.myEvery = function (callback) {
    for (let i = 0; i < this.length; i++) {
        /* Agar callback false return kare, toh false return karo aur loop roko */
        if (!callback(this[i], i, this)) {
            return false;
        }
    }

    /* Agar sab elements pass ho gaye, toh true return karo */
    return true;
};

/* Test 1: Simple check — kya saare even numbers hain? */
const numbers = [2, 4, 6, 8];

const allEven = numbers.myEvery(function (number) {
    return number % 2 === 0;
});

console.log(allEven); // ✅ true

/**
 * Step-by-step flow:
 *
 * numbers = [2, 4, 6, 8]
 *
 * Check 0 → 2 % 2 === 0 ✅ → aage badho
 * Check 1 → 4 % 2 === 0 ✅ → aage badho
 * Check 2 → 6 % 2 === 0 ✅ → aage badho
 * Check 3 → 8 % 2 === 0 ✅ → aage badho
 * Loop khatam → sab pass → return true
 */

/* Test 2: Ek bhi fail ho toh false */
const mixed = [2, 4, 5, 8];

const allEven2 = mixed.myEvery((num) => num % 2 === 0);
console.log(allEven2); // ✅ false

/**
 * mixed = [2, 4, 5, 8]
 * Check 0 → 2 % 2 === 0 ✅ → aage badho
 * Check 1 → 4 % 2 === 0 ✅ → aage badho
 * Check 2 → 5 % 2 === 0 ❌ → fail mil gaya, loop stop, return false
 */

/* Test 3: Empty array — hamesha true (kyunki koi fail nahi hua) */
const empty = [].myEvery((x) => x > 0);
console.log(empty); // ✅ true

/**
 * every() vs some() vs find() ka difference:
 *
 * every():
 * - Returns true agar SAARE elements match kare.
 * - "Sab pass?" — AND operation jaisa.
 * - Pehle false pe stop.
 *
 * some():
 * - Returns true agar KAM SE KAM EK element match kare.
 * - "Koi ek bhi?" — OR operation jaisa.
 * - Pehle true pe stop.
 *
 * find():
 * - Returns the matching ELEMENT (not boolean).
 * - Pehle match pe stop.
 *
 * every() = "sab pass?"
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 4 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Callback nahi diya gaya
 * - Agar koi [1, 2, 3].myEvery() bina argument ke call kare,
 *   toh 'callback' undefined ban jata hai.
 * - undefined() ko call karna allowed nahi — JS throw karta hai:
 *   TypeError: callback is not a function
 */

const failed1 = [1, 2, 3].myEvery();  /* TypeError: callback is not a function */

/**
 * FAILURE 2 — thisArg ignore ho raha hai
 *
 * Pehle samajhte hain thisArg kya hai:
 * - thisArg ek second argument hai jo every() ko pass hota hai.
 * - Iska kaam: callback ke ANDAR 'this' ka value set karna.
 * - Matlab callback ke andar jab hum 'this' likhte hain,
 *   toh woh thisArg object ko point karega.
 *
 * Example:
 *   const obj = { min: 0 };
 *   [1, 2, 3, 4, 5].myEvery(function (num) {
 *       return num > this.min;  // yahan 'this' = obj hona chahiye
 *   }, obj);
 *
 * - Hamara polyfill thisArg ko accept hi nahi karta!
 * - Toh 'this' ka default value lagta hai (undefined ya global).
 * - Iska matlab this.min undefined hai.
 * - num > undefined → hamesha false aata hai.
 * - Result: false (pehle element pe fail) instead of expected true
 */

const obj = { min: 0 };

const failed2 = [1, 2, 3, 4, 5].myEvery(function (num) {
    return num > this.min;
}, obj);

console.log(failed2); // ❌ false — should be true

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta,
 *   undefined value nahi hai.
 *
 * - Native every() holes ko SKIP karta hai — callback unke liye nahi chalta.
 *
 * - Hamara polyfill blindly 0 se length-1 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, sparse[1] undefined padta hai, callback phir bhi chalta hai.
 * - Agar callback undefined ke liye false return kare, toh false
 *   return ho jata hai — yeh galat hai.
 *
 * Result: false (wrong fail on hole) instead of expected true
 */

/* Native every() skips holes — callback call nahi karta */
const sparse = [1, , 3];

const failed3 = sparse.myEvery((x) => x !== undefined);
console.log(failed3); // ❌ false — native every() skips hole, gives true

/**
 * FAILURE 4 — null/undefined pe call karna
 * - Agar koi null ya undefined pe every() call kare:
 *     null.myEvery(...)
 *     undefined.myEvery(...)
 * - Hamara polyfill crash ho jayega — TypeError.
 * - Native every() bhi TypeError throw karta hai, par clear message ke saath.
 * - Humein bhi check karna chahiye ki 'this' null ya undefined na ho.
 */


In [ ]:
/**
 * Edge Cases 1, 2 & 4 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar callback function nahi hai toh?
 * - Agar koi myEvery() bina argument ke call kare, ya null ya number de,
 *   toh code unexpectedly crash ho jayega.
 * - Native every() clear TypeError throw karta hai, humein bhi karna chahiye.
 *
 * Interviewer: thisArg ka issue?
 *
 * You: Haan, basic version second argument ignore karta hai.
 * - Mujhe thisArg ko callback ke andar 'this' banana hai.
 * - Iske liye callback.call(thisArg, ...) use karunga,
 *   direct call ki jagah.
 *
 * Interviewer: null/undefined pe call karna?
 *
 * You: Haan, agar 'this' null ya undefined hai, toh TypeError throw karna chahiye.
 */

/* Step 2: Type check + thisArg binding + null check */
Array.prototype.myEvery = function (callback, thisArg) {
    /* Edge Case 4: this null ya undefined nahi hona chahiye */
    if (this == null) {
        throw new TypeError('Array.prototype.every called on null or undefined');
    }

    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
        if (!callback.call(thisArg, this[i], i, this)) {
            return false;
        }
    }

    return true;
};

/* Test 1: No callback — TypeError aana chahiye */
try {
    [1, 2, 3].myEvery();
} catch (e) {
    console.log(e.message); // ✅ "undefined is not a function"
}

/* Test 2: null as callback — TypeError aana chahiye */
try {
    [1, 2, 3].myEvery(null);
} catch (e) {
    console.log(e.message); // ✅ "null is not a function"
}

/* Test 3: thisArg ab sahi kaam karta hai */
const obj2 = { min: 0 };
const fixedResult = [1, 2, 3, 4, 5].myEvery(function (num) {
    return num > this.min;  // 'this' = obj2, this.min = 0
}, obj2);
console.log(fixedResult); // ✅ true

/* Test 4: null pe call — TypeError aana chahiye */
try {
    Array.prototype.myEvery.call(null, (x) => x > 0);
} catch (e) {
    console.log(e.message); // ✅ "Array.prototype.every called on null or undefined"
}

/* Test 5: Empty array — hamesha true (kyunki koi fail nahi hua) */
const emptyResult = [].myEvery((x) => x > 0);
console.log(emptyResult); // ✅ true


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 3 edge cases fix ho gaye:
 * a. Type check — non-function callbacks ab TypeError dete hain.
 * b. thisArg — 'this' ab sahi se callback ke andar bind hota hai.
 * c. null/undefined check — ab clear TypeError deta hai.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays abhi bhi broken
 * - Loop abhi bhi blindly 0 se length-1 tak chalta hai, koi hole check nahi.
 *
 * [1, , 3] ke liye every(x => x !== undefined):
 *   i = 0 → this[0] = 1     → callback chalta → 1 !== undefined → true ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → callback undefined ke saath phir bhi chalta hai
 *           → undefined !== undefined → false → returns false  ❌
 *
 * Result:   false
 * Expected: true (hole skip hona chahiye, sirf 1 aur 3 check honge, dono !== undefined)
 *
 * Problem: Hum index 1 pe callback call kar rahe hain,
 * jabki woh hole hai. Native every() kabhi nahi karta.
 *
 * Fix: 'i in this' check use karo — yeh batata hai ki
 * index i array mein ACTUALLY exist karta hai ya nahi.
 * Holes ke liye false return karta hai.
 */

const sparse2 = [1, , 3];
const stillBroken = sparse2.myEvery((x) => x !== undefined);
console.log(stillBroken); // ❌ false — native every() skips hole, gives true


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native every() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myEvery = function (callback, thisArg) {
    /* Edge Case 4: this null ya undefined nahi hona chahiye */
    if (this == null) {
        throw new TypeError('Array.prototype.every called on null or undefined');
    }

    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
            if (!callback.call(thisArg, this[i], i, this)) {
                return false;
            }
        }
    }

    return true;
};

/* Test 1: Sparse array — hole skip hoga */
const sparse3 = [1, , 3];
const sparseResult = sparse3.myEvery((x) => x !== undefined);
console.log(sparseResult); // ✅ true — kyunki hole skip ho gaya, sirf 1 aur 3 check hue

/* Test 2: Normal (dense) array abhi bhi theek chalega */
const dense = [2, 4, 6, 8];
const denseResult = dense.myEvery((x) => x % 2 === 0);
console.log(denseResult); // ✅ true

/* Test 3: thisArg abhi bhi kaam karta hai */
const obj3 = { min: 0 };
const thisArgResult = [1, 2, 3, 4, 5].myEvery(function (num) {
    return num > this.min;
}, obj3);
console.log(thisArgResult); // ✅ true

/* Test 4: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myEvery("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 5: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.myEvery((x) => x > 0);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native every() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myEvery = function (callback, thisArg) {
    /* Edge Case 4: this null ya undefined nahi hona chahiye */
    if (this == null) {
        throw new TypeError('Array.prototype.every called on null or undefined');
    }

    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
            if (!callback.call(thisArg, this[i], i, this)) {
                return false;
            }
        }
    }

    return true;
};

/* Test 1: Sparse array — hole skip hoga */
const sparse3 = [1, , 3];
const sparseResult = sparse3.myEvery((x) => x !== undefined);
console.log(sparseResult); // ✅ true — kyunki hole skip ho gaya, sirf 1 aur 3 check hue

/* Test 2: Normal (dense) array abhi bhi theek chalega */
const dense = [2, 4, 6, 8];
const denseResult = dense.myEvery((x) => x % 2 === 0);
console.log(denseResult); // ✅ true

/* Test 3: thisArg abhi bhi kaam karta hai */
const obj3 = { min: 0 };
const thisArgResult = [1, 2, 3, 4, 5].myEvery(function (num) {
    return num > this.min;
}, obj3);
console.log(thisArgResult); // ✅ true

/* Test 4: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myEvery("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 5: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.myEvery((x) => x > 0);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * thisArg Concept — Detail Mein (Hinglish)
 */

/**
 * thisArg kya hai?
 * - thisArg every() ka second argument hai.
 * - Iska kaam: callback ke andar 'this' keyword ka value decide karna.
 *
 * 'this' JavaScript mein tricky hai:
 * - Normal function mein 'this' depends karta hai HOW it was called.
 * - Arrow function mein 'this' apne surrounding scope se aata hai.
 *
 * Agar hum thisArg NAHI dete, toh:
 * - Non-strict mode mein: 'this' global object (window) ban jata hai.
 * - Strict mode mein: 'this' undefined ho jata hai.
 * - Dono mein this.min undefined milega → comparison false hoga → every() returns false.
 *
 * Agar hum thisArg DETE hain, toh:
 * - callback.call(thisArg, ...) 'this' ko thisArg pe set karta hai.
 * - Toh this.min = obj.min = 0 milta hai.
 *
 * Real-world example:
 *   const config = { min: 0 };
 *   [1, 2, 3, 4, 5].every(function (n) {
 *       return n > this.min;  // 'this' = config
 *   }, config);
 *   // Result: true — kyunki saare numbers > 0 hain
 *
 * Yaad rakho: thisArg sirf normal (non-arrow) functions pe asar karta hai.
 * - Arrow functions apna 'this' nahi lete, woh surrounding scope se lete hain.
 * - So arrow function ke saath thisArg pass karna koi farak nahi padta.
 */

/* Demo: thisArg normal function ke saath */
const config1 = { min: 0 };
const normalFnResult = [1, 2, 3, 4, 5].myEvery(function (n) {
    return n > this.min;  // 'this' = config1
}, config1);
console.log(normalFnResult); // ✅ true

/* Demo: thisArg arrow function ke saath — farak nahi padta */
const config2 = { min: 0 };
const arrowFnResult = [1, 2, 3, 4, 5].myEvery((n) => {
    return n > config2.min;  // arrow function 'this' ignore karta hai
}, config2);
console.log(arrowFnResult); // ✅ true — lekin config2 directly use kiya

/**
 * every() vs some() — quick comparison:
 *
 * every():
 * - Returns true agar SAARE elements match kare.
 * - "Sab pass?" — AND jaisa.
 * - Pehle false match pe stop ho jata hai.
 * - Empty array → true (kyunki koi fail nahi hua).
 *
 * some():
 * - Returns true agar KAM SE KAM EK element match kare.
 * - "Koi ek bhi?" — OR jaisa.
 * - Pehle true match pe stop ho jata hai.
 * - Empty array → false (kyunki koi pass nahi hua).
 *
 * every() = "sab pass?"
 * some()   = "koi ek bhi pass?"
 *
 * Empty array behavior:
 * - [].every() → true (vacuously true — koi fail nahi hua)
 * - [].some()  → false (koi pass nahi hua)
 */


In [ ]:
/**
 * Final Code:
 * 1. Type check — non-function callback pe TypeError throw hota hai
 * 2. thisArg — 'this' sahi se callback ke andar bind hota hai
 * 3. Sparse arrays — holes skip hote hain, callback call nahi hota
 * 4. null/undefined check — this null ya undefined na ho
 *
 * every() = "sab pass?" — ek bhi fail milte hi false return karo
 */

Array.prototype.myEvery = function (callback, thisArg) {
  if (this == null) {
    throw new TypeError('Array.prototype.every called on null or undefined');
  }

  if (typeof callback !== 'function') {
    throw new TypeError(callback + ' is not a function');
  }

  for (let i = 0; i < this.length; i++) {
    if (i in this) {
      if (!callback.call(thisArg, this[i], i, this)) {
        return false;
      }
    }
  }

  return true;
};


In [ ]:
/**
 * Flat Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.flat ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain flat() kaise kaam karta hai:
 * - flat() array ke andar ke sub-arrays ko flatten kar deta hai.
 * - Matlab nested arrays ko ek level upar laata hai.
 * - Yeh ek NAYA array return karta hai — original array change nahi hota.
 *
 * Simple example:
 *   [1, [2, 3], 4].flat()       → [1, 2, 3, 4]  (depth 1)
 *   [1, [2, [3]]].flat()        → [1, 2, [3]]   (sirf 1 level)
 *   [1, [2, [3]]].flat(2)       → [1, 2, 3]     (2 levels)
 *   [1, [2, [3]]].flat(Infinity) → [1, 2, 3]    (completely flat)
 *
 * Important points:
 * 1. flat() NAYA array return karta hai.
 * 2. Default depth = 1 hai (agar argument na diya jaye).
 * 3. Original array change nahi hota.
 * 4. depth = 0 ka matlab koi flattening nahi — sirf shallow copy.
 * 5. depth = Infinity ka matlab completely flat (saare levels).
 *
 * flat() ka argument hota hai 'depth' —
 * yeh batata hai kitne levels tak flatten karna hai.
 *
 * Syntax: array.flat(depth)
 */

/* Step 1: Basic version — sirf depth 1, edge cases baad mein */
Array.prototype.myFlat = function () {
    const result = [];

    for (let i = 0; i < this.length; i++) {
        /* Agar element ek array hai, toh uske elements spread karo */
        if (Array.isArray(this[i])) {
            for (let j = 0; j < this[i].length; j++) {
                result.push(this[i][j]);
            }
        } else {
            /* Agar array nahi hai, toh directly push karo */
            result.push(this[i]);
        }
    }

    return result;
};

/* Test 1: Simple flat — depth 1 */
const arr1 = [1, [2, 3], 4];
console.log(arr1.myFlat()); // ✅ [1, 2, 3, 4]

/* Test 2: Nested arrays — sirf 1 level flatten hoga */
const arr2 = [1, [2, [3, 4]]];
console.log(arr2.myFlat()); // ✅ [1, 2, [3, 4]] — andar wala array nahi flat hua

/**
 * forEach vs map vs filter vs reduce vs flat ka difference:
 *
 * forEach() → sirf loop chalata hai, kuch return nahi karta
 * map()     → har element ko transform karta hai, naya array deta hai
 * filter()  → unwanted elements hatata hai, naya array deta hai
 * reduce()  → saare elements ko combine karke EK value deta hai
 * flat()    → nested arrays ko flatten karta hai, naya array deta hai
 *
 * flat() = "nested arrays ko ek level upar laao"
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 3 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Depth parameter ignore ho raha hai
 * - Hamara polyfill hamesha sirf 1 level flatten karta hai.
 * - Agar user 2, 3 ya Infinity depth pass kare, toh bhi sirf 1 level flat hoga.
 *
 * Example:
 *   [1, [2, [3, [4]]]].myFlat(2)
 *   Expected: [1, 2, 3, [4]]  (2 levels)
 *   Got:      [1, 2, [3, [4]]] (sirf 1 level) ❌
 *
 * - Native flat() depth argument accept karta hai aur
 *   utne levels tak flatten karta hai.
 */

const failed1 = [1, [2, [3, [4]]]].myFlat(2);
console.log(failed1); // ❌ [1, 2, [3, [4]]] — should be [1, 2, 3, [4]]

/**
 * FAILURE 2 — Deep nesting handle nahi ho rahi
 * - Humein RECURSION use karna padega — agar element array hai
 *   aur depth bach rahi hai, toh uspe myFlat(depth - 1) call karo.
 * - Hamara basic version sirf ek level jaata hai, uske aage nahi.
 *
 * Fix: recursion use karo — har sub-array pe myFlat(depth - 1) call karo.
 */

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta.
 *
 * - Native flat() holes ko REMOVE karta hai — yeh ek special
 *   behaviour hai flat() ka jo baaki methods se alag hai.
 *   map/filter/forEach holes ko skip karte hain,
 *   lekin flat() holes ko completely remove kar deta hai.
 *
 *   [1, , 3].flat() → [1, 3] — hole gayab ho gaya!
 *
 * - Hamara polyfill blindly 0 se length-1 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, this[1] undefined padta hai, aur undefined push ho jata hai.
 *
 * Result: [1, undefined, 3] instead of expected [1, 3]
 */

const sparse = [1, , 3];
const failed3 = sparse.myFlat();
console.log(failed3); // ❌ [1, undefined, 3] — native flat() gives [1, 3]

/**
 * FAILURE 4 — Negative depth
 * - Agar koi negative depth pass kare:
 *     [1, [2, 3]].myFlat(-1)
 * - Native flat() negative depth ko 0 treat karta hai
 *   (koi flattening nahi, sirf shallow copy).
 * - Hamara polyfill negative depth ko ignore karta hai aur
 *   1 level flatten kar deta hai — galat!
 */

const failed4 = [1, [2, 3]].myFlat(-1);
console.log(failed4); // ❌ [1, 2, 3] — should be [1, [2, 3]] (depth 0 = no flatten)


In [ ]:
/**
 * Edge Cases 1, 2 & 4 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — depth parameter handle karna hai.
 * - Default depth = 1 hona chahiye (agar argument na diya jaye).
 * - Agar depth 0 hai, toh koi flattening nahi — sirf shallow copy.
 * - Agar depth Infinity hai, toh completely flatten.
 * - Agar depth negative hai, toh 0 treat karo.
 *
 * Interviewer: Deep nesting ka issue?
 *
 * You: Haan, recursion use karunga.
 * - Agar element array hai AUR depth > 0 hai,
 *   toh us element pe myFlat(depth - 1) call karo.
 * - Har recursive call depth ko 1 kam kar degi.
 * - Jab depth 0 ho jayega, flattening stop ho jayegi.
 */

/* Step 2: Depth parameter + recursion + negative check */
Array.prototype.myFlat = function (depth = 1) {
    const result = [];

    /* Edge Case 4: negative depth ko 0 treat karo */
    if (depth < 0) {
        depth = 0;
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 1 & 2: agar element array hai aur depth > 0, toh recursively flatten */
        if (Array.isArray(this[i]) && depth > 0) {
            /* Recursion: sub-array pe myFlat(depth - 1) call karo */
            const flattened = this[i].myFlat(depth - 1);
            for (let j = 0; j < flattened.length; j++) {
                result.push(flattened[j]);
            }
        } else {
            /* Agar array nahi hai ya depth 0 hai, toh directly push karo */
            result.push(this[i]);
        }
    }

    return result;
};

/* Test 1: Default depth (1) */
const arr1 = [1, [2, 3], 4];
console.log(arr1.myFlat()); // ✅ [1, 2, 3, 4]

/* Test 2: Depth 2 — 2 levels flatten */
const arr2 = [1, [2, [3, [4]]]];
console.log(arr2.myFlat(2)); // ✅ [1, 2, 3, [4]]

/* Test 3: Infinity — completely flat */
const arr3 = [1, [2, [3, [4, [5]]]]];
console.log(arr3.myFlat(Infinity)); // ✅ [1, 2, 3, 4, 5]

/* Test 4: Depth 0 — no flattening, sirf copy */
const arr4 = [1, [2, 3]];
console.log(arr4.myFlat(0)); // ✅ [1, [2, 3]]

/* Test 5: Negative depth — treated as 0 */
const arr5 = [1, [2, 3]];
console.log(arr5.myFlat(-1)); // ✅ [1, [2, 3]]

/**
 * Recursion ka flow samajhte hain:
 *
 * [1, [2, [3]]].myFlat(2)
 *
 * i = 0 → this[0] = 1 (not array) → push 1
 * i = 1 → this[1] = [2, [3]] (array, depth=2 > 0)
 *         → call [2, [3]].myFlat(1)
 *              i = 0 → 2 (not array) → push 2
 *              i = 1 → [3] (array, depth=1 > 0)
 *                      → call [3].myFlat(0)
 *                           i = 0 → 3 (array, but depth=0)
 *                                  → push 3 (no flatten)
 *                      → return [3]
 *              → push 3
 *         → return [2, 3]
 *         → push 2, push 3
 * → return [1, 2, 3] ✅
 */


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 2 edge cases fix ho gaye:
 * a. Depth parameter — recursion se handle ho raha hai.
 * b. Negative depth — 0 treat ho raha hai.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays abhi bhi broken
 * - Loop abhi bhi blindly 0 se length-1 tak chalta hai, koi hole check nahi.
 *
 * [1, , 3] ke liye flat():
 *   i = 0 → this[0] = 1     → push 1                  ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → undefined push ho jata hai                ❌
 *   i = 2 → this[2] = 3     → push 3                   ✅
 *
 * Result:   [1, undefined, 3]
 * Expected: [1, 3] — hole remove hona chahiye
 *
 * Problem: Hum index 1 pe push kar rahe hain,
 * jabki woh hole hai.
 *
 * Flat() ka special behaviour:
 * - map/filter/forEach holes ko SKIP karte hain (callback nahi chalate).
 * - flat() holes ko REMOVE kar deta hai — result mein nahi laata.
 *
 * Fix: 'i in this' check use karo — yeh batata hai ki
 * index i array mein ACTUALLY exist karta hai ya nahi.
 * Holes ke liye false return karta hai.
 * Agar hole hai, toh push hi mat karo.
 */

const sparse2 = [1, , 3];
const stillBroken = sparse2.myFlat();
console.log(stillBroken); // ❌ [1, undefined, 3] — native flat() gives [1, 3]


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native flat() holes ko REMOVE kar deta hai — yeh flat() ka
 *   special behaviour hai. map/filter holes ko skip karte hain,
 *   lekin flat() unhe result se complately hata deta hai.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Important: 1 in [1, undefined, 3] → true
 * Kyunki index 1 wahan EXIST karta hai — value coincidentally
 * undefined hai, lekin woh hole nahi hai.
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myFlat = function (depth = 1) {
    const result = [];

    /* Edge Case 4: negative depth ko 0 treat karo */
    if (depth < 0) {
        depth = 0;
    }

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo (remove them) */
        if (i in this) {
            /* Edge Case 1 & 2: agar element array hai aur depth > 0, toh recursively flatten */
            if (Array.isArray(this[i]) && depth > 0) {
                const flattened = this[i].myFlat(depth - 1);
                for (let j = 0; j < flattened.length; j++) {
                    result.push(flattened[j]);
                }
            } else {
                /* Agar array nahi hai ya depth 0 hai, toh directly push karo */
                result.push(this[i]);
            }
        }
    }

    return result;
};

/* Test 1: Sparse array — hole remove hoga */
const sparse3 = [1, , 3];
const sparseResult = sparse3.myFlat();
console.log(sparseResult); // ✅ [1, 3] — hole gayab ho gaya

/* Test 2: Sparse array with nested arrays */
const sparseNested = [1, , [2, , 4]];
const sparseNestedResult = sparseNested.myFlat();
console.log(sparseNestedResult); // ✅ [1, 2, 4] — dono holes remove ho gaye

/* Test 3: Normal (dense) array abhi bhi theek chalega */
const dense = [1, [2, 3], [4, [5]]];
console.log(dense.myFlat()); // ✅ [1, 2, 3, 4, [5]]

/* Test 4: Infinity depth with sparse */
const sparseDeep = [1, , [2, , [3, , 4]]];
console.log(sparseDeep.myFlat(Infinity)); // ✅ [1, 2, 3, 4] — all holes removed

/* Test 5: Depth 0 — no flattening, but holes removed */
const sparse0 = [1, , 3];
console.log(sparse0.myFlat(0)); // ✅ [1, 3] — holes removed even at depth 0

/* Test 6: Original array MUTATE nahi hota */
const original = [1, [2, 3], , 4];
original.myFlat();
console.log(original); // ✅ [1, [2, 3], empty, 4] — unchanged

/* Test 7: Explicit undefined is NOT a hole */
const explicitUndef = [1, undefined, 3];
console.log(explicitUndef.myFlat()); // ✅ [1, undefined, 3] — undefined preserved


In [ ]:
/**
 * Depth Concept — Detail Mein (Hinglish)
 */

/**
 * Depth kya hai?
 * - Depth flat() ka argument hai.
 * - Yeh batata hai kitne levels tak arrays ko flatten karna hai.
 *
 * Do cases hain:
 *
 * CASE 1: Depth DIYA gaya
 * - Utne levels tak flatten hoga jitna depth diya gaya.
 * - depth = 1 → sirf 1 level (default)
 * - depth = 2 → 2 levels
 * - depth = Infinity → saare levels (completely flat)
 *
 *   Example: [1, [2, [3, [4]]]]
 *     .flat(1)        → [1, 2, [3, [4]]]     (1 level)
 *     .flat(2)        → [1, 2, 3, [4]]        (2 levels)
 *     .flat(Infinity) → [1, 2, 3, 4]         (all levels)
 *
 * CASE 2: Depth NAHI diya gaya
 * - Default depth = 1 use hota hai.
 * - Sirf 1 level flatten hoga.
 *
 *   Example: [1, [2, [3]]]
 *     .flat()         → [1, 2, [3]]          (default depth = 1)
 *
 * Special cases:
 * - depth = 0 → koi flattening nahi, sirf shallow copy (aur holes remove).
 * - depth = negative → 0 treat hota hai (koi flattening nahi).
 * - depth = undefined → default 1 use hota hai.
 *
 * Recursion ka funda:
 * - Har level pe, agar element array hai aur depth > 0 hai,
 *   toh us element pe myFlat(depth - 1) call karo.
 * - depth har recursive call pe 1 kam hoti hai.
 * - Jab depth 0 ho jaye, toh flattening stop — elements directly push ho jate hain.
 */

/* Demo: depth = 1 (default) */
const d1 = [1, [2, [3, [4]]]];
console.log(d1.myFlat(1)); // ✅ [1, 2, [3, [4]]]

/* Demo: depth = 2 */
const d2 = [1, [2, [3, [4]]]];
console.log(d2.myFlat(2)); // ✅ [1, 2, 3, [4]]

/* Demo: depth = Infinity — completely flat */
const dInf = [1, [2, [3, [4, [5, [6]]]]]];
console.log(dInf.myFlat(Infinity)); // ✅ [1, 2, 3, 4, 5, 6]

/* Demo: depth = 0 — no flatten, just copy (+ holes removed) */
const d0 = [1, [2, 3]];
console.log(d0.myFlat(0)); // ✅ [1, [2, 3]]

/**
 * flat() vs baaki array methods — quick comparison:
 *
 * flat():
 * - NAYA array return karta hai.
 * - Nested arrays ko flatten karta hai.
 * - Callback NAHI leta — sirf depth argument.
 * - thisArg NAHI hota.
 * - Holes ko REMOVE kar deta hai (special behaviour!).
 *
 * map/filter/reduce/forEach/find/some/every:
 * - Callback function lete hain.
 * - thisArg optional argument hota hai.
 * - Holes ko SKIP karte hain (callback nahi chalate).
 *
 * flat() ek alag type ka method hai — yeh transform ya filter
 * nahi karta, yeh STRUCTURE change karta hai.
 */


In [ ]:
/**
 * Final Code:
 * 1. Depth parameter — default 1, negative → 0, Infinity → completely flat
 * 2. Recursion — sub-arrays pe myFlat(depth - 1) call karo
 * 3. Sparse arrays — holes remove ho jate hain (i in this check)
 * 4. Original array change nahi hota
 *
 * flat() = "nested arrays ko ek level upar laao"
 */

Array.prototype.myFlat = function (depth = 1) {
  const result = [];

  if (depth < 0) {
    depth = 0;
  }

  for (let i = 0; i < this.length; i++) {
    if (i in this) {
      if (Array.isArray(this[i]) && depth > 0) {
        const flattened = this[i].myFlat(depth - 1);
        for (let j = 0; j < flattened.length; j++) {
          result.push(flattened[j]);
        }
      } else {
        result.push(this[i]);
      }
    }
  }

  return result;
};


In [ ]:
/**
 * FlatMap Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.flatMap ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain flatMap() kaise kaam karta hai:
 * - flatMap() do kaam ek saath karta hai:
 *   1. map() ki tarah har element pe callback chalata hai.
 *   2. phir result ko depth 1 tak flatten karta hai.
 *
 * Simple words mein: flatMap = map() + flat(1)
 *
 * Example:
 *   [1, 2, 3].flatMap(x => [x, x * 2])
 *   Step 1 (map): [[1, 2], [2, 4], [3, 6]]
 *   Step 2 (flat): [1, 2, 2, 4, 3, 6]
 *
 * Important points:
 * 1. flatMap() NAYA array return karta hai.
 * 2. Yeh hamesha sirf 1 level flatten karta hai (depth fixed to 1).
 * 3. Original array change nahi hota.
 *
 * Callback function ko 3 cheezein milti hain:
 * 1. currentValue  : current element
 * 2. index (optional) : element ka position
 * 3. array (optional) : original array jis pe flatMap() call hua
 *
 * flatMap() ka second argument hota hai 'thisArg' —
 * yeh callback ke andar 'this' ko control karta hai.
 *
 * Syntax: array.flatMap(callback, thisArg)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.myFlatMap = function (callback) {
    /* Pehle map karo, phir 1 level flat karo */
    const mappedArray = [];

    for (let i = 0; i < this.length; i++) {
        mappedArray.push(callback(this[i], i, this));
    }

    /* Ab is mapped array ko flat(1) karo */
    const result = [];
    for (let i = 0; i < mappedArray.length; i++) {
        if (Array.isArray(mappedArray[i])) {
            for (let j = 0; j < mappedArray[i].length; j++) {
                result.push(mappedArray[i][j]);
            }
        } else {
            result.push(mappedArray[i]);
        }
    }

    return result;
};

/* Test 1: Simple flatMap */
const numbers = [1, 2, 3];
const result = numbers.myFlatMap((num) => [num, num * 2]);
console.log(result); // ✅ [1, 2, 2, 4, 3, 6]

/**
 * Step-by-step flow:
 *
 * Step 1 (map): [1, 2, 3] → [[1, 2], [2, 4], [3, 6]]
 * Step 2 (flat): [[1, 2], [2, 4], [3, 6]] → [1, 2, 2, 4, 3, 6]
 */

/* Test 2: Agar callback array return na kare, toh map jaisa behave karega */
const noArray = [1, 2, 3].myFlatMap((x) => x * 2);
console.log(noArray); // ✅ [2, 4, 6]

/**
 * flatMap vs map vs flat ka difference:
 *
 * map():
 * - Callback ek element return karta hai, naye array mein jaata hai.
 * - Agar callback array return kare, toh array of arrays milta hai.
 *
 * flat():
 * - Existing array ke nested arrays ko flatten karta hai.
 * - Koi callback nahi hota, sirfl depth argument.
 *
 * flatMap():
 * - Pehle map karta hai (callback chalata hai).
 * - Phir result ko 1 level flatten karta hai.
 * - Agar map non-array return kare, toh flat ka koi asar nahi padta.
 *
 * flatMap() = "map() + flat(1)"
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 3 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Callback nahi diya gaya
 * - Agar koi [1, 2, 3].myFlatMap() bina argument ke call kare,
 *   toh 'callback' undefined ban jata hai.
 * - undefined() ko call karna allowed nahi — JS throw karta hai:
 *   TypeError: callback is not a function
 */

const failed1 = [1, 2, 3].myFlatMap();  /* TypeError: callback is not a function */

/**
 * FAILURE 2 — thisArg ignore ho raha hai
 *
 * Pehle samajhte hain thisArg kya hai:
 * - thisArg ek second argument hai jo flatMap() ko pass hota hai.
 * - Iska kaam: callback ke ANDAR 'this' ka value set karna.
 * - Matlab callback ke andar jab hum 'this' likhte hain,
 *   toh woh thisArg object ko point karega.
 *
 * Example:
 *   const obj = { multiplier: 2 };
 *   [1, 2, 3].myFlatMap(function (num) {
 *       return [num, num * this.multiplier];  // yahan 'this' = obj hona chahiye
 *   }, obj);
 *
 * - Hamara polyfill thisArg ko accept hi nahi karta!
 * - Toh 'this' ka default value lagta hai (undefined ya global).
 * - Iska matlab this.multiplier undefined hai.
 * - num * undefined = NaN aata hai.
 * - Result: [1, NaN, 2, NaN, 3, NaN] instead of [1, 2, 2, 4, 3, 6]
 */

const obj = { multiplier: 2 };

const failed2 = [1, 2, 3].myFlatMap(function (num) {
    return [num, num * this.multiplier];
}, obj);

console.log(failed2); // ❌ [1, NaN, 2, NaN, 3, NaN]

/**
 * FAILURE 3 — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta.
 *
 * - Native flatMap() holes ko SKIP karta hai — callback unke liye nahi chalta.
 *   Result mein holes map ki tarah skip hote hain.
 *
 * - Hamara polyfill blindly 0 se length-1 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, this[1] undefined padta hai, callback phir bhi chalta hai.
 * - Agar callback undefined return kare, toh undefined flatten hoke
 *   result mein aa jata hai — yeh galat hai.
 *
 * Result: [1, undefined, 3] instead of expected [1, 3]
 */

const sparse = [1, , 3];
const failed3 = sparse.myFlatMap((x) => [x]);
console.log(failed3); // ❌ [1, undefined, 3] — native flatMap() gives [1, 3]


In [ ]:
/**
 * Edge Cases 1 & 2 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar callback function nahi hai toh?
 * - Agar koi myFlatMap() bina argument ke call kare, ya null ya number de,
 *   toh code unexpectedly crash ho jayega.
 * - Native flatMap() clear TypeError throw karta hai, humein bhi karna chahiye.
 *
 * Interviewer: thisArg ka issue?
 *
 * You: Haan, basic version second argument ignore karta hai.
 * - Mujhe thisArg ko callback ke andar 'this' banana hai.
 * - Iske liye callback.call(thisArg, ...) use karunga,
 *   direct call ki jagah.
 */

/* Step 2: Type check + thisArg binding fix */
Array.prototype.myFlatMap = function (callback, thisArg) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    /* Pehle map karo thisArg ke saath, phir 1 level flat karo */
    const mappedArray = [];

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
        mappedArray.push(callback.call(thisArg, this[i], i, this));
    }

    /* Ab is mapped array ko flat(1) karo */
    const result = [];
    for (let i = 0; i < mappedArray.length; i++) {
        if (Array.isArray(mappedArray[i])) {
            for (let j = 0; j < mappedArray[i].length; j++) {
                result.push(mappedArray[i][j]);
            }
        } else {
            result.push(mappedArray[i]);
        }
    }

    return result;
};

/* Test 1: No callback — TypeError aana chahiye */
try {
    [1, 2, 3].myFlatMap();
} catch (e) {
    console.log(e.message); // ✅ "undefined is not a function"
}

/* Test 2: null as callback — TypeError aana chahiye */
try {
    [1, 2, 3].myFlatMap(null);
} catch (e) {
    console.log(e.message); // ✅ "null is not a function"
}

/* Test 3: thisArg ab sahi kaam karta hai */
const obj2 = { multiplier: 2 };
const fixedResult = [1, 2, 3].myFlatMap(function (num) {
    return [num, num * this.multiplier];  // 'this' = obj2
}, obj2);
console.log(fixedResult); // ✅ [1, 2, 2, 4, 3, 6]


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 2 edge cases fix ho gaye:
 * a. Type check — non-function callbacks ab TypeError dete hain.
 * b. thisArg — 'this' ab sahi se callback ke andar bind hota hai.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays abhi bhi broken
 * - Loop abhi bhi blindly 0 se length-1 tak chalta hai, koi hole check nahi.
 *
 * [1, , 3] ke liye flatMap(x => [x]):
 *   i = 0 → this[0] = 1     → callback chalta → returns [1] → flatten → push 1  ✅
 *   i = 1 → this[1] = hole  → undefined padta hai
 *           → callback undefined ke saath phir bhi chalta hai
 *           → returns [undefined] → flatten → push undefined              ❌
 *   i = 2 → this[2] = 3     → callback chalta → returns [3] → flatten → push 3  ✅
 *
 * Result:   [1, undefined, 3]
 * Expected: [1, 3] — hole skip hona chahiye
 *
 * Problem: Hum index 1 pe callback call kar rahe hain,
 * jabki woh hole hai. Native flatMap() kabhi nahi karta.
 *
 * Fix: 'i in this' check use karo — yeh batata hai ki
 * index i array mein ACTUALLY exist karta hai ya nahi.
 * Holes ke liye false return karta hai.
 */

const sparse2 = [1, , 3];
const stillBroken = sparse2.myFlatMap((x) => [x]);
console.log(stillBroken); // ❌ [1, undefined, 3] — native flatMap() gives [1, 3]


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native flatMap() holes ko skip karta hai — unke liye callback
 *   kabhi call nahi karta. flatMap basically map hai, toh map ki
 *   tarah holes pe callback nahi chalata.
 *
 * Fix: 'i in this' operator use karo.
 *
 * 'i in this' check karta hai ki index i array mein exist karta hai ya nahi:
 *   0 in [1, , 3]  → true   (index 0 exist karta hai)
 *   1 in [1, , 3]  → false  (index 1 hole hai)
 *   2 in [1, , 3]  → true   (index 2 exist karta hai)
 *
 * Toh 'i in this' sahi se "hole" aur "explicit undefined value"
 * mein difference pata lagata hai.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myFlatMap = function (callback, thisArg) {
    /* Edge Case 1: callback function hona chahiye */
    if (typeof callback !== 'function') {
        throw new TypeError(callback + ' is not a function');
    }

    const result = [];

    for (let i = 0; i < this.length; i++) {
        /* Edge Case 3: sparse arrays ke holes skip karo */
        if (i in this) {
            /* Edge Case 2: thisArg ko callback ke andar 'this' banao */
            const mappedValue = callback.call(thisArg, this[i], i, this);

            /* Ab mapped value ko 1 level flatten karo */
            if (Array.isArray(mappedValue)) {
                for (let j = 0; j < mappedValue.length; j++) {
                    result.push(mappedValue[j]);
                }
            } else {
                result.push(mappedValue);
            }
        }
    }

    return result;
};

/* Test 1: Sparse array — hole skip hoga */
const sparse3 = [1, , 3];
const sparseResult = sparse3.myFlatMap((x) => [x]);
console.log(sparseResult); // ✅ [1, 3] — hole skip ho gaya

/* Test 2: Normal (dense) array abhi bhi theek chalega */
const dense = [1, 2, 3];
const denseResult = dense.myFlatMap((x) => [x, x * 2]);
console.log(denseResult); // ✅ [1, 2, 2, 4, 3, 6]

/* Test 3: thisArg abhi bhi kaam karta hai */
const obj3 = { multiplier: 2 };
const thisArgResult = [1, 2, 3].myFlatMap(function (num) {
    return [num, num * this.multiplier];
}, obj3);
console.log(thisArgResult); // ✅ [1, 2, 2, 4, 3, 6]

/* Test 4: Non-function callback pe TypeError abhi bhi aayega */
try {
    [1, 2, 3].myFlatMap("not a function");
} catch (e) {
    console.log(e.message); // ✅ "not a function is not a function"
}

/* Test 5: Original array MUTATE nahi hota */
const original = [1, 2, 3];
original.myFlatMap((x) => [x, x * 2]);
console.log(original); // ✅ [1, 2, 3] — unchanged


In [ ]:
/**
 * flatMap vs map + flat — Detail Mein (Hinglish)
 */

/**
 * flatMap kya hai?
 * - flatMap do operations combine karta hai: map() + flat(1)
 * - Pehle har element pe callback chalta hai.
 * - Phir result ko 1 level flatten karta hai.
 *
 * Important: flatMap sirf 1 level flatten karta hai.
 * - Agar zyada depth chahiye, toh map().flat(n) use karo.
 * - flatMap(2) jaisa kuch nahi hota — depth fix 1 hai.
 *
 * Example:
 *   [1, 2, 3].flatMap(x => [[x, x*2]])
 *   Step 1 (map): [[[1, 2]], [[2, 4]], [[3, 6]]]
 *   Step 2 (flat 1): [[1, 2], [2, 4], [3, 6]]
 *   - Sirf 1 level flat hua, andar wale arrays nahi.
 *
 * Real-world use cases:
 * 1. Words split karna:
 *    ["hello world", "foo bar"].flatMap(s => s.split(' '))
 *    → ["hello", "world", "foo", "bar"]
 *
 * 2. Duplicate elements banana:
 *    [1, 2, 3].flatMap(x => [x, x])
 *    → [1, 1, 2, 2, 3, 3]
 *
 * 3. Filtering + mapping ek saath:
 *    [1, -2, 3, -4].flatMap(x => x > 0 ? [x * 2] : [])
 *    → [2, 6]  (negatives skip, positives double)
 */

/* Demo: Words split */
const sentences = ["hello world", "foo bar"];
const words = sentences.myFlatMap((s) => s.split(' '));
console.log(words); // ✅ ["hello", "world", "foo", "bar"]

/* Demo: Filter + map together */
const nums = [1, -2, 3, -4];
const filtered = nums.myFlatMap((x) => x > 0 ? [x * 2] : []);
console.log(filtered); // ✅ [2, 6]

/* Demo: map + flat vs flatMap */
const arr = [1, 2, 3];

// map + flat(1) — same result as flatMap
const mapThenFlat = arr.map((x) => [x, x * 2]).flat(1);
console.log(mapThenFlat); // ✅ [1, 2, 2, 4, 3, 6]

// flatMap — same result
const flatMapResult = arr.myFlatMap((x) => [x, x * 2]);
console.log(flatMapResult); // ✅ [1, 2, 2, 4, 3, 6]

/**
 * flatMap vs map + flat — quick comparison:
 *
 * flatMap():
 * - Ek hi operation mein map + flat(1) karta hai.
 * - Hamesha depth 1 flatten karta hai.
 * - Memory efficient — do arrays nahi banata.
 *
 * map().flat(1):
 * - Pehle map poora array banata hai.
 * - Phir us array pe flat(1) chalta hai.
 * - Do arrays bante hain — memory thoda zyada use hota hai.
 *
 * Functionally dono same hain, bas flatMap efficient hai.
 */


In [ ]:
/**
 * Final Code:
 * 1. Type check — non-function callback pe TypeError throw hota hai
 * 2. thisArg — 'this' sahi se callback ke andar bind hota hai
 * 3. Sparse arrays — holes skip hote hain, callback call nahi hota
 * 4. Flatten — mapped value agar array hai, toh 1 level flatten karo
 *
 * flatMap() = "map() + flat(1)"
 */

Array.prototype.myFlatMap = function (callback, thisArg) {
  if (typeof callback !== 'function') {
    throw new TypeError(callback + ' is not a function');
  }

  const result = [];

  for (let i = 0; i < this.length; i++) {
    if (i in this) {
      const mappedValue = callback.call(thisArg, this[i], i, this);

      if (Array.isArray(mappedValue)) {
        for (let j = 0; j < mappedValue.length; j++) {
          result.push(mappedValue[j]);
        }
      } else {
        result.push(mappedValue);
      }
    }
  }

  return result;
};

In [ ]:
/**
 * Includes Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.includes ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain includes() kaise kaam karta hai:
 * - includes() check karta hai ki array mein koi specific value hai ya nahi.
 * - Agar value mil gayi → true return karo.
 * - Agar poori array check kar li aur nahi mili → false return karo.
 *
 * Simple words mein: "Yeh value array mein hai ya nahi?"
 *
 * Important points:
 * 1. includes() boolean return karta hai (true ya false).
 * 2. Yeh NAYA array NAHI banata.
 * 3. Original array change nahi hota.
 * 4. Yeh case-sensitive hota hai (strings ke liye).
 * 5. Yeh type-strict check karta hai (jaise ===).
 *
 * includes() do arguments leta hai:
 * 1. searchElement : jo value dhundhni hai
 * 2. fromIndex (optional) : kahan se search shuru karna hai (default 0)
 *
 * Syntax: array.includes(searchElement, fromIndex)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.myIncludes = function (searchElement) {
    for (let i = 0; i < this.length; i++) {
        /* Agar element mil gaya, toh true return karo */
        if (this[i] === searchElement) {
            return true;
        }
    }

    /* Agar nahi mila, toh false */
    return false;
};

/* Test 1: Simple search */
const numbers = [1, 2, 3, 4, 5];

console.log(numbers.myIncludes(3)); // ✅ true
console.log(numbers.myIncludes(6)); // ✅ false

/* Test 2: String search (case-sensitive) */
const words = ['hello', 'world', 'js'];

console.log(words.myIncludes('js'));    // ✅ true
console.log(words.myIncludes('Js'));    // ✅ false (capital J)

/**
 * includes() vs indexOf() ka difference:
 *
 * includes():
 * - Returns true ya false.
 * - NaN ko dhundh sakta hai ([NaN].includes(NaN) → true).
 *
 * indexOf():
 * - Returns index (number) ya -1.
 * - NaN nahi dhundh pata ([NaN].indexOf(NaN) → -1).
 *
 * includes() = "hai ya nahi?"
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 2 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — NaN (Not-a-Number) nahi mil pata
 * - JavaScript mein NaN ek special value hai.
 * - Khass baat yeh hai ki NaN kabhi kisi value ke barabar NAHI hota,
 *   apne aap ke bhi nahi! (NaN === NaN → false)
 *
 * - Hamara polyfill === use karta hai, isliye yeh NaN ko nahi dhundh pata.
 * - Native includes() NaN ko dhundh leta hai.
 *
 * Result: false instead of expected true
 */

const arr = [1, NaN, 3];

const failed1 = arr.myIncludes(NaN);
console.log(failed1); // ❌ false — native includes() gives true

/**
 * FAILURE 2 — fromIndex ignore ho raha hai
 * - includes() ka second argument 'fromIndex' hota hai.
 * - Yeh batata hai ki search kahan se shuru karna hai.
 *
 * Example:
 *   [1, 2, 3, 4].includes(1, 2)
 *   - Yahan 1 ko index 2 se shuru karke dhundhna hai.
 *   - Index 2 pe 3 hai, index 3 pe 4 hai. 1 nahi hai.
 *   - Toh result false aana chahiye.
 *
 * - Hamara polyfill fromIndex accept hi nahi karta!
 * - Yeh hamesha 0 se shuru karta hai, isliye 1 mil jayega.
 *
 * Result: true instead of expected false
 */

const failed2 = [1, 2, 3, 4].myIncludes(1, 2);
console.log(failed2); // ❌ true — native includes() gives false


In [ ]:
/**
 * Edge Cases 1 & 2 Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — NaN ka issue.
 * - === use karne se NaN match nahi hota.
 * - Isliye humein SameValueZero algorithm use karna hoga.
 * - SameValueZero ek simple check hai:
 *   1. Agar dono NaN hain, toh return true.
 *   2. Warna, dono === ke barabar hain ya nahi check karo.
 *
 * Interviewer: fromIndex ka issue?
 *
 * You: Haan, basic version second argument ignore karta hai.
 * - Mujhe fromIndex default 0 set karna hoga.
 * - Aur loop ko fromIndex se shuru karna hoga.
 */

/* SameValueZero check banata hain NaN ko handle karne ke liye */
function isSameValueZero(x, y) {
    if (typeof x === 'number' && typeof y === 'number') {
        // x !== x tabhi true hota hai jab x NaN ho.
        // Same to y !== y. Agar dono NaN hain, toh true.
        return x === y || (x !== x && y !== y);
    }
    return x === y;
}

/* Step 2: fromIndex + NaN (SameValueZero) fix */
Array.prototype.myIncludes = function (searchElement, fromIndex) {
    /* Edge Case 2: fromIndex default 0 hai agar pass nahi kiya gaya */
    if (fromIndex === undefined) {
        fromIndex = 0;
    }

    /* Loop ko fromIndex se shuru karo */
    for (let i = fromIndex; i < this.length; i++) {
        /* Edge Case 1: SameValueZero use karo NaN ko handle karne ke liye */
        if (isSameValueZero(this[i], searchElement)) {
            return true;
        }
    }

    return false;
};

/* Test 1: NaN ab sahi se mil jayega */
const arr2 = [1, NaN, 3];
console.log(arr2.myIncludes(NaN)); // ✅ true

/* Test 2: fromIndex ab kaam karta hai */
console.log([1, 2, 3, 4].myIncludes(1, 2)); // ✅ false (search starts at index 2)

/* Test 3: Default search (0 se) */
console.log([1, 2, 3, 4].myIncludes(1)); // ✅ true


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 2 edge cases fix ho gaye:
 * a. NaN — SameValueZero se handle ho raha hai.
 * b. fromIndex — positive fromIndex loop ko control karta hai.
 * Lekin negative fromIndex aur out-of-bounds fromIndex abhi bhi broken hain.
 */

/**
 * FAILURE — Negative fromIndex handling
 * - Agar user negative fromIndex de:
 *     [1, 2, 3, 4].myIncludes(3, -1)
 * - Native includes() negative index ko end se calculate karta hai.
 *   -1 matlab length - 1 = 4 - 1 = 3. Index 3 se shuru karo.
 *   Index 3 pe 4 hai, index 4 nahi hai. Toh false aana chahiye.
 *
 * - Hamara polyfill loop i = -1 se shuru karta hai.
 * - this[-1] undefined hota hai, phir this[0] = 1, this[1] = 2, this[2] = 3.
 * - 3 mil gaya! Toh true return kar dega. ❌
 *
 * - Agar fromIndex bahut zyada negative ho jaye (e.g. -10)?
 *   - Native includes() use 0 kar leta hai.
 *   - Hamara polyfill i = -10 se shuru karta hai, jo galat hai.
 */

const failed1 = [1, 2, 3, 4].myIncludes(3, -1);
console.log(failed1); // ❌ true — native includes() gives false (starts at index 3)

const failed2 = [1, 2, 3, 4].myIncludes(2, -10);
console.log(failed2); // ✅ true (accidentally works, but internally starts at -10)

/**
 * FAILURE 2 — fromIndex >= length
 * - Agar fromIndex array ki length se bada ya barabar hai:
 *     [1, 2, 3].myIncludes(1, 5)
 * - Native includes() search hi nahi karta, false return karta hai.
 * - Hamara polyfill loop i = 5 se shuru karta hai, loop nahi chalega
 *   (kyunki 5 < 3 false hai), toh false return karega.
 * - Yeh coincidentally toh kaam karega, lekin explicitly handle karna chahiye
 *   kyunki fromIndex NaN ho sakta hai (jise 0 treat karte hain).
 */


In [ ]:
/**
 * Edge Case 3 Fix Karna (Negative & Out-of-bounds fromIndex)
 */

/**
 * Interviewer: Negative fromIndex kaise handle karoge?
 *
 * You: Agar fromIndex negative hai, toh humein array ki length mein add karna hoga.
 * - Naya index = length + fromIndex
 * - Agar naya index bhi negative hai, toh use 0 kar lenge.
 *
 * Example:
 *   [1, 2, 3, 4].includes(3, -1)
 *   length = 4, fromIndex = -1
 *   Naya index = 4 + (-1) = 3. Search starts at index 3.
 *
 * Interviewer: fromIndex length se bada ho toh?
 * - Agar fromIndex >= length hai, toh search karne ka matlab nahi.
 *   Seedha false return karo.
 */

/* Step 3: Negative fromIndex + length check handle karo */
Array.prototype.myIncludes = function (searchElement, fromIndex) {
    /* fromIndex ko number mein convert karo (agar string pass ho) */
    fromIndex = Number(fromIndex);

    /* Agar fromIndex NaN hai (e.g. undefined), toh 0 use karo */
    if (isNaN(fromIndex)) {
        fromIndex = 0;
    }

    /* Edge Case 2: fromIndex >= length → array mein nahi mil sakta */
    if (fromIndex >= this.length) {
        return false;
    }

    /* Edge Case 3: Negative fromIndex ko end se calculate karo */
    if (fromIndex < 0) {
        fromIndex = this.length + fromIndex;
        /* Agar still negative hai, toh 0 se shuru karo */
        if (fromIndex < 0) {
            fromIndex = 0;
        }
    }

    /* Loop ko calculated fromIndex se shuru karo */
    for (let i = fromIndex; i < this.length; i++) {
        if (isSameValueZero(this[i], searchElement)) {
            return true;
        }
    }

    return false;
};

/* Test 1: Negative fromIndex ab sahi kaam karta hai */
console.log([1, 2, 3, 4].myIncludes(3, -1)); // ✅ false (starts at index 3, value is 4)
console.log([1, 2, 3, 4].myIncludes(4, -1)); // ✅ true (starts at index 3, value is 4)

/* Test 2: Bahut zyada negative fromIndex */
console.log([1, 2, 3, 4].myIncludes(2, -10)); // ✅ true (clamped to 0, finds 2)

/* Test 3: fromIndex >= length */
console.log([1, 2, 3].myIncludes(1, 5)); // ✅ false

/* Test 4: NaN search abhi bhi kaam karta hai */
console.log([1, NaN, 3].myIncludes(NaN)); // ✅ true

/* Test 5: Normal search */
console.log([1, 2, 3].myIncludes(2)); // ✅ true


In [ ]:
/**
 * SameValueZero Concept — Detail Mein (Hinglish)
 */

/**
 * SameValueZero kya hai?
 * - includes() value compare karne ke liye ek special algorithm use karta hai:
 *   SameValueZero.
 * - Yeh === (strict equality) jaisa hi hai, bas ek difference hai:
 *   NaN ko NaN ke barabar mana jata hai.
 *
 * === (Strict Equality) behaviour:
 *   1 === 1       → true
 *   'a' === 'a'   → true
 *   NaN === NaN   → false ❌ (JS ka bug/gotcha)
 *   +0 === -0     → true
 *
 * SameValueZero behaviour:
 *   1 === 1       → true
 *   'a' === 'a'   → true
 *   NaN === NaN   → true ✅ (includes() yeh use karta hai)
 *   +0 === -0     → true
 *
 * Kyun zaroori hai?
 * - Agar aapko array mein NaN check karna ho, toh indexOf() fail ho jayega.
 *   [NaN].indexOf(NaN) → -1 (kyunki indexOf === use karta hai)
 * - Isliye ES2016 mein includes() aaya, jo SameValueZero use karta hai.
 *
 * Implementation:
 *   function isSameValueZero(x, y) {
 *       if (typeof x === 'number' && typeof y === 'number') {
 *           return x === y || (x !== x && y !== y); // x !== x means x is NaN
 *       }
 *       return x === y;
 *   }
 */

/* Demo: indexOf vs includes for NaN */
const arr = [1, NaN, 3];

console.log(arr.indexOf(NaN));     // ❌ -1 (cannot find NaN)
console.log(arr.myIncludes(NaN));  // ✅ true (finds NaN using SameValueZero)

/* Demo: +0 and -0 are considered equal */
const zeros = [+0];
console.log(zeros.myIncludes(-0)); // ✅ true (SameValueZero treats them as equal)

/**
 * includes() vs indexOf() — quick comparison:
 *
 * includes():
 * - Returns true / false.
 * - Uses SameValueZero algorithm.
 * - Can find NaN.
 * - Supports fromIndex (negative as well).
 *
 * indexOf():
 * - Returns index number / -1.
 * - Uses === (strict equality).
 * - Cannot find NaN.
 * - Supports fromIndex.
 *
 * Rule of thumb: Agar sirf presence check karni hai, toh includes() use karo.
 * Agar index chahiye, toh indexOf() use karo.
 */


In [ ]:
/**
 * Final Code:
 * 1. fromIndex — default 0, NaN ko 0 treat karo
 * 2. Negative fromIndex — length + fromIndex se calculate karo, clamped to 0
 * 3. Out-of-bounds fromIndex — agar >= length, toh false return karo
 * 4. NaN handling — SameValueZero algorithm use karo
 *
 * includes() = "yeh value array mein hai ya nahi?"
 */

Array.prototype.myIncludes = function (searchElement, fromIndex) {
    fromIndex = Number(fromIndex);

    if (isNaN(fromIndex)) {
        fromIndex = 0;
    }

    if (fromIndex >= this.length) {
        return false;
    }

    if (fromIndex < 0) {
        fromIndex = this.length + fromIndex;
        if (fromIndex < 0) {
            fromIndex = 0;
        }
    }

    for (let i = fromIndex; i < this.length; i++) {
        if (typeof this[i] === 'number' && typeof searchElement === 'number') {
            if (this[i] === searchElement || (this[i] !== this[i] && searchElement !== searchElement)) {
                return true;
            }
        } else if (this[i] === searchElement) {
            return true;
        }
    }

    return false;
};


In [ ]:
/**
 * Join Polyfill (Basic Version)
 */

/**
 * Interviewer: Array.prototype.join ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain join() kaise kaam karta hai:
 * - join() array ke saare elements ko ek single STRING mein convert karta hai.
 * - Elements ke beech mein ek separator lagta hai (jaise comma ya koi aur character).
 * - Agar element string nahi hai (number, object, etc.), toh pehle string mein convert karta hai.
 *
 * Simple words mein: "Array ko string banao, aur beech mein separator daalo"
 *
 * Example:
 *   ['a', 'b', 'c'].join('-') → "a-b-c"
 *   [1, 2, 3].join()         → "1,2,3" (default separator is comma)
 *
 * Important points:
 * 1. join() ek STRING return karta hai.
 * 2. Original array change nahi hota.
 * 3. Default separator ',' (comma) hota hai.
 *
 * join() ka argument hota hai 'separator':
 * - Yeh batata hai elements ke beech mein kya aayega.
 * - Agar pass na kiya jaye, toh ',' use hota hai.
 * - Agar '' (empty string) pass kiya jaye, toh koi gap nahi aayega.
 *
 * Syntax: array.join(separator)
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
Array.prototype.myJoin = function (separator) {
    let result = "";

    for (let i = 0; i < this.length; i++) {
        /* Pehla element ko direct add karo */
        if (i === 0) {
            result += this[i];
        } else {
            /* Baaki elements ke pehle separator add karo */
            result += separator + this[i];
        }
    }

    return result;
};

/* Test 1: Default comma separator */
const arr1 = [1, 2, 3];
console.log(arr1.myJoin()); // ✅ "1,2,3"

/* Test 2: Custom separator */
const arr2 = ['a', 'b', 'c'];
console.log(arr2.myJoin('-')); // ✅ "a-b-c"

/* Test 3: Empty string separator */
const arr3 = ['Hello', 'World'];
console.log(arr3.myJoin('')); // ✅ "HelloWorld"

/**
 * join() ka use cases:
 *
 * 1. URL path banana:
 *    ['users', '123', 'edit'].join('/')
 *    → "users/123/edit"
 *
 * 2. CSV data banana:
 *    ['Aslam', '25', 'Developer'].join(',')
 *    → "Aslam,25,Developer"
 *
 * 3. Sentence banana:
 *    ['Hello', 'World'].join(' ')
 *    → "Hello World"
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 4 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Default separator handling
 * - Agar koi separator pass na kare:
 *     [1, 2, 3].myJoin()
 * - Hamara polyfill `undefined` ko separator bana deta hai.
 * - Phir result: "1undefined2undefined3" aata hai. ❌
 * - Native join() default ',' use karta hai agar separator undefined ho.
 *
 * Result: "1undefined2undefined3" instead of expected "1,2,3"
 */

const failed1 = [1, 2, 3].myJoin();
console.log(failed1); // ❌ "1undefined2undefined3"

/**
 * FAILURE 2 — null aur undefined elements
 * - Agar array mein null ya undefined ho:
 *     ['a', null, 'b'].myJoin('-')
 * - Native join() null aur undefined ko empty string ("") treat karta hai.
 *     Result: "a--b"
 * - Hamara polyfill unhe directly add karta hai.
 *     Result: "a-null-b" ❌
 */

const failed2 = ['a', null, 'b'].myJoin('-');
console.log(failed2); // ❌ "a-null-b" — native join() gives "a--b"

/**
 * FAILURE 3 — Empty array
 * - Agar array khali ho:
 *     [].myJoin('-')
 * - Native join() empty array ke liye "" (empty string) return karta hai.
 * - Hamara polyfill bhi "" return karta hai, kyunki loop nahi chalta.
 * - Yeh coincidentally toh sahi kaam karta hai, lekin explicitly handle karna chahiye.
 */

const failed3 = [].myJoin('-');
console.log(failed3); // ✅ "" (works, but by luck)


In [ ]:
/**
 * Edge Cases Fix Karna
 */

/**
 * Interviewer: Basic version theek hai. Aur edge cases soch sakte ho?
 *
 * You: Pehle — agar separator undefined ho toh?
 * - Default ',' use karna chahiye.
 *
 * Interviewer: null aur undefined elements ka issue?
 *
 * You: Haan, array mein agar null ya undefined value ho, toh usko
 * empty string ("") treat karna chahiye, taaki output string mein
 * "null" ya "undefined" text na aaye.
 */

/* Step 2: Default separator + null/undefined handling fix */
Array.prototype.myJoin = function (separator) {
    /* Edge Case 1: Agar separator undefined hai, toh ',' use karo */
    if (separator === undefined) {
        separator = ',';
    }

    let result = "";

    for (let i = 0; i < this.length; i++) {
        let element = this[i];

        /* Edge Case 2: null aur undefined ko empty string banao */
        if (element === null || element === undefined) {
            element = '';
        } else {
            /* Baaki sab ko string mein convert karo */
            element = String(element);
        }

        /* Pehla element ko direct add karo */
        if (i === 0) {
            result += element;
        } else {
            /* Baaki elements ke pehle separator add karo */
            result += separator + element;
        }
    }

    return result;
};

/* Test 1: Default separator (undefined) */
console.log([1, 2, 3].myJoin()); // ✅ "1,2,3"

/* Test 2: null aur undefined elements */
console.log(['a', null, 'b'].myJoin('-')); // ✅ "a--b"
console.log(['a', undefined, 'b'].myJoin('-')); // ✅ "a--b"

/* Test 3: Objects aur booleans (String conversion) */
console.log([1, true, {}].myJoin(', ')); // ✅ "1, true, [object Object]"

/* Test 4: Empty string separator */
console.log(['H', 'e', 'l', 'l', 'o'].myJoin('')); // ✅ "Hello"


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 2 edge cases fix ho gaye:
 * a. Default separator — undefined hone pe ',' use hota hai.
 * b. null/undefined — empty string treat hote hain.
 * Lekin sparse arrays (holes wale) abhi bhi broken hain.
 */

/**
 * FAILURE — Sparse arrays (arrays with holes)
 * - JavaScript arrays mein "holes" ho sakte hain:
 *     const sparse = [1, , 3];
 * - Yahan index 1 ek HOLE hai — woh exist hi nahi karta.
 *
 * - Native join() holes ko empty string ("") treat karta hai.
 *     [1, , 3].join('-') → "1--3"
 *
 * - Hamara polyfill blindly 0 se length-1 tak loop chalata hai,
 *   koi hole check nahi karta.
 * - i = 1 pe, this[1] undefined padta hai.
 * - Hamara code undefined ko empty string bana deta hai (Edge Case 2 ki wajah se).
 * - Isliye yeh coincidentally toh "1--3" hi dega.
 * - Lekin! Agar separator chhota ho, ya complex logic ho, toh issue aa sakta hai.
 *   Better hai explicitly 'i in this' check karein.
 */

const sparse = [1, , 3];
const stillBroken = sparse.myJoin('-');
console.log(stillBroken); // ✅ "1--3" (works, but by luck, not by design)


In [ ]:
/**
 * Edge Case 3 Fix Karna (Sparse Arrays)
 */

/**
 * Interviewer: Sparse arrays ke baare mein? Holes wale arrays?
 *
 * You: JavaScript mein [1, , 3] mein index 1 pe HOLE hai.
 * - Yeh [1, undefined, 3] ke BARABAR nahi hai.
 *
 * Difference:
 *   [1, undefined, 3] → index 1 EXIST karta hai, value undefined hai
 *   [1, , 3]          → index 1 HOLE hai, exist hi nahi karta
 *
 * - Native join() holes ko bhi empty string ("") treat karta hai.
 *   [1, , 3].join('-') → "1--3"
 *
 * - `i in this` check karta hai ki index exist karta hai ya nahi.
 * - Agar hole ho, toh usko empty string bana do.
 */

/* Step 3: Sparse arrays handle karo */
Array.prototype.myJoin = function (separator) {
    /* Edge Case 1: Agar separator undefined hai, toh ',' use karo */
    if (separator === undefined) {
        separator = ',';
    }

    let result = "";

    for (let i = 0; i < this.length; failures++) {
        let element;

        /* Edge Case 3: sparse arrays ke holes check karo */
        if (i in this) {
            element = this[i];
            /* Edge Case 2: null aur undefined ko empty string banao */
            if (element === null || element === undefined) {
                element = '';
            } else {
                element = String(element);
            }
        } else {
            /* Agar hole hai, toh empty string use karo */
            element = '';
        }

        /* Pehla element ko direct add karo */
        if (i === 0) {
            result += element;
        } else {
            /* Baaki elements ke pehle separator add karo */
            result += separator + element;
        }
    }

    return result;
};

/* Test 1: Sparse array — hole empty string ban jayega */
const sparse3 = [1, , 3];
console.log(sparse3.myJoin('-')); // ✅ "1--3"

/* Test 2: Sparse array with explicit undefined */
const sparse4 = [1, undefined, 3];
console.log(sparse4.myJoin('-')); // ✅ "1--3" (same result for both)

/* Test 3: Normal (dense) array abhi bhi theek chalega */
const dense = ['a', 'b', 'c'];
console.log(dense.myJoin(', ')); // ✅ "a, b, c"

/* Test 4: Empty array — empty string return karega */
console.log([].myJoin('-')); // ✅ ""

/* Test 5: Non-string separator (number) — string mein convert hoga */
console.log([2023, 10, 25].myJoin(0)); // ✅ "2023010025" (0 converts to "0")

/* Test 6: Array as separator */
console.log([1, 2, 3].myJoin([0])); // ✅ "10020" ([0] converts to "0")


In [ ]:
/**
 * Separator Concept — Detail Mein (Hinglish)
 */

/**
 * Separator kya hai?
 * - Separator join() ka argument hai.
 * - Yeh batata hai elements ke beech mein kya aayega.
 *
 * Do cases hain:
 *
 * CASE 1: Separator DIYA gaya
 * - Elements ke beech mein yeh string aayega.
 * - Agar '' (empty string) ho, toh elements bina gap ke jud jate hain.
 *
 *   Example: ['a', 'b', 'c'].join('-')
 *   Result: "a-b-c"
 *
 *   Example: ['a', 'b', 'c'].join('')
 *   Result: "abc"
 *
 * CASE 2: Separator NAHI diya gaya
 * - Default ',' (comma) use hota hai.
 *
 *   Example: ['a', 'b', 'c'].join()
 *   Result: "a,b,c"
 *
 * Special cases:
 * - Agar separator undefined hai, toh ',' (comma) use hota hai.
 * - Agar separator ek array ho, toh usko string mein convert karke use karega.
 * - Agar separator object ho, toh usko "[object Object]" string use karega.
 *
 * Important: Separator length 1 hona zaroori nahi.
 * - Yeh koi bhi string ho sakti hai.
 * - Agar '' (empty string) pass kiya, toh elements bina gap ke jud jate hain.
 */

/* Demo: separator = '-' */
console.log(['2023', '10', '25'].myJoin('-')); // ✅ "2023-10-25"

/* Demo: separator = ' ' (space) */
console.log(['Hello', 'World'].myJoin(' ')); // ✅ "Hello World"

/* Demo: separator = ', ' (comma space) */
console.log(['apple', 'banana', 'cherry'].myJoin(', ')); // ✅ "apple, banana, cherry"

/* Demo: separator = undefined → default ',' */
console.log([1, 2, 3].myJoin(undefined)); // ✅ "1,2,3"

/* Demo: separator = [] (empty array) → converts to "" */
console.log([1, 2, 3].myJoin([])); // ✅ "123" (because [].toString() is "")

/**
 * join() vs toString() — quick comparison:
 *
 * join():
 * - Custom separator use kar sakte ho.
 * - Default separator ',' hota hai.
 * - null/undefined/holes ko empty string treat karta hai.
 *
 * toString():
 * - Hamesha ',' use karta hai (default separator).
 * - Koi argument nahi leta.
 * - null/undefined/holes ko empty string treat karta hai.
 * - join() ka hi ek special case hai: arr.toString() === arr.join(',')
 */


In [ ]:
/**
 * Final Code:
 * 1. Default separator — undefined hone pe ',' use hota hai
 * 2. null/undefined — empty string treat hote hain
 *  3. Sparse arrays — holes empty string treat hote hain
 * 4. String conversion — elements string mein convert hote hain
 *
 * join() = "Array ko string banao, aur beech mein separator daalo"
 */

Array.prototype.myJoin = function (separator) {
  if (separator === undefined) {
    separator = ',';
  }

  let result = '';

  for (let i = 0; i < this.length; i++) {
    let element;

    if (i in this) {
      element = this[i];
      if (element === null || element === undefined) {
        element = '';
      } else {
        element = String(element);
      }
    } else {
      element = '';
    }

    if (i === 0) {
      result += element;
    } else {
      result += separator + element;
    }
  }

  reverseResult; // bug
  return result;
};


In [ ]:
/**
 * Function Polyfills:
 * 1. Function.prototype.call
 * 2. Function.prototype.apply
 * 3. Function.prototype.bind
*/

In [ ]:
/**
 * Call Polyfill (Basic Version)
 */

/**
 * Interviewer: Function.prototype.call ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain call() kaise kaam karta hai:
 * - call() ek function ko call karta hai, lekin usme 'this' ka value
 *   hum khud set kar sakte hain.
 * - Iske alawa, hum arguments bhi pass kar sakte hain (ek-ek karke).
 *
 * Simple words mein: "Function ko call karo, par 'this' apne marzi ka lo"
 *
 * Example:
 *   function greet() { console.log(this.name); }
 *   const obj = { name: 'Aslam' };
 *   greet.call(obj); // 'this' becomes obj, prints 'Aslam'
 *
 * Important points:
 * 1. call() function ka result return karta hai.
 * 2. Pehla argument 'thisArg' hota hai — yeh function ke andar 'this' ban jata hai.
 * 3. Baaki arguments function ke normal arguments ban jaate hain.
 *
 * Syntax: func.call(thisArg, arg1, arg2, ...)
 */

/* Step 1: Basic version — sirf thisArg set karna, arguments baad mein */
Function.prototype.myCall = function (thisArg) {
    /* thisArg pe ek temporary property banao aur function uspe daal do */
    thisArg.tempFn = this;

    /* Function ko call karo, jisse 'this' thisArg ban jayega */
    const result = thisArg.tempFn();

    /* Temporary property ko delete kar do taaki object clean rahe */
    delete thisArg.tempFn;

    return result;
};

/* Test 1: Simple this binding */
function sayName() {
    console.log(this.name);
}

const obj = { name: 'Aslam' };
sayName.myCall(obj); // ✅ 'Aslam'

/**
 * Step-by-step flow:
 *
 * 1. sayName.myCall(obj) call hota hai.
 * 2. thisArg = obj ho gaya.
 * 3. obj.tempFn = sayName ho gaya.
 * 4. obj.tempFn() call kiya, toh sayName chalega aur 'this' = obj rahega.
 * 5. delete obj.tempFn karke object wapas clean kar diya.
 *
 * Funda simple hai: JavaScript mein jab object ki property ke through
 * function call hota hai, toh 'this' wahi object ban jata hai.
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 3 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Arguments pass nahi ho rahe
 * - call() mein hum function ko arguments bhi pass karte hain:
 *     function greet(greeting) { console.log(greeting + ' ' + this.name); }
 *     greet.call(obj, 'Hello');
 *
 * - Hamara polyfill sirf thisArg leta hai, baaki arguments ignore karta hai.
 * - Agar function arguments expect karta hai, toh woh undefined mil jayenge.
 */

function greet(greeting) {
    console.log(greeting + ' ' + this.name);
}

const obj1 = { name: 'Aslam' };

try {
    greet.myCall(obj1, 'Hello');
} catch (e) {
    console.log(e.message); // ❌ "undefined Aslam" (greeting is undefined)
}

/**
 * FAILURE 2 — thisArg null ya undefined hone par
 * - Agar koi thisArg na pass kare ya null/undefined de:
 *     greet.call(null)
 * - Native call() mein 'this' global object (window) ya undefined ban jata hai.
 * - Hamara polyfill null pe property add karne ki koshish karega:
 *     null.tempFn = this
 *   Yeh crash ho jayega — TypeError.
 */

try {
    greet.myCall(null);
} catch (e) {
    console.log(e.message); // ❌ TypeError: Cannot set properties of null
}

/**
 * FAILURE 3 — Primitives (number, string) as thisArg
 * - Agar primitive value pass karein:
 *     greet.myCall(42)
 * - Native call() primitive ko object mein wrap kar deta hai (Object(42)).
 * - Hamara polyfill 42 pe property add karega:
 *     42.tempFn = this
 *   Strict mode mein yeh crash ho jayega.
 */


In [ ]:
/**
 * Edge Cases Fix Karna
 */

/**
 * Interviewer: Arguments kaise pass karoge?
 *
 * You: JavaScript mein 'arguments' ek array-like object hota hai
 *   jisme function ko mile saare arguments aate hain.
 * - arguments[0] thisArg hota hai.
 * - arguments[1] se leke baaki sab function ke arguments hain.
 * - Main in baaki arguments ko loop karke nikalunga aur pass karunga.
 *
 * Interviewer: null/undefined/primitives ka issue?
 *
 * You: thisArg ko agar null/undefined hai, toh global object (window/globalThis)
 *   use karenge. Agar primitive hai, toh Object() se wrap karenge.
 */

/* Step 2: Arguments + thisArg fallback fix */
Function.prototype.myCall = function (thisArg, ...args) {
    /* Edge Case 1: Agar function hi nahi hai, toh TypeError */
    if (typeof this !== 'function') {
        throw new TypeError(this + ' is not a function');
    }

    /* Edge Case 3: null/undefined ko global object banao, primitives ko wrap karo */
    if (thisArg === undefined || thisArg === null) {
        thisArg = globalThis; // non-strict mode behavior
    } else {
        thisArg = Object(thisArg); // primitive ko object mein wrap karo
    }

    /* Ek unique property name banao taaki existing property overwrite na ho */
    const tempKey = Symbol('tempFn');
    thisArg[tempKey] = this;

    /* Arguments array (args) pass karte hue function call karo */
    const result = thisArg[tempKey](...args);

    /* Temporary property delete karo */
    delete thisArg[tempKey];

    return result;
};

/* Test 1: Arguments ab sahi pass ho rahe hain */
function greet2(greeting, punctuation) {
    console.log(greeting + ' ' + this.name + punctuation);
}

const obj2 = { name: 'Aslam' };
greet2.myCall(obj2, 'Hello', '!'); // ✅ "Hello Aslam!"

/* Test 2: null as thisArg — ab crash nahi hoga (non-strict mode) */
function showGlobal() {
    console.log(this === globalThis);
}
showGlobal.myCall(null); // ✅ true (this becomes global object)

/* Test 3: Primitive as thisArg — ab wrap ho jayega */
function checkType() {
    console.log(typeof this);
}
checkType.myCall(42); // ✅ 'object' (wrapped in Object)


In [ ]:
/**
 * Symbol aur Object() Concept — Detail Mein (Hinglish)
 */

/**
 * Symbol kyun use kiya?
 * - Humne thisArg pe ek property daali: thisArg[tempKey] = this
 * - Agar hum simple string use karte (jaise 'tempFn'), toh ek problem hoti:
 *   Agar object mein pehle se hi 'tempFn' naam ki property ho, toh woh overwrite ho jayegi.
 *   const obj = { tempFn: 'important data', name: 'Aslam' };
 *   greet.call(obj) // obj.tempFn = greet ho jayega, data lost!
 *
 * - Symbol() ek UNIQUE identifier banata hai.
 *   const s1 = Symbol('temp');
 *   const s2 = Symbol('temp');
 *   s1 === s2 → false (hamesha unique hote hain)
 *
 * - Isliye Symbol use karne se humein 100% guarantee hoti hai ki
 *   hum kisi existing property ko overwrite nahi kar rahe.
 *
 * Object() kyun use kiya?
 * - Agar thisArg primitive ho (number, string, boolean):
 *     42.tempFn = this; // Error in strict mode!
 * - Primitives pe properties add nahi ho sakti.
 * - Isliye hum Object(42) karte hain, jo ek Number wrapper object banata hai.
 *   const wrappedNum = Object(42);
 *   typeof wrappedNum → 'object'
 *   wrappedNum.tempFn = this; // Works! No error.
 * - Native call() bhi yahi karta hai — primitives ko object mein wrap karta hai.
 */

/* Demo: Symbol uniqueness */
const s1 = Symbol('temp');
const s2 = Symbol('temp');
console.log(s1 === s2); // false

/* Demo: Object wrapping */
const primitiveNum = 42;
console.log(typeof primitiveNum); // 'number'

const wrappedNum = Object(42);
console.log(typeof wrappedNum); // 'object'
console.log(wrappedNum.valueOf()); // 42

/**
 * call() vs apply() vs bind() — quick comparison:
 *
 * call():
 * - thisArg + arguments ek-ek karke (comma separated).
 * - Function turant call hota hai.
 *   func.call(thisArg, arg1, arg2)
 *
 * apply():
 * - thisArg + arguments ek ARRAY mein.
 * - Function turant call hota hai.
 *   func.apply(thisArg, [arg1, arg2])
 *
 * bind():
 * - thisArg + arguments set karta hai, par function turant call nahi hota.
 * - Ek NAYA function return karta hai jo baad mein call kiya ja sakta hai.
 *   const newFn = func.bind(thisArg, arg1);
 *   newFn();
 */


In [ ]:
/**
 * Final Code:
 * 1. Type check — agar 'this' function nahi hai, toh TypeError
 * 2. thisArg — null/undefined ko global object banao, primitives ko wrap karo
 * 3. Unique key — Symbol use karke property add karo taaki overwrite na ho
 * 4. Arguments spread karo — ...args se function ko pass karo
 * 5. Cleanup — temporary property delete kar do
 *
 * call() = "function call karo, this apne marzi ka lo, arguments comma se do"
 */

Function.prototype.myCall = function (thisArg, ...args) {
  if (typeof this !== 'function') {
    throw new TypeError(this + ' is not a function');
  }

  if (thisArg === undefined || thisArg === null) {
    thisArg = globalThis;
  } else {
    thisArg = Object(thisArg);
  }

  const tempKey = Symbol('tempFn');
  thisArg[tempKey] = this;

  const result = thisArg[tempKey](...args);

  delete thisArg[tempKey];

  return result;
};


In [ ]:
/**
 * Apply Polyfill (Basic Version)
 */

/**
 * Interviewer: Function.prototype.apply ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain apply() kaise kaam karta hai:
 * - apply() call() jaisa hi hai, bas arguments pass karne ka tarika alag hai.
 * - call() mein arguments ek-ek karke (comma separated) dete hain.
 * - apply() mein arguments ek ARRAY mein dete hain.
 *
 * Simple words mein: "Function ko call karo, 'this' apne marzi ka lo, aur arguments array mein do"
 *
 * Example:
 *   function greet(greeting) { console.log(greeting + ' ' + this.name); }
 *   const obj = { name: 'Aslam' };
 *   greet.apply(obj, ['Hello']); // 'this' = obj, arguments = ['Hello']
 *
 * Important points:
 * 1. apply() function ka result return karta hai.
 * 2. Pehla argument 'thisArg' hota hai — yeh function ke andar 'this' ban jata hai.
 * 3. Doosra argument arguments array hota hai.
 *
 * Syntax: func.apply(thisArg, [arg1, arg2, ...])
 */

/* Step 1: Basic version — sirf thisArg set karna, arguments array handling baad mein */
Function.prototype.myApply = function (thisArg, argsArray) {
    /* thisArg pe ek temporary property banao aur function uspe daal do */
    thisArg.tempFn = this;

    /* Function ko call karo, jisse 'this' thisArg ban jayega */
    const result = thisArg.tempFn(...argsArray);

    /* Temporary property ko delete kar do taaki object clean rahe */
    delete thisArg.tempFn;

    return result;
};

/* Test 1: Simple this binding with arguments array */
function introduce(greeting, punctuation) {
    console.log(greeting + ', I am ' + this.name + punctuation);
}

const person = { name: 'Aslam' };
introduce.myApply(person, ['Hello', '!']); // ✅ "Hello, I am Aslam!"

/**
 * Step-by-step flow:
 *
 * 1. introduce.myApply(person, ['Hello', '!']) call hota hai.
 * 2. thisArg = person ho gaya.
 * 3. argsArray = ['Hello', '!'] ho gaya.
 * 4. person.tempFn = introduce ho gaya.
 * 5. person.tempFn('Hello', '!') call kiya, toh introduce chalega.
 *    'this' = person rahega aur arguments mil jayenge.
 * 6. delete person.tempFn karke object wapas clean kar diya.
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 4 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — null/undefined thisArg par crash
 * - Agar koi thisArg na pass kare ya null/undefined de:
 *     greet.myApply(null, ['Hello'])
 * - Hamara polyfill null pe property add karne ki koshish karega:
 *     null.tempFn = this
 *   Yeh crash ho jayega — TypeError: Cannot set properties of null.
 */

try {
    introduce.myApply(null, ['Hello', '!']);
} catch (e) {
    console.log(e.message); // ❌ TypeError: Cannot set properties of null
}

/**
 * FAILURE 2 — Primitives (number, string) as thisArg
 * - Agar primitive value pass karein:
 *     introduce.myApply(42, ['Hello'])
 * - Hamara polyfill 42 pe property add karega:
 *     42.tempFn = this
 *   Strict mode mein yeh crash ho jayega.
 */

/**
 * FAILURE 3 — argsArray pass na karna
 * - Agar user doosra argument (argsArray) pass na kare:
 *     introduce.myApply(person)
 * - argsArray undefined ho jayega.
 * - undefined spread karne par JS error throw karta hai:
 *   TypeError: undefined is not iterable
 */

try {
    introduce.myApply(person); // argsArray is undefined
} catch (e) {
    console.log(e.message); // ❌ TypeError: undefined is not iterable
}

/**
 * FAILURE 4 — Existing property overwrite ho jana
 * - Agar object mein pehle se 'tempFn' property ho:
 *     const obj = { tempFn: 'important', name: 'Aslam' }
 *     introduce.myApply(obj, ['Hi'])
 * - Hamara polyfill obj.tempFn = introduce kar dega.
 *   Isse original 'tempFn' property (yaad rakho, apply() bhi same issue rakhta tha).
 */


In [ ]:
/**
 * Edge Cases Fix Karna
 */

/**
 * Interviewer: null/undefined/primitives ka issue kaise fix karoge?
 *
 * You: thisArg ko agar null/undefined hai, toh global object (window/globalThis)
 *   use karenge. Agar primitive hai, toh Object() se wrap karenge.
 *
 * Interviewer: argsArray undefined hone par kya hoga?
 *
 * You: Agar argsArray undefined ya null ho, toh hum use empty array [] maan lenge.
 *   Isse function bina arguments ke call ho jayega.
 *
 * Interviewer: Property overwrite ka issue?
 *
 * You: Ek unique property name banao taaki existing property overwrite na ho.
 *   Symbol() use karenge.
 */

/* Step 2: All edge cases fix karna */
Function.prototype.myApply = function (thisArg, argsArray) {
    /* Edge Case 1: Agar function hi nahi hai, toh TypeError */
    if (typeof this !== 'function') {
        throw new TypeError(this + ' is not a function');
    }

    /* Edge Case 2: null/undefined ko global object banao, primitives ko wrap karo */
    if (thisArg === undefined || thisArg === null) {
        thisArg = globalThis; // non-strict mode behavior
    } else {
        thisArg = Object(thisArg); // primitive ko object mein wrap karo
    }

    /* Edge Case 3: argsArray undefined/null hone par empty array use karo */
    if (argsArray === undefined || argsArray === null) {
        argsArray = [];
    }

    /* Ek unique property name banao taaki existing property overwrite na ho */
    const tempKey = Symbol('tempFn');
    thisArg[tempKey] = this;

    /* Arguments array spread karo aur function call karo */
    const result = thisArg[tempKey](...argsArray);

    /* Temporary property delete karo */
    delete thisArg[tempKey];

    return result;
};

/* Test 1: null as thisArg — ab crash nahi hoga */
function showGlobal() {
    console.log(this === globalThis);
}
showGlobal.myApply(null); // ✅ true (this becomes global object)

/* Test 2: Primitive as thisArg — ab wrap ho jayega */
function checkType() {
    console.log(typeof this);
}
checkType.myApply(42); // ✅ 'object' (wrapped in Object)

/* Test 3: No argsArray — ab empty array use hoga */
function sayHi() {
    console.log('Hi ' + this.name);
}
const user = { name: 'Aslam' };
sayHi.myApply(user); // ✅ "Hi Aslam" (no args needed, but no crash)


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 3 edge cases fix ho gaye:
 * a. thisArg — null/undefined/primitives handle ho gaye.
 * b. argsArray undefined/null — empty array use ho raha hai.
 * c. Existing property — Symbol se overwrite issue fix ho gaya.
 * Lekin ek aur important edge case bacha hai jo array-like objects ka hai.
 */

/**
 * FAILURE — Array-like objects (Arguments object) as argsArray
 * - apply() sirf arrays nahi, array-like objects bhi accept karta hai.
 * - Array-like object wo hota hai jisme 'length' property ho aur index keys ho.
 *   Example: arguments object (function ke andar milta hai)
 *
 *   function test() {
 *       console.log(arguments); // { '0': 'a', '1': 'b', length: 2 }
 *   }
 *
 * - Native apply() array-like object ko spread kar sakta hai.
 * - Hamara polyfill `...argsArray` use karta hai.
 * - Spread operator array-like objects par kaam karta hai agar unme length ho.
 * - Isliye yeh coincidentally to kaam karta hai, lekin humein explicitly check
 *   karna chahiye ki argsArray valid hai ya nahi.
 *
 * - Agar koi number ya string de:
 *     introduce.myApply(person, 123)
 *   Number mein 'length' nahi hoti. Spread karega toh crash ho jayega.
 *
 * - Native apply() is case mein TypeError throw karta hai:
 *   "CreateListFromArrayLike called on non-object"
 */

try {
    introduce.myApply(person, 123);
} catch (e) {
    console.log(e.message); // ❌ TypeError: 123 is not iterable
}


In [ ]:
/**
 * Edge Case 4 Fix Karna (Array-like objects & Type check)
 */

/**
 * Interviewer: Agar argsArray mein array na de kar koi object ya number de to?
 *
 * You: Native apply() ek specific error throw karta hai:
 *   "CreateListFromArrayLike called on non-object"
 * - Humein check karna hoga ki argsArray null/undefined na ho (jo hum kar chuke),
 *   aur yeh ek object ya array ho jisme 'length' property ho.
 * - Agar length property nahi hai, toh error throw karna chahiye.
 *
 * Yeh check `typeof argsArray !== 'object'` se nahi hoga, kyunki array bhi object hota hai.
 * Isliye hum check karenge ki argsArray mein length property hai ya nahi.
 */

/* Step 3: Array-like type check add karo */
Function.prototype.myApply = function (thisArg, argsArray) {
    /* Edge Case 1: Agar function hi nahi hai, toh TypeError */
    if (typeof this !== 'function') {
        throw new TypeError(this + ' is not a function');
    }

    /* Edge Case 2: null/undefined ko global object banao, primitives ko wrap karo */
    if (thisArg === undefined || thisArg === null) {
        thisArg = globalThis;
    } else {
        thisArg = Object(thisArg);
    }

    /* Edge Case 3: argsArray undefined/null hone par empty array use karo */
    if (argsArray === undefined || argsArray === null) {
        argsArray = [];
    }

    /* Edge Case 4: Array-like object check karo (length property honi chahiye) */
    if (typeof argsArray !== 'object' || !('length' in argsArray)) {
        throw new TypeError('CreateListFromArrayLike called on non-object');
    }

    /* Ek unique property name banao taaki existing property overwrite na ho */
    const tempKey = Symbol('tempFn');
    thisArg[tempKey] = this; // assign 'this' (the function) to the temporary key

    /* Arguments array spread karo aur function call karo */
    const result = thisArg[tempKey](...argsArray);

    /* Temporary property delete karo */
    delete thisArg[tempKey];

    return result;
};


/* Test 1: Array-like object (arguments) */
function testArgs() {
    introduce.myApply(person, arguments); // arguments is array-like
}
testArgs('Hello', '!'); // ✅ "Hello, I am Aslam!"

/* Test 2: Number as argsArray — TypeError aana chahiye */
try {
    introduce.myApply(person, 123);
} catch (e) {
    console.log(e.message); // ✅ "CreateListFromArrayLike called on non-object"
}

/* Test 3: Normal array abhi bhi theek chalega */
introduce.myApply(person, ['Hi', '.']); // ✅ "Hi, I am Aslam."


In [ ]:
/**
 * apply() vs call() vs bind() — Detail Mein (Hinglish)
 */

/**
 * apply() kya hai?
 * - apply() function ko turant call karta hai.
 * - 'this' value apne marzi ka set karta hai.
 * - arguments ek ARRAY (ya array-like object) mein pass karta hai.
 *
 * call() kya hai?
 * - call() function ko turant call karta hai.
 * - 'this' value apne marzi ka set karta hai.
 * - arguments ek-ek karke (comma separated) pass karta hai.
 *
 * bind() kya hai?
 * - bind() function ko turant call NAHI karta.
 * - 'this' value aur arguments set karke ek NAYA function return karta hai.
 * - Naya function baad mein call kiya ja sakta hai.
 *
 * Kab kya use karna hai?
 * - Agar arguments paashe mein array mein hain, toh apply() use karo.
 *   Example: Math.max.apply(null, [1, 2, 3]) → 3
 *   Math.max([1, 2, 3]) → NaN (Math.max expects comma separated args)
 *
 * - Agar arguments individually pata hain, toh call() use karo.
 *   Example: greet.call(obj, 'Hello', '!')
 *
 * - Agar baad mein function call karna hai aur this/args pehle se set karne hain:
 *   Example: const newFn = greet.bind(obj, 'Hello'); newFn('!');
 */

/* Demo: apply() for Math.max */
const numbers = [5, 2, 9, 1, 7];
const maxNum = Math.max.myApply(null, numbers);
console.log(maxNum); // ✅ 9

/* Math.max(numbers) deta NaN, kyunki Math.max ek array nahi leta.
 * Math.max(5, 2, 9, 1, 7) deta 9.
 * apply() array ko individual arguments mein convert kar deta hai.
 */

/* Demo: call() for direct arguments */
const obj = { name: 'Aslam' };
function greet(greeting, punctuation) {
    console.log(greeting + ' ' + this.name + punctuation);
}
greet.myCall(obj, 'Hello', '!'); // ✅ "Hello Aslam!"

/* Demo: bind() for later execution */
const boundGreet = greet.bind(obj, 'Hi');
boundGreet('!'); // ✅ "Hi Aslam!"


In [ ]:
/**
 * Final Code:
 * 1. Type check — agar 'this' function nahi hai, toh TypeError
 * 2. thisArg — null/undefined ko global object banao, primitives ko wrap karo
 *  3. argsArray — undefined/null ko empty array banao
 * 4. Array-like check — argsArray mein 'length' property honi chahiye
 * 5. Unique key — Symbol use karke property add karo taaki overwrite na ho
 * 6. Cleanup — temporary property delete kar do
 *
 * apply() = "function call karo, this apne marzi ka lo, arguments array mein do"
 */

Function.prototype.myApply = function (thisArg, argsArray) {
  if (typeof this !== 'function') {
    throw new TypeError(this + ' is not a function');
  }

  if (thisArg === undefined || thisArg === null) {
    thisArg = globalThis;
  } else {
    thisArg = Object(thisArg);
  }

  if (argsArray === undefined || argsArray === null) {
    argsArray = [];
  }

  if (typeof argsArray !== 'object' || !('length' in argsArray)) {
    throw new TypeError('CreateListFromArrayLike called on non-object');
  }

  const tempKey = Symbol('tempFn');
  thisArg[tempKey] = this;

  const result = thisArg[tempKey](...argsArray);

  delete thisArg[tempKey];

  return result;
};

In [ ]:
/**
 * Bind Polyfill (Basic Version)
 */

/**
 * Interviewer: Function.prototype.bind ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain bind() kaise kaam karta hai:
 * - bind() ek function ko call NAHI karta. Yeh ek NAYA function return karta hai.
 * - Is naye function ka 'this' pehle se set hota hai.
 * - Iske alawa, hum arguments bhi pehle se set kar sakte hain (Partial Application).
 *
 * Simple words mein: "Function ko call mat karo, par uska 'this' aur arguments fix kar do, aur ek naya function de do"
 *
 * Example:
 *   function greet(greeting) { console.log(greeting + ' ' + this.name); }
 *   const obj = { name: 'Aslam' };
 *   const boundGreet = greet.bind(obj, 'Hello');
 *   boundGreet(); // 'this' = obj, prints 'Hello Aslam'
 *
 * Important points:
 * 1. bind() ek NAYA function return karta hai. Turant call nahi karta.
 * 2. Pehla argument 'thisArg' hota hai — yeh naye function ke andar 'this' ban jata hai.
 * 3. Baaki arguments function ke normal arguments ban jaate hain.
 *
 * Syntax: const newFunc = func.bind(thisArg, arg1, arg2, ...)
 */

/* Step 1: Basic version — sirf thisArg set karna, arguments baad mein */
Function.prototype.myBind = function (thisArg) {
    /* 'this' wo original function hai jispe myBind call hua */
    const originalFunc = this;

    /* Naya function return karo (Arrow function use mat karna, warna 'this' issue hoga) */
    return function () {
        /* originalFunc ko call karo, aur 'this' ko thisArg set karo */
        return originalFunc.call(thisArg);
    };
};

/* Test 1: Simple this binding */
function sayName() {
    console.log(this.name);
}

const obj = { name: 'Aslam' };
const boundFn = sayName.myBind(obj);

boundFn(); // ✅ 'Aslam'

/**
 * Step-by-step flow:
 *
 * 1. sayName.myBind(obj) call hota hai.
 * 2. originalFunc = sayName ho gaya.
 * 3. Ek naya function return hua jo closure mein originalFunc aur thisArg yaad rakhta hai.
 * 4. boundFn() call kiya.
 * 5. Naye function ne originalFunc.call(obj) chala diya.
 * 6. sayName chala aur 'this' = obj mila, jisse 'Aslam' print hua.
 *
 * Closure ka funda:
 * - Naya function jo return hua, woh originalFunc aur thisArg ko apne scope mein rakhta hai.
 * - Jab bhi boundFn call hota hai, yeh us scope se values le leta hai. Yahi closure hai.
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 4 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Arguments pass nahi ho rahe (Bind time aur Call time)
 * - bind() mein hum arguments do jagah de sakte hain:
 *   1. Bind karte waqt: const fn = greet.bind(obj, 'Hello');
 *   2. Call karte waqt: fn('!');
 *   - Function ko dono milne chahiye: ('Hello', '!')
 *
 * - Hamara polyfill sirf thisArg leta hai, baaki sab ignore karta hai.
 */

function greet(greeting, punctuation) {
    console.log(greeting + ' ' + this.name + punctuation);
}

const obj1 = { name: 'Aslam' };

try {
    const boundGreet = greet.myBind(obj1, 'Hello');
    boundGreet('!'); // ❌ "undefined Aslamundefined" (arguments lost)
} catch (e) {
    console.log(e.message);
}

/**
 * FAILURE 2 — 'this' agar new ke saath use ho (Constructor function)
 * - Agar bound function ko `new` keyword ke saath call kare:
 *     function Person(name) { this.name = name; }
 *     const BoundPerson = Person.myBind({ country: 'India' });
 *     const p = new BoundPerson('Aslam');
 *
 * - Native bind() mein, jab aap `new` use karte ho, toh 'this' thisArg IGNORE ho jata hai.
 *   Naya object khud ban jata hai (jaise normal constructor mein hota hai).
 *
 * - Hamara polyfill hamesha thisArg pe hi force karega, isliye `new` ka behaviour break hoga.
 *   p.country = 'India' ho jayega (wrong), aur p.name nahi milega.
 */

function Person(name) {
    this.name = name;
}

const BoundPerson = Person.myBind({ country: 'India' });
const p = new BoundPerson('Aslam');
console.log(p.name); // ❌ undefined (should be 'Aslam')
console.log(p.country); // ❌ 'India' (should be undefined)

/**
 * FAILURE 3 — Primitives (number, string) as thisArg
 * - Agar primitive value pass karein:
 *     greet.myBind(42)
 * - Native bind() primitive ko object mein wrap kar deta hai.
 * - Hamara polyfill waise toh kaam karega, lekin strict mode mein 'this' 42 hi rahega.
 */


In [ ]:
/**
 * Edge Cases 1 & 3 Fix Karna (Arguments + Primitives)
 */

/**
 * Interviewer: Arguments kaise pass karoge?
 *
 * You: Do jagah arguments mil sakte hain:
 *   1. myBind() ko call karte waqt (args ke pehle waale part).
 *   2. Naye function ko call karte waqt (args ke doosre waale part).
 * - main dono ko merge karunga: [...bindArgs, ...callArgs]
 *
 * Interviewer: Primitives ka issue?
 *
 * You: thisArg ko Object() se wrap karenge taaki primitive object ban jaye.
 */

/* Step 2: Arguments merge + Primitives wrap fix */
Function.prototype.myBind = function (thisArg, ...bindArgs) {
    const originalFunc = this;

    /* Edge Case 3: Primitives ko wrap karo */
    if (thisArg !== null && thisArg !== undefined) {
        thisArg = Object(thisArg);
    }

    /* Naya function return karo */
    return function (...callArgs) {
        /* Edge Case 1: Bind time aur Call time arguments merge karo */
        const allArgs = [...bindArgs, ...callArgs];

        /* originalFunc ko call karo, aur 'this' ko thisArg set karo */
        return originalFunc.apply(thisArg, allArgs);
    };
};

/* Test 1: Arguments ab sahi pass ho rahe hain (Partial Application) */
function greet2(greeting, punctuation) {
    console.log(greeting + ' ' + this.name + punctuation);
}

const obj2 = { name: 'Aslam' };
const boundGreet2 = greet2.myBind(obj2, 'Hello');
boundGreet2('!'); // ✅ "Hello Aslam!"

/* Test 2: All arguments at bind time */
const boundGreet3 = greet2.myBind(obj2, 'Hi', '.');
boundGreet3(); // ✅ "Hi Aslam."

/* Test 3: Primitive as thisArg — ab wrap ho jayega */
function checkType() {
    console.log(typeof this);
}
const boundCheck = checkType.myBind(42);
boundCheck(); // ✅ 'object' (wrapped in Object)

/* Test 4: null/undefined as thisArg — global object ban jayega */
function showGlobal() {
    console.log(this === globalThis);
}
const boundGlobal = showGlobal.myBind(null);
boundGlobal(); // ✅ true (this becomes global object in non-strict mode)


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 2 edge cases fix ho gaye:
 * a. Arguments — bind time aur call time arguments merge ho rahe hain.
 * b. Primitives — Object() mein wrap ho rahe hain.
 * Lekin `new` keyword wala issue abhi bhi bacha hai.
 */

/**
 * FAILURE — `new` keyword ke saath bound function call karna
 * - Agar bound function ko `new` se call karein:
 *     const p = new BoundPerson('Aslam');
 * - JavaScript engine naya object banata hai aur use constructor ke 'this' banata hai.
 * - Lekin hamara polyfill hamesha thisArg pe force karta hai.
 * - Isliye p.name set nahi hoga (kyunki 'this' thisArg ban gaya, p nahi).
 */

function Person2(name) {
    this.name = name;
}

const BoundPerson2 = Person2.myBind({ country: 'India' });
const p2 = new BoundPerson2('Aslam');

console.log(p2.name); // ❌ undefined (should be 'Aslam')
console.log(p2.country); // ❌ 'India' (should be undefined on the new instance)
console.log(p2 instanceof Person2); // ❌ false (should be true)


In [ ]:
/**
 * Edge Case 2 Fix Karna (`new` keyword handling)
 */

/**
 * Interviewer: `new` keyword ke saath bind kaise handle karoge?
 *
 * You: JavaScript mein, jab aap `new` se function call karte ho, toh ek naya object
 *   ban jata hai aur woh 'this' ban jata hai.
 * - Lekin hamara returned function hamesha thisArg apply karta hai.
 * - Is issue ko fix karne ke liye, humein pata karna hoga ki function
 *   `new` ke saath call hua hai ya normal call kiya gaya.
 *
 * - Hum check karenge: `new.target` ya `this instanceof` ka use karke.
 * - Agar function `new` ke saath call hua, toh thisArg ignore karo aur naya 'this' use karo.
 * - Warna, normal thisArg use karo.
 *
 * Funda:
 * - Naye function ke andar 'this' check karte hain.
 * - Agar `this instanceof returnedFunction` true hai, matlab `new` use hua.
 * - Toh 'this' ko hi use karo, thisArg ko ignore karo.
 */

/* Step 3: `new` keyword handling add karo */
Function.prototype.myBind = function (thisArg, ...bindArgs) {
    const originalFunc = this;

    if (thisArg !== null && thisArg !== undefined) {
        thisArg = Object(thisArg);
    }

    /* Naya function banata hain jiska naam hum denge taaki instanceof kaam kare */
    const boundFunction = function (...callArgs) {
        /* Edge Case 2: Agar `new` se call hua, toh 'this' boundFunction ka instance hoga.
         * Warna normal call mein 'this' global ya undefined hoga.
         * Agar `new` use hua, toh thisArg ignore karo aur naya 'this' use karo.
         */
        const isNewCall = this instanceof boundFunction;

        /* Agar new use hua, toh 'this' (naya instance) use karo, warna thisArg use karo */
        const context = isNewCall ? this : thisArg;

        /* Arguments merge karo */
        const allArgs = [...bindArgs, ...callArgs];

        /* originalFunc ko call karo.
         * Agar new use hua, toh originalFunc bhi new ki tarah behave karega.
         */
        return originalFunc.apply(context, allArgs);
    };

    /* Link prototype taaki instanceof properly kaam kare aur inherited methods milein */
    boundFunction.prototype = Object.create(originalFunc.prototype);

    return boundFunction;
};

/* Test 1: `new` keyword ab sahi kaam karega */
function Person3(name) {
    this.name = name;
}

const BoundPerson3 = Person3.myBind({ country: 'India' });
const p3 = new BoundPerson3('Aslam');

console.log(p3.name); // ✅ 'Aslam' (this was the new instance, not the thisArg)
console.log(p3.country); // ✅ undefined (thisArg ignored)
console.log(p3 instanceof Person3); // ✅ true

/* Test 2: Normal call abhi bhi theek chalega (thisArg use hoga) */
const normalCall = Person3.myBind({ country: 'India' });
const normalResult = normalCall('Rahul'); // Yahan `new` nahi hai.
// Note: Person3 return nahi karta, so normalResult undefined hoga, lekin thisArg set hoga.

function simpleGreet(greeting) {
    console.log(greeting + ' ' + this.name);
}
const boundGreet3 = simpleGreet.myBind({ name: 'Aslam' }, 'Hello');
boundGreet3(); // ✅ "Hello Aslam" (normal call, thisArg used)


In [ ]:
/**
 * `new` keyword handling & Prototype chain — Detail Mein (Hinglish)
 */

/**
 * `new` keyword kaise kaam karta hai?
 * - Jab aap `new Person('Aslam')` likhte ho, JavaScript 4 kaam karta hai:
 *   1. Ek naya blank object banata hai: {}
 *   2. Naye object ka proto link Person.prototype se jod deta hai.
 *   3. Person function chalata hai, aur uske andar 'this' naya object ban jata hai.
 *   4. Agar Person koi object return nahi karta, toh naya object return ho jata hai.
 *
 * Bind() ke context mein issue:
 * - Hamara bound function hamesha thisArg apply karta hai (Step 3 tak).
 * - Lekin `new` use hone par, JavaScript naya object banata hai aur use 'this' banata hai.
 * - Agar hum thisArg force karte hain, toh naya object waste ho jata hai.
 *
 * Fix:
 * - Naye function ke andar, hum check karte hain ki `this` boundFunction ka instance hai ya nahi.
 * - Agar `new` se call hua, toh `this` boundFunction ka instance hoga (naya object).
 *   Is case mein, hum naye `this` ko use karte hain aur thisArg ignore karte hain.
 *
 * Prototype linking:
 * - `boundFunction.prototype = Object.create(originalFunc.prototype)`
 * - Yeh isliye zaroori hai kyunki `new` ka step 2 (proto link) boundFunction.prototype se karta hai.
 * - Agar hum yeh link nahi karenge, toh `p3 instanceof Person3` false aayega.
 * - Object.create se hum ek aisa object banate hain jiska proto originalFunc.prototype hai.
 *   Isse bound function bhi original function ke prototype chain se jud jata hai.
 */

/* Demo: instanceof check without prototype link */
function withoutProtoLink() {}
const boundNoProto = withoutProtoLink.myBind({});
const instanceNoProto = new boundNoProto();
// console.log(instanceNoProto instanceof withoutProtoLink); // false, because prototype wasn't linked

/* Demo: instanceof check with prototype link */
function withProtoLink() {}
const boundWithProto = withProtoLink.myBind({});
const instanceWithProto = new boundWithProto();
// console.log(instanceWithProto instanceof withProtoLink); // true, because we linked it


In [ ]:
/**
 * Final Code:
 * 1. thisArg — null/undefined ko global object banao, primitives ko wrap karo
 * 2. Arguments — bind time aur call time arguments merge karo ([...bindArgs, ...callArgs])
 * 3. `new` keyword — agar `new` se call hua, toh thisArg ignore karo aur naya 'this' use karo
 * 4. Prototype — bound function ka prototype original function ke prototype se link karo
 *
 * bind() = "function call mat karo, this aur args fix karke naya function de do"
 */

Function.prototype.myBind = function (thisArg, ...bindArgs) {
  const originalFunc = this;

  if (thisArg !== null && thisArg !== undefined) {
    thisArg = Object(thisArg);
  }

  const boundFunction = function (...callArgs) {
    const isNewCall = this instanceof boundFunction;
    const context = isNewCall ? this : thisArg;
    const allArgs = [...bindArgs, ...callArgs];

    return originalFunc.apply(context, allArgs);
  };

  boundFunction.prototype = Object.create(originalFunc.prototype);

  return boundFunction;
};


In [ ]:
/**
 * Promise Polyfills:
 * 1. Promise
 * 2. Promise.all
 * 3. Promise.allSettled
 * 4. Promise.race
 * 5. Promise.any
*/

In [ ]:
/**
 * Promise Polyfill (Basic Version)
 */

/**
 * Interviewer: Promise ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain Promise kaise kaam karta hai:
 * - Promise ek object hai jo future mein complete hone wale operation ko represent karta hai.
 * - Iske 3 states hote hain:
 *   1. PENDING   : Abhi chal raha hai (initial state)
 *   2. FULFILLED : Successfully complete ho gaya (resolved)
 *   3. REJECTED  : Fail ho gaya (rejected)
 *
 * Simple words mein: "Main abhi kar raha hoon, ho gaya toh value dunga, fail hua toh error dunga"
 *
 * Important points:
 * 1. Promise banane pe hum ek executor function pass karte hain.
 *    executor(resolved, rejected)
 * 2. resolve() call karne pe state FULFILLED ho jata hai.
 * 3. reject() call karne pe state REJECTED ho jata hai.
 * 4. Ek baar state change hua (resolve/reject), toh wapas nahi badlega.
 *
 * Syntax: new Promise((resolve, reject) => { ... })
 */

/* Step 1: Basic version — states aur then() handle karna */
function MyPromise(executor) {
    /* State aur value/reason store karne ke liye variables */
    let state = 'PENDING';
    let value = undefined;

    /* Callbacks store karne ke liye arrays (kyunki multiple .then() lag sakte hain) */
    let onResolvedCallbacks = [];
    let onRejectedCallbacks = [];

    /* Resolve function */
    function resolve(val) {
        if (state === 'PENDING') {
            state = 'FULFILLED';
            value = val;
            /* Saare success callbacks ko call karo */
            onResolvedCallbacks.forEach(cb => cb(value));
        }
    }

    /* Reject function */
    function reject(reason) {
        if (state === 'PENDING') {
            state = 'REJECTED';
            value = reason;
            /* Saare failure callbacks ko call karo */
            onRejectedCallbacks.forEach(cb => cb(value));
        }
    }

    /* then() method — yeh success aur failure dono callbacks leta hai */
    this.then = function (onFulfilled, onRejected) {
        if (state === 'FULFILLED') {
            onFulfilled(value);
        } else if (state === 'REJECTED') {
            onRejected(value);
        } else {
            /* Agar pending hai, toh callbacks array mein daal do */
            onResolvedCallbacks.push(onFulfilled);
            onRejectedCallbacks.push(onRejected);
        }
    };

    /* Executor function turant call karo */
    try {
        executor(resolve, reject);
    } catch (err) {
        reject(err);
    }
}

/* Test 1: Simple resolve */
const p1 = new MyPromise((resolve, reject) => {
    resolve('Success!');
});

p1.then((val) => console.log(val)); // ✅ "Success!"

/* Test 2: Async resolve (setTimeout) */
const p2 = new MyPromise((resolve, reject) => {
    setTimeout(() => resolve('Async Success!'), 1000);
});

p2.then((val) => console.log(val)); // ✅ (1 sec baad) "Async Success!"


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 3 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Chaining (.then().then()) support nahi hai
 * - Native Promise mein, .then() ek NAYA promise return karta hai.
 * - Isliye hum chain kar sakte hain: p.then().then().then()
 * - Hamara polyfill .then() mein kuch return nahi karta (undefined).
 * - Agar hum chain karenge, toh TypeError aayega:
 *   "Cannot read properties of undefined (reading 'then')"
 */

const p1 = new MyPromise((resolve) => resolve(10));

try {
    p1.then((val) => val * 2).then((val) => console.log(val));
} catch (e) {
    console.log(e.message); // ❌ Cannot read properties of undefined (reading 'then')
}

/**
 * FAILURE 2 — Synchronous execution
 * - Native Promise ke callbacks hamesha ASYNC run hote hain (microtask queue).
 * - Matlab pehle poora synchronous code chalega, phir .then() chalega.
 *
 * - Hamara polyfill agar state FULFILLED hai, toh TURANT chala deta hai.
 *   console.log('Before');
 *   resolvedPromise.then(() => console.log('Inside then'));
 *   console.log('After');
 *
 * - Native output: Before, After, Inside then (Async)
 * - Hamara output: Before, Inside then, After (Sync) ❌
 */

console.log('Before');
const syncPromise = new MyPromise((resolve) => resolve(1));
syncPromise.then(() => console.log('Inside then'));
console.log('After');
// ❌ Output: Before, Inside then, After
// ✅ Expected: Before, After, Inside then


In [ ]:
/**
 * Edge Cases 1 & 2 Fix Karna (Async + Chaining)
 */

/**
 * Interviewer: Chaining kaise fix karoge?
 *
 * You: .then() ko modify karna hoga. Woh ek NAYA MyPromise return karega.
 * - Agar callback value return kare, toh naya promise us value se resolve hoga.
 * - Agar callback error throw kare, toh naya promise reject hoga.
 *
 * Interviewer: Async execution kaise fix karoge?
 *
 * You: Callbacks ko setTimeout(..., 0) mein wrap karenge.
 * - Isse woh event loop ke next tick pe chalega (macrotask queue).
 * - Native promises microtask queue use karte hain, lekin setTimeout basic polyfill ke liye kaafi hai.
 */

/* Step 2: Async + Chaining fix */
function MyPromise(executor) {
    let state = 'PENDING';
    let value = undefined;
    let onResolvedCallbacks = [];
    let onRejectedCallbacks = [];

    function resolve(val) {
        if (state === 'PENDING') {
            state = 'FULFILLED';
            value = val;
            onResolvedCallbacks.forEach(cb => cb(value));
        }
    }

    function reject(reason) {
        if (state === 'PENDING') {
            state = 'REJECTED';
            value = reason;
            onRejectedCallbacks.forEach(cb => cb(value));
        }
    }

    this.then = function (onFulfilled, onRejected) {
        /* NAYA promise return karo chaining ke liye */
        return new MyPromise((resolve, reject) => {
            /* Ek helper function banate hain jo logic handle karega */
            function handleCallback(cb, val, resolve, reject) {
                /* setTimeout se async banayo */
                setTimeout(() => {
                    try {
                        /* Agar onFulfilled function nahi hai, toh value pass through karo */
                        if (typeof cb === 'function') {
                            const result = cb(val);
                            /* Agar result promise hai, toh uska result lo */
                            if (result instanceof MyPromise) {
                                result.then(resolve, reject);
                            } else {
                                resolve(result);
                            }
                        } else {
                            resolve(val);
                        }
                    } catch (err) {
                        reject(err);
                    }
                }, 0);
            }

            if (state === 'FULFILLED') {
                handleCallback(onFulfilled, value, resolve, reject);
            } else if (state === 'REJECTED') {
                handleCallback(onRejected, value, resolve, reject);
            } else {
                onResolvedCallbacks.push((val) => handleCallback(onFulfilled, val, resolve, reject));
                onRejectedCallbacks.push((val) => handleCallback(onRejected, val, resolve, reject));
            }
        });
    };

    try {
        executor(resolve, reject);
    } catch (err) {
        reject(err);
    }
}

/* Test 1: Chaining ab kaam karegi */
const p1 = new MyPromise((resolve) => resolve(10));
p1.then((val) => val * 2)
  .then((val) => console.log(val)); // ✅ 20

/* Test 2: Async execution ab sahi hai */
console.log('Before');
const syncPromise = new MyPromise((resolve) => resolve(1));
syncPromise.then(() => console.log('Inside then'));
console.log('After');
// ✅ Output: Before, After, Inside then


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak chaining aur async fix ho gaye.
 * Lekin ek important edge case bacha hai jo interview mein bahut pucha jata hai.
 */

/**
 * FAILURE — Resolving a Promise with another Promise
 * - Agar hum resolve() ke andar ek AUR promise pass karein:
 *     new MyPromise((resolve) => {
 *         resolve(new MyPromise((innerResolve) => innerResolve(42)));
 *     });
 *
 * - Native Promise: Yeh inner promise ko wait karega.
 *   Jab inner resolve hoga (42), tabhi outer promise resolve hoga (42).
 *
 * - Hamara polyfill: resolve() outer state ko FULFILLED kar dega,
 *   aur value = innerPromise set kar dega.
 * - Phir .then() chalega, aur usko innerPromise milega (value 42 nahi).
 * - Yeh galat behaviour hai.
 *
 * Fix: resolve() function ke andar check karo ki value promise toh nahi hai.
 * Agar promise hai, toh uske then() ko wait karo.
 */

const innerPromise = new MyPromise((resolve) => resolve(42));

const outerPromise = new MyPromise((resolve) => {
    resolve(innerPromise);
});

outerPromise.then((val) => {
    console.log('Outer resolved with:', val);
    // ❌ "Outer resolved with: MyPromise {...}"
    // ✅ Expected: "Outer resolved with: 42"
});


In [ ]:
/**
 * Edge Case 3 Fix Karna (Resolve with Promise)
 */

/**
 * Interviewer: Agar resolve() mein promise pass ho toh?
 *
 * You: resolve() function ko smart banana padega.
 * - Agar val ek MyPromise hai, toh hum uske then() ko call karenge.
 * - Jab inner promise resolve hoga, tab hum apne outer resolve ko call karenge.
 * - Agar val promise nahi hai (normal value), toh seedla resolve kar denge.
 */

/* Step 3: Resolve with promise fix */
function MyPromise(executor) {
    let state = 'PENDING';
    let value = undefined;
    let onResolvedCallbacks = [];
    let onRejectedCallbacks = [];

    function resolve(val) {
        if (state === 'PENDING') {
            /* Edge Case 3: Agar val promise hai, toh wait karo */
            if (val instanceof MyPromise) {
                val.then(
                    (innerVal) => resolve(innerVal), // Inner resolve hua toh outer resolve karo
                    (innerErr) => reject(innerErr)   // Inner reject hua toh outer reject karo
                );
                return;
            }
            state = 'FULFILLED';
            value = val;
            onResolvedCallbacks.forEach(cb => cb(value));
        }
    }

    function reject(reason) {
        if (state === 'PENDING') {
            state = 'REJECTED';
            value = reason;
            onRejectedCallbacks.forEach(cb => cb(value));
        }
    }

    this.then = function (onFulfilled, onRejected) {
        return new MyPromise((resolve, reject) => {
            function handleCallback(cb, val, resolve, reject) {
                setTimeout(() => {
                    try {
                        if (typeof cb === 'function') {
                            const result = cb(val);
                            if (result instanceof MyPromise) {
                                result.then(resolve, reject);
                            } else {
                                resolve(result);
                            }
                        } else {
                            resolve(val);
                        }
                    } catch (err) {
                        reject(err);
                    }
                }, 0);
            }

            if (state === 'FULFILLED') {
                handleCallback(onFulfilled, value, resolve, reject);
            } else if (state === 'REJECTED') {
                handleCallback(onRejected, value, resolve, reject);
            } else {
                onResolvedCallbacks.push((val) => handleCallback(onFulfilled, val, resolve, reject));
                onRejectedCallbacks.push((val) => handleCallback(onRejected, val, resolve, reject));
            }
        });
    };

    /* catch() method bhi add kar dete hain, yeh just then() ka wrapper hai */
    this.catch = function (onRejected) {
        return this.then(undefined, onRejected);
    };

    try {
        executor(resolve, reject);
    } catch (err) {
        reject(err);
    }
}

/* Test 1: Resolving with a promise */
const inner = new MyPromise((resolve) => setTimeout(() => resolve(42), 100));
const outer = new MyPromise((resolve) => resolve(inner));

outer.then((val) => console.log('Outer resolved with:', val)); // ✅ "Outer resolved with: 42"

/* Test 2: Rejecting with a promise */
const innerReject = new MyPromise((_, reject) => reject("Error!"));
const outerReject = new MyPromise((resolve) => resolve(innerReject));

outerReject.catch((err) => console.log('Outer rejected with:', err)); // ✅ "Outer rejected with: Error!"


In [ ]:
/**
 * Microtask vs Macrotask Concept — Detail Mein (Hinglish)
 */

/**
 * Hamara polyfill setTimeout use karta hai.
 * Lekin native Promise setTimeout use NAHI karta.
 * Native Promise Microtask queue use karta hai.
 *
 * Event Loop mein 2 queues hoti hain:
 * 1. Macrotask Queue (Task Queue)
 *    - setTimeout, setInterval, I/O operations
 *    - Ek macrotask run hone ke baad, microtask queue poori khali hoti hai.
 *
 * 2. Microtask Queue
 *    - Promises (.then/.catch), MutationObserver
 *    - Microtasks hamesha macrotasks se pehle chalte hain.
 *    - Agar 100 microtasks hain, toh saare pehle run honge, koi macrotask nahi.
 *
 * Example difference:
 *   console.log('1');
 *   setTimeout(() => console.log('2'), 0);
 *   Promise.resolve().then(() => console.log('3'));
 *   console.log('4');
 *
 * Native JS Output:
 *   1, 4, 3, 2
 *   (3 pehle aayega kyunki microtask queue setTimeout se pehle run hoti hai)
 *
 * Hamara Polyfill Output:
 *   1, 4, 2, 3 (Agar hum setTimeout use karte hain)
 *   (Ya 1, 4, 3, 2 agar queueMicrotask() use karein)
 */

console.log('1');
setTimeout(() => console.log('2'), 0);
MyPromise.resolve = function(val) { return new MyPromise(res => res(val)); }
MyPromise.resolve('3').then(() => console.log('3'));
console.log('4');

/* Hamara output: 1, 4, 2, 3 (kyunki setTimeout macrotask hai) */
/* Native output: 1, 4, 3, 2 (kyunki microtask pehle chalta hai) */

/**
 * Note for Interview:
 * - Agar interviewer precise behavior maange, toh setTimeout ki jagah
 *   queueMicrotask() use karna chahiye.
 *   setTimeout(() => {...}, 0) => queueMicrotask(() => {...})
 * - Tab yeh native promise ki tarah behave karega.
 */


In [ ]:
/**
 * Final Code:
 * 1. States — PENDING, FULFILLED, REJECTED
 * 2. Async — setTimeout se async banaya
 * 3. Chaining — .then() naya promise return karta hai
 * 4. Resolve Promise — agar resolve() mein promise mile, toh wait karo
 * 5. Error Handling — catch() add kiya, try/catch se executor errors handle kiye
 *
 * Promise = "Future value ka contract"
 */

function MyPromise(executor) {
  let state = 'PENDING';
  let value = undefined;
  let onResolvedCallbacks = [];
  let onRejectedCallbacks = [];

  function resolve(val) {
    if (state === 'PENDING') {
      if (val instanceof MyPromise) {
        val.then(resolve, reject);
        return;
      }
      state = 'FULFILLED';
      value = val;
      onResolvedCallbacks.forEach(cb => cb(value));
    }
  }

  function reject(reason) {
    if (state === 'PENDING') {
      state = 'REJECTED';
      value = reason;
      onRejectedCallbacks.forEach(cb => cb(value));
    }
  }

  this.then = function (onFulfilled, onRejected) {
    return new MyPromise((resolve, reject) => {
      function handleCallback(cb, val, resolve, reject) {
        setTimeout(() => {
          try {
            if (typeof cb === 'function') {
              const result = cb(val);
              if (result instanceof MyPromise) {
                result.then(resolve, reject);
              } else {
                resolve(result);
              }
            } else {
              resolve(val);
            }
          } catch (err) {
            reject(err);
          }
        }, 0);
      }

      if (state === 'FULFILLED') {
        handleCallback(onFulfilled, value, resolve, reject);
      } else if (state === 'REJECTED') {
        handleCallback(onRejected, value, resolve, reject);
      } else {
        onResolvedCallbacks.push((val) => handleCallback(onFulfilled, val, resolve, reject));
        onRejectedCallbacks.push((val) => handleCallback(onRejected, val, resolve, reject));
      }
    });
  };

  this.catch = function (onRejected) {
    return this.then(undefined, onRejected);
  };

  try {
    executor(resolve, reject);
  } catch (err) {
    reject(err);
  }
}


In [ ]:
/**
 * Promise.all Polyfill (Basic Version)
 */

/**
 * Interviewer: Promise.all ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain Promise.all kaise kaam karta hai:
 * - Promise.all ek array of promises leta hai.
 * - Yeh tab resolve hota hai jab SAARE promises resolve ho jayein.
 * - Agar EK bhi promise reject hua, toh turant reject ho jata hai
 *   (baaki promises ka wait nahi karta).
 *
 * Simple words mein: "Sab pass honge tabhi pass, koi ek bhi fail hua toh fail"
 *
 * Important points:
 * 1. Promise.all ek NAYA promise return karta hai.
 * 2. Result hamesha ARRAY hota hai — input promises ke order mein.
 * 3. Agar koi reject hua, toh poora reject — baaki results ignore.
 * 4. Agar input array empty hai [], toh turant resolve [].
 *
 * Example:
 *   Promise.all([p1, p2, p3])
 *   - p1 resolves to 10
 *   - p2 resolves to 20
 *   - p3 resolves to 30
 *   → Result: [10, 20, 30] (order maintained)
 *
 *   - Agar p2 reject ho gaya → Result: reject(p2 ka error)
 *
 * Syntax: Promise.all(promises) → returns a new Promise
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
MyPromise.all = function (promises) {
    return new MyPromise((resolve, reject) => {
        let results = [];
        let completed = 0;

        for (let i = 0; i < promises.length; i++) {
            /* Har promise ka then() call karo */
            promises[i].then(
                function (value) {
                    /* Value ko result array mein index pe store karo */
                    results[i] = value;
                    completed++;

                    /* Agar saare complete ho gaye, toh resolve karo */
                    if (completed === promises.length) {
                        resolve(results);
                    }
                },
                /* Agar koi reject hua, toh turant reject karo */
                reject
            );
        }
    });
};

/* Test 1: Sab promises resolve hote hain */
const p1 = new MyPromise((resolve) => setTimeout(() => resolve(10), 100));
const p2 = new MyPromise((resolve) => setTimeout(() => resolve(20), 200));
const p3 = new MyPromise((resolve) => setTimeout(() => resolve(30), 50));

MyPromise.all([p1, p2, p3]).then((val) => console.log(val));
// ✅ [10, 20, 30] — order maintained despite p3 resolving first

/**
 * Step-by-step flow:
 *
 * [p1(100ms), p2(200ms), p3(50ms)]
 *
 * Time 50ms:  p3 resolves → results[2] = 30, completed = 1
 * Time 100ms: p1 resolves → results[0] = 10, completed = 2
 * Time 200ms: p2 resolves → results[1] = 20, completed = 3
 *             completed === 3 (length) → resolve([10, 20, 30]) ✅
 *
 * Important: results[i] use kiya, results.push() nahi!
 * - Agar push karte, toh order bigad jata: [30, 10, 20]
 * - Index se store karne se order hamesha maintained rehta hai.
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 3 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Non-promise values (plain numbers, strings)
 * - Agar array mein koi promise nahi, balki direct value ho:
 *     Promise.all([1, 2, 3])
 * - 1.then() call karne ki koshish karega → TypeError.
 * - Kyunki 1 ek number hai, uspe then() nahi hota.
 *
 * - Native Promise.all non-promise values ko automatically
 *   Promise.resolve(value) se wrap kar deta hai.
 */

try {
    MyPromise.all([1, 2, 3]).then((val) => console.log(val));
} catch (e) {
    console.log(e.message); // ❌ promises[i].then is not a function
}

/**
 * FAILURE 2 — Empty array
 * - Agar input array khali ho:
 *     Promise.all([])
 * - Hamara polyfill: loop nahi chalega, completed = 0, length = 0.
 *   0 === 0 true hai, par loop nahi chala isliye resolve kabhi call nahi hoga.
 *   Promise hamesha PENDING mein rah jayega.
 * - Native Promise.all([]) turant resolve([]) karta hai.
 */

const emptyResult = MyPromise.all([]);
// ❌ Hamesha pending rahega — resolve kabhi call nahi hoga

/**
 * FAILURE 3 — Iterator vs Array
 * - Native Promise.all sirf arrays nahi, koi bhi iterable accept karta hai
 *   (Set, Map, generators, etc.)
 * - Hamara polyfill promises.length use karta hai, jo arrays pe kaam karta hai.
 * - Iterator pe length property nahi hoti.
 *
 * - Fix: Array.from(promises) se iterator ko array mein convert karo.
 */


In [ ]:
/**
 * Edge Cases Fix Karna
 */

/**
 * Interviewer: Non-promise values ko kaise handle karoge?
 *
 * You: Har value ko MyPromise.resolve() se wrap karenge.
 * - MyPromise.resolve(1) ek promise banata hai jo turant 1 se resolve hota hai.
 * - Agar value pehle se promise hai, toh waise hi use karta hai.
 *
 * Interviewer: Empty array ka issue?
 *
 * You: Loop se pehle check lagao — agar length 0 hai, toh turant resolve([]).
 *
 * Interviewer: Iterator ka issue?
 *
 * You: Array.from(promises) se koi bhi iterable ko array mein convert kar lenge.
 */

/* Step 2: Non-promise wrapping + empty array + iterator fix */
MyPromise.all = function (promises) {
    /* Edge Case 3: Iterator ko array mein convert karo */
    const promisesArray = Array.from(promises);

    return new MyPromise((resolve, reject) => {
        /* Edge Case 2: Empty array → turant resolve([]) */
        if (promisesArray.length === 0) {
            resolve([]);
            return;
        }

        let results = [];
        let completed = 0;

        for (let i = 0; i < promisesArray.length; i++) {
            /* Edge Case 1: Non-promise values ko wrap karo */
            MyPromise.resolve(promisesArray[i]).then(
                function (value) {
                    results[i] = value;
                    completed++;
                    if (completed === promisesArray.length) {
                        resolve(results);
                    }
                },
                reject
            );
        }
    });
};

/* Test 1: Non-promise values ab kaam karenge */
MyPromise.all([1, 2, 3]).then((val) => console.log(val)); // ✅ [1, 2, 3]

/* Test 2: Mixed promises and non-promises */
MyPromise.all([
    MyPromise.resolve(10),
    20,
    MyPromise.resolve(30)
]).then((val) => console.log(val)); // ✅ [10, 20, 30]

/* Test 3: Empty array ab turant resolve hoga */
MyPromise.all([]).then((val) => console.log(val)); // ✅ []

/* Test 4: Reject case abhi bhi kaam karega */
MyPromise.all([
    MyPromise.resolve(10),
    MyPromise.reject('Error!'),
    MyPromise.resolve(30)
]).then(
    (val) => console.log(val),
    (err) => console.log(err) // ✅ "Error!" — turant reject
);

/* MyPromise.resolve helper add karna mat bhoolna */
MyPromise.resolve = function (value) {
    if (value instanceof MyPromise) {
        return value;
    }
    return new MyPromise((resolve) => resolve(value));
};


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 3 edge cases fix ho gaye:
 * a. Non-promise values — MyPromise.resolve() se wrap ho rahe hain.
 * b. Empty array — turant resolve([]) ho raha hai.
 * c. Iterator — Array.from() se handle ho raha hai.
 *
 * Ab ek subtle issue bacha hai jo bahut kam log notice karte hain.
 */

/**
 * FAILURE — Results array ke holes
 * - Humne `results = []` banaya, aur `results[i] = value` karte hain.
 * - Agar promises alag time pe resolve hote hain, toh intermediate
 *   state mein results array mein HOLES ho sakte hain.
 *
 *   Example: [p1(100ms), p2(200ms), p3(50ms)]
 *   Time 50ms: results = [empty × 2, 30] — index 0 aur 1 khali hain
 *   Time 100ms: results = [10, empty, 30] — index 1 abhi khali hai
 *   Time 200ms: results = [10, 20, 30] — sab bhar gaye
 *
 * - Yeh technically kaam karta hai kyunki final result sahi hai.
 * - Lekin agar koi intermediate code results array check kare,
 *   toh holes dikhega.
 *
 * - Native Promise.all yeh issue nahi karta kyunki woh internally
 *   ek fixed-length array banata hai.
 *
 * - Fix: results array ko fixed length se initialize karo:
 *   results = new Array(promisesArray.length)
 *   Isse array mein pehle se sahi number of slots hote hain.
 */

/* Yeh ek minor optimization hai, but interviewer ko impress karega */


In [ ]:
/**
 * Edge Case 4 Fix Karna (Results Array Optimization)
 */

/**
 * Interviewer: Results array mein holes aa sakte hain, kya issue hai?
 *
 * You: Technically final result sahi aata hai, but intermediate
 *   state mein holes hote hain. Yeh best practice nahi hai.
 * - Fix: results ko fixed-length array se initialize karo:
 *   results = new Array(length)
 * - Isse array mein pehle se sahi number of slots hote hain.
 *   Holes hote hain but unka expected size pata hota hai.
 */

/* Step 3: Results array optimization */
MyPromise.all = function (promises) {
    const promisesArray = Array.from(promises);

    return new MyPromise((resolve, reject) => {
        if (promisesArray.length === 0) {
            resolve([]);
            return;
        }

        /* Edge Case 4: Fixed-length array banao */
        let results = new Array(promisesArray.length);
        let completed = 0;

        for (let i = 0; i < promisesArray.length; i++) {
            MyPromise.resolve(promisesArray[i]).then(
                function (value) {
                    results[i] = value;
                    completed++;
                    if (completed === promisesArray.length) {
                        resolve(results);
                    }
                },
                reject
            );
        }
    });
};

/* Test 1: Sab resolve — order maintained */
const p1 = new MyPromise((resolve) => setTimeout(() => resolve(10), 100));
const p2 = new MyPromise((resolve) => setTimeout(() => resolve(20), 200));
const p3 = new MyPromise((resolve) => setTimeout(() => resolve(30), 50));

MyPromise.all([p1, p2, p3]).then((val) => console.log(val)); // ✅ [10, 20, 30]

/* Test 2: Reject case */
MyPromise.all([
    MyPromise.resolve(10),
    MyPromise.reject('Failed!'),
    MyPromise.resolve(30)
]).then(
    (val) => console.log(val),
    (err) => console.log('Rejected:', err) // ✅ "Rejected: Failed!"
);

/* Test 3: Non-promise values */
MyPromise.all([1, 2, 3]).then((val) => console.log(val)); // ✅ [1, 2, 3]

/* Test 4: Empty array */
MyPromise.all([]).then((val) => console.log(val)); // ✅ []

/* Test 5: Single promise */
MyPromise.all([MyPromise.resolve(42)]).then((val) => console.log(val)); // ✅ [42]


In [ ]:
/**
 * Promise.all vs Promise.race vs Promise.allSettled — Detail Mein (Hinglish)
 */

/**
 * Promise.all kya hai?
 * - Array of promises leta hai.
 * - SAARE resolve hone pe resolve hota hai → result array milta hai.
 * - EK bhi reject hua → turant reject.
 * - "Sab pass honge tabhi pass, koi ek fail toh fail"
 *
 * Promise.race kya hai?
 * - Array of promises leta hai.
 * - JO BHI pehle resolve/reject ho, wahi result ho jata hai.
 * - "Pehla jeeta, baaki ignore"
 *
 * Promise.allSettled kya hai?
 * - Array of promises leta hai.
 * - HAMESHA resolve hota hai (kabhi reject nahi).
 * - Har result ek object hota hai:
 *   { status: 'fulfilled', value: 42 }
 *   { status: 'rejected', reason: 'Error' }
 * - "Sab ka result chahiye, pass ho ya fail"
 *
 * Comparison table:
 *
 * | Method      | Resolve When       | Reject When        | Result Format            |
 * |-------------|--------------------|--------------------|--------------------------|
 * | all         | Sab resolve        | Ek bhi reject      | [val1, val2, ...]        |
 * | race        | Pehla resolve      | Pehla reject       | single val/err           |
 * | allSettled  | Hamesha            | Never              | [{status, value/reason}] |
 */

/* Demo: Promise.all — sab pass */
MyPromise.all([
    MyPromise.resolve(1),
    MyPromise.resolve(2),
    MyPromise.resolve(3)
]).then((val) => console.log('all:', val)); // ✅ "all: [1, 2, 3]"

/* Demo: Promise.all — ek fail */
MyPromise.all([
    MyPromise.resolve(1),
    MyPromise.reject('Error!'),
    MyPromise.resolve(3)
]).then(
    (val) => console.log('all:', val),
    (err) => console.log('all rejected:', err) // ✅ "all rejected: Error!"
);

/* Demo: Promise.race — pehla jeeta */
const slow = new MyPromise((resolve) => setTimeout(() => resolve('slow'), 200));
const fast = new MyPromise((resolve) => setTimeout(() => resolve('fast'), 50));

MyPromise.race([slow, fast]).then((val) => console.log('race:', val));
// ✅ "race: fast" — fast pehle resolve hua

/**
 * Promise.all ka real-world use case:
 *
 * 1. Multiple API calls parallel karke saara data ek saath chahiye:
 *    const [user, posts, comments] = await Promise.all([
 *        fetchUser(),
 *        fetchPosts(),
 *        fetchComments()
 *    ]);
 *
 * 2. Images preload karna:
 *    await Promise.all(imageUrls.map(url => loadImage(url)));
 *
 * 3. Database queries parallel:
 *    const results = await Promise.all(queries.map(q => db.query(q)));
 */


In [ ]:
/**
 * Final Code:
 * 1. Array.from — iterator ko array mein convert karo
 * 2. Empty array — turant resolve([])
 * 3. MyPromise.resolve — non-promise values ko wrap karo
 * 4. results[i] = value — order maintain karo (push nahi karna!)
 * 5. completed counter — saare complete hone pe resolve
 * 6. reject direct pass — ek bhi reject pe turant reject
 *
 * Promise.all = "sab pass tabhi pass, ek fail toh fail"
 */

MyPromise.all = function (promises) {
  const promisesArray = Array.from(promises);

  return new MyPromise((resolve, reject) => {
    if (promisesArray.length === 0) {
      resolve([]);
      return;
    }

    let results = new Array(promisesArray.length);
    let completed = 0;

    for (let i = 0; i < promisesArray.length; i++) {
      MyPromise.resolve(promisesArray[i]).then(
        function (value) {
          results[i] = value;
          completed++;
          if (completed === promisesArray.length) {
            resolve(results);
          }
        },
        reject
      );
    }
  });
};


In [ ]:
/**
 * Promise.race Polyfill (Basic Version)
 */

/**
 * Interviewer: Promise.race ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain Promise.race kaise kaam karta hai:
 * - Promise.race ek array of promises leta hai.
 * - Yeh tab resolve/reject hota hai jab PehLA promise settle ho jaye.
 * - Jo promise sabse pehle complete hota hai (chahe resolve ho ya reject),
 *   wahi final result ban jata hai. Baaki sab ignore ho jate hain.
 *
 * Simple words mein: "Jo pehle aaya, wahi jeeta (race lagao)"
 *
 * Important points:
 * 1. Promise.race ek NAYA promise return karta hai.
 * 2. Pehla resolve hua toh race resolve, pehla reject hua toh race reject.
 * 3. Order matter nahi karta, speed (time) matter karta hai.
 * 4. Agar input array empty hai [], toh hamesha PENDING rahega (kabhi settle nahi hoga).
 *
 * Example:
 *   Promise.race([p1(100ms), p2(50ms), p3(200ms)])
 *   - p2 sabse pehle (50ms mein) resolve hua.
 *   → Result: p2 ka value. p1 aur p3 ignore.
 *
 * Syntax: Promise.race(promises) → returns a new Promise
 */

/* Step 1: Basic version — simple case, edge cases baad mein */
MyPromise.race = function (promises) {
    return new MyPromise((resolve, reject) => {
        for (let i = 0; i < promises.length; i++) {
            /* Har promise ka then() call karo */
            promises[i].then(
                /* Jo bhi pehle resolve hua, outer resolve ko call kar do */
                function (value) {
                    resolve(value);
                },
                /* Jo bhi pehle reject hua, outer reject ko call kar do */
                function (reason) {
                    reject(reason);
                }
            );
        }
    });
};

/* Test 1: Pehla resolve wala promise jeet gaya */
const p1 = new MyPromise((resolve) => setTimeout(() => resolve('Fast'), 50));
const p2 = new MyPromise((resolve) => setTimeout(() => resolve('Slow'), 200));

MyPromise.race([p1, p2]).then((val) => console.log(val));
// ✅ "Fast" — kyunki p1 50ms mein settle hua, p2 200ms mein

/* Test 2: Pehla reject wala promise jeet gaya */
const p3 = new MyPromise((_, reject) => setTimeout(() => reject('Error Fast'), 50));
const p4 = new MyPromise((resolve) => setTimeout(() => resolve('Success Slow'), 200));

MyPromise.race([p3, p4]).then(
    (val) => console.log(val),
    (err) => console.log(err)
);
// ✅ "Error Fast" — kyunki p3 pehle reject hua

/**
 * Step-by-step flow for Test 1:
 *
 * [p1(50ms), p2(200ms)]
 *
 * Time 0ms:   Loop chal raha hai. p1.then() aur p2.then() register ho gaye.
 * Time 50ms:  p1 resolve hua → resolve("Fast") call hua.
 *             Outer promise FULFILLED ho gaya.
 * Time 200ms: p2 resolve hua → resolve("Slow") call hua.
 *             Lekin outer promise pehle se FULFILLED hai, isliye ignore!
 *
 * Funda simple hai: Har promise apna then() chala raha hai.
 * Jo pehle `resolve(value)` ya `reject(reason)` call karega, outer promise
 * us value ke saath settle ho jayega. Kyunki ek baar state change hui,
 * wapas nahi badlegi.
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 3 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — Non-promise values (plain numbers, strings)
 * - Agar array mein koi direct value ho:
 *     Promise.race([1, 2, 3])
 * - 1.then() call karne ki koshish karega → TypeError.
 * - Kyunki 1 ek number hai, uspe then() nahi hota.
 *
 * - Native Promise.race non-promise values ko automatically
 *   Promise.resolve(value) se wrap kar deta hai.
 * - 1 turant resolve ho jayega aur race jeet jayega.
 */

try {
    MyPromise.race([1, 2, 3]).then((val) => console.log(val));
} catch (e) {
    console.log(e.message); // ❌ promises[i].then is not a function
}

/**
 * FAILURE 2 — Iterator vs Array
 * - Native Promise.race sirf arrays nahi, koi bhi iterable accept karta hai
 *   (Set, Map, generators, etc.)
 * - Hamara polyfill promises.length use karta hai, jo arrays pe kaam karta hai.
 * - Iterator pe length property nahi hoti.
 *
 * - Fix: Array.from(promises) se iterator ko array mein convert karo.
 */

/**
 * FAILURE 3 — Empty array
 * - Agar input array khali ho:
 *     Promise.race([])
 * - Hamara polyfill: loop nahi chalega, resolve/reject kabhi call nahi honge.
 *   Promise hamesha PENDING mein rah jayega.
 * - Yeh actually CORRECT behaviour hai! Native Promise.race([]) bhi hamesha
 *   pending rehta hai kyunki race mein koi participant hi nahi hai.
 * - Lekin is baare mein interviewer se baat karna zaroori hai taaki pata chale
 *   ki aapko yeh edge case pata hai.
 */

const emptyRace = MyPromise.race([]);
// ✅ Hamesha pending rahega (correct behaviour, but should be discussed)


In [ ]:
/**
 * Edge Cases Fix Karna
 */

/**
 * Interviewer: Non-promise values ko kaise handle karoge?
 *
 * You: Har value ko MyPromise.resolve() se wrap karenge.
 * - MyPromise.resolve(1) ek promise banata hai jo turant 1 se resolve hota hai.
 * - Agar value pehle se promise hai, toh waise hi use karta hai.
 *
 * Interviewer: Iterator ka issue?
 *
 * You: Array.from(promises) se koi bhi iterable ko array mein convert kar lenge.
 */

/* Step 2: Non-promise wrapping + iterator fix */
MyPromise.race = function (promises) {
    /* Edge Case 2: Iterator ko array mein convert karo */
    const promisesArray = Array.from(promises);

    return new MyPromise((resolve, reject) => {
        for (let i = 0; i < promisesArray.length; i++) {
            /* Edge Case 1: Non-promise values ko wrap karo */
            MyPromise.resolve(promisesArray[i]).then(
                function (value) {
                    resolve(value);
                },
                function (reason) {
                    reject(reason);
                }
            );
        }
    });
};

/* Test 1: Non-promise values ab kaam karenge */
MyPromise.race([1, 2, 3]).then((val) => console.log(val)); // ✅ 1 (pehla element jeet gaya)

/* Test 2: Mixed promises and non-promises */
const slowPromise = new MyPromise((resolve) => setTimeout(() => resolve('Slow'), 100));

MyPromise.race([
    slowPromise,
    'Instant'  // Non-promise value, turant resolve hoga
]).then((val) => console.log(val));
// ✅ "Instant" — kyunki non-promise value turant resolve hoti hai

/* Test 3: Iterator (Set) ab kaam karega */
const promiseSet = new Set([
    MyPromise.resolve('From Set 1'),
    MyPromise.resolve('From Set 2')
]);

MyPromise.race(promiseSet).then((val) => console.log(val));
// ✅ "From Set 1" (Set converted to array, pehla element jeeta)


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 2 edge cases fix ho gaye:
 * a. Non-promise values — MyPromise.resolve() se wrap ho rahe hain.
 * b. Iterator — Array.from() se handle ho raha hai.
 *
 * Promise.race ka logic bahut simple hai, isliye isme zyada issues nahi aate.
 * Lekin ek subtle point hai jo interviewer check kar sakta hai.
 */

/**
 * POTENTIAL ISSUE — Multiple resolve/reject calls
 * - Hamara code har promise pe then() laga raha hai.
 * - Jab p1 pehle resolve hua, toh outer resolve() call hua.
 * - Jab p2 baad mein resolve hua, toh outer resolve() FIRSE call hua.
 *
 * - Kya yeh issue hai?
 * - Answer: NAHI. Kyunki humara MyPromise resolve/reject ke andar
 *   `if (state === 'PENDING')` check lagata hai.
 * - Ek baar state change ho gayi, toh baaki calls silently ignore ho jate hain.
 *
 * - Lekin agar aap native Promise use kar rahe ho (MyPromise nahi), toh
 *   yahi logic kaam karega kyunki native Promise bhi state immutable rakhta hai.
 */

/* Demo: Multiple calls handle ho jate hain */
const p1 = new MyPromise((resolve) => setTimeout(() => resolve('First'), 50));
const p2 = new MyPromise((resolve) => setTimeout(() => resolve('Second'), 100));

const racePromise = MyPromise.race([p1, p2]);

racePromise.then((val) => console.log('Result:', val));
// ✅ "Result: First" — Second call ignore ho jayegi


In [ ]:
/**
 * Edge Case 3 Discussion (Empty Array)
 */

/**
 * Interviewer: Agar empty array pass karein toh kya hoga?
 *
 * You: Agar input array khali ho, toh loop nahi chalega.
 * - Koi resolve() ya reject() call nahi hoga.
 * - Promise hamesha PENDING state mein rah jayega.
 * - Yeh CORRECT behaviour hai! Native Promise.race([]) bhi same karta hai.
 *
 * Interviewer: Kyun pending rehta hai? Resolve ya reject kyun nahi hota?
 *
 * You: Kyunki race ka matlab hai "pehla aaya jeeta".
 *   Agar race mein koi participant hi nahi hai, toh koi winner nahi hoga.
 *   Isliye race kabhi khatam nahi hogi (pending hi rahegi).
 *
 * - Yeh Promise.all se alag hai.
 *   Promise.all([]) turant resolve([]) karta hai (kyunki "sab" pass ho gaye,
 *   0 out of 0 = 100%).
 *   Promise.race([]) pending rehta hai (kyunki koi racer hi nahi hai).
 */

/* Test: Empty array — hamesha pending */
const emptyRace = MyPromise.race([]);

emptyRace.then(
    (val) => console.log('Resolved:', val),
    (err) => console.log('Rejected:', err)
);
// ✅ Kuch print nahi hoga — promise hamesha pending rahega

/* Comparison: Promise.all([]) vs Promise.race([]) */
MyPromise.all([]).then((val) => console.log('all([]):', val));
// ✅ "all([]): []" — turant resolve

MyPromise.race([]).then((val) => console.log('race([]):', val));
// ✅ Kuch nahi — pending forever


In [ ]:
/**
 * Promise.race Real-world Use Cases — Detail Mein (Hinglish)
 */

/**
 * Promise.race kahan use hota hai?
 *
 * 1. Timeout implement karna:
 *    - Agar API call 5 second mein response na de, toh error do.
 *    - Race lagao API call aur setTimeout ke beech mein.
 *
 *    function fetchWithTimeout(url, ms) {
 *        const apiPromise = fetch(url);
 *        const timeoutPromise = new Promise((_, reject) => {
 *            setTimeout(() => reject('Timeout!'), ms);
 *        });
 *        return Promise.race([apiPromise, timeoutPromise]);
 *    }
 *
 *    - Agar API 5 sec se pehle ho gaya → API jeeta.
 *    - Agar 5 sec ho gaye → Timeout jeeta (reject).
 *
 * 2. Multiple sources se data fetch karna (jo sabse fast ho):
 *    - 3 alag servers se same data maango, jo pehle de, wahi use karo.
 *
 *    const fastestData = Promise.race([
 *        fetchFromServer1(),
 *        fetchFromServer2(),
 *        fetchFromServer3()
 *    ]);
 *
 * 3. User input cancel karna:
 *    - Agar user 3 sec mein form submit nahi karta, toh auto-cancel.
 */

/* Demo: Timeout implementation using race */
function fetchWithTimeout(url, ms) {
    const apiPromise = new MyPromise((resolve) => {
        setTimeout(() => resolve('Data from ' + url), 2000); // 2 sec
    });

    const timeoutPromise = new MyPromise((_, reject) => {
        setTimeout(() => reject('Timeout after ' + ms + 'ms'), ms);
    });

    return MyPromise.race([apiPromise, timeoutPromise]);
}

/* Test 1: API timeout se pehle ho gaya */
fetchWithTimeout('api.example.com', 3000).then(
    (val) => console.log('Success:', val), // ✅ "Success: Data from api.example.com"
    (err) => console.log('Error:', err)
);

/* Test 2: Timeout pehle ho gaya */
fetchWithTimeout('api.example.com', 1000).then(
    (val) => console.log('Success:', val),
    (err) => console.log('Error:', err) // ✅ "Error: Timeout after 1000ms"
);


In [ ]:
/**
 * Final Code:
 * 1. Array.from — iterator ko array mein convert karo
 * 2. MyPromise.resolve — non-promise values ko wrap karo
 * 3. Pehla resolve → outer resolve, pehla reject → outer reject
 * 4. Empty array → hamesha pending (kabhi settle nahi hoga)
 * 5. State immutability — ek baar settle hone pe baaki calls ignore
 *
 * Promise.race = "jo pehle aaya, wahi jeeta"
 */

MyPromise.race = function (promises) {
  const promisesArray = Array.from(promises);

  return new MyPromise((resolve, reject) => {
    for (let i = 0; i < promisesArray.length; i++) {
      MyPromise.resolve(promisesArray[i]).then(
        function (value) {
          resolve(value);
        },
        function (reason) {
          reject(reason);
        }
      );
    }
  });
};


In [ ]:
/**
 * Utility Polyfills:
 * 1. Debounce
 * 2. Throttle
 * 3. Curry
 * 4. Memoize
 * 5. Once
 * 6. Pipe
 * 7. Compose
 * 8. Chunk
*/

In [ ]:
/**
 * Debounce Polyfill (Basic Version)
 */

/**
 * Interviewer: Debounce function ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain Debounce kaise kaam karta hai:
 * - Debounce ek function ko temporarily "pause" kar deta hai.
 * - Jab user bahut tezi se koi action kare (jaise type karna ya resize karna),
 *   toh debounce us action ko ignore karta hai.
 * - Jab user STOP ho jata hai aur thoda time pass ho jata hai (wait time),
 *   tabhi main function chalta hai.
 *
 * Simple words mein: "Ruk ja, agar user thodi der tak kuch nahi karta, tabhi function chala."
 *
 * Real-world example:
 * - Search box mein user "JavaScript" type kar raha hai.
 * - Har letter pe API call karna expensive hai.
 * - Debounce lagate hain: agar user 500ms tak type nahi karta (ruk gaya),
 *   tabhi search API call karo.
 *
 * Important points:
 * 1. Debounce ek NAYA function return karta hai (wrapper).
 * 2. Isme ek 'wait' time hota hai (e.g., 500ms).
 * 3. Har naye call pe purana timer clear ho jata hai aur naya timer set hota hai.
 *
 * Syntax: const debouncedFn = debounce(actualFunc, waitTime)
 */

/* Step 1: Basic version — sirf timer clear karna */
function debounce(func, wait) {
    let timeoutId;

    /* Naya function return karo */
    return function () {
        /* Agar pehle se koi timer chal raha hai, toh use clear karo */
        clearTimeout(timeoutId);

        /* Naya timer set karo 'wait' milliseconds ke liye */
        timeoutId = setTimeout(function () {
            func(); // Asli function ko call karo
        }, wait);
    };
};

/* Test 1: Simple search simulation */
function searchQuery() {
    console.log('Searching...');
}

const debouncedSearch = debounce(searchQuery, 500);

/* Agar user tezi se type kare */
debouncedSearch(); // timer set (500ms)
debouncedSearch(); // purana timer clear, naya set (500ms)
debouncedSearch(); // purana timer clear, naya set (500ms)

/* User ruk gaya... 500ms baad "Searching..." print hoga */
// ✅ Output: "Searching..." (sirf ek baar, aakhri call ke 500ms baad)

/**
 * Step-by-step flow:
 *
 * Call 1: timeoutId = T1 (500ms)
 * Call 2: clearTimeout(T1) -> timeoutId = T2 (500ms)
 * Call 3: clearTimeout(T2) -> timeoutId = T3 (500ms)
 * (User ruk gaya)
 * 500ms baad: T3 chala, searchQuery() call hua.
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * lekin 2 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — 'this' context lost ho raha hai
 * - Agar hum object method ko debounce karein:
 *     const obj = { name: 'Aslam', greet() { console.log(this.name) } }
 *     const debouncedGreet = debounce(obj.greet, 100);
 *     debouncedGreet(); // undefined print hoga ❌
 *
 * - Kyunki jab setTimeout ke andar func() call hua, toh uska 'this'
 *   object nahi, balki global object (window) ban gaya.
 * - Humein 'this' ko preserve karna hoga.
 */

const obj = {
    name: 'Aslam',
    greet: function() {
        console.log('Hello, ' + this.name);
    }
};

const debouncedGreet = debounce(obj.greet, 100);
debouncedGreet(); // ❌ "Hello, undefined"

/**
 * FAILURE 2 — Arguments pass nahi ho rahe
 * - Agar hum function ko arguments pass karein:
 *     function sum(a, b) { console.log(a + b) }
 *     const debouncedSum = debounce(sum, 100);
 *     debouncedSum(2, 3); // NaN print hoga ❌
 *
 * - Kyunki setTimeout ke andar func() bina arguments ke call hua.
 * - Humein arguments bhi preserve karne honge.
 */

function sum(a, b) {
    console.log('Sum is: ' + (a + b));
}

const debouncedSum = debounce(sum, 100);
debouncedSum(2, 3); // ❌ "Sum is: NaN"


In [ ]:
/**
 * Edge Cases 1 & 2 Fix Karna (this context + arguments)
 */

/**
 * Interviewer: 'this' aur arguments kaise preserve karoge?
 *
 * You: Return hone wale function ke andar mujhe 'this' aur 'arguments'
 *   milte hain (kyunki woh normal function hai).
 * - Main inko closure mein capture karunga.
 * - Jab setTimeout chalega, tab main func.apply(this, arguments) use karunga.
 *   Isse original function ko sahi context aur arguments mil jayenge.
 */

/* Step 2: this context + arguments fix */
function debounce(func, wait) {
    let timeoutId;

    /* Arrow function use mat karna, warna 'arguments' nahi milega */
    return function () {
        /* 'this' aur 'arguments' ko capture karo (wrapper function ke) */
        const context = this;
        const args = arguments;

        clearTimeout(timeoutId);

        timeoutId = setTimeout(function () {
            /* Asli function ko sahi 'this' aur args ke saath call karo */
            func.apply(context, args);
        }, wait);
    };
};

/* Test 1: 'this' context ab sahi kaam karega */
const obj2 = {
    name: 'Aslam',
    greet: function() {
        console.log('Hello, ' + this.name);
    }
};

const debouncedGreet2 = debounce(obj2.greet, 100);
debouncedGreet2(); // ✅ "Hello, Aslam"

/* Test 2: Arguments ab sahi pass honge */
function sum2(a, b) {
    console.log('Sum is: ' + (a + b));
}

const debouncedSum2 = debounce(sum2, 100);
debouncedSum2(2, 3); // ✅ "Sum is: 5"

/* Test 3: Search with event-like argument */
function handleSearch(event) {
    console.log('Searching for: ' + event.target.value);
}

const debouncedSearch2 = debounce(handleSearch, 500);

/* Simulating an event object */
debouncedSearch2({ target: { value: 'JS' } });
// ✅ 500ms baad: "Searching for: JS"


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 'this' aur arguments fix ho gaye.
 * Ab ek advanced feature bacha hai jo production libraries (jaise lodash)
 * provide karti hain: "Immediate" ya "Leading Edge" execution.
 */

/**
 * FAILURE — Leading Edge (Immediate execution)
 * - Current behavior: Function hamesha "wait" time ke BAAD chalta hai (trailing edge).
 * - Lekin kabhi-kabhi hum chahte hain ki function pehli baar call pe TURANT chale,
 *   aur agar user wait time ke andar dobara call kare, toh woh ignore ho.
 * - Example: "Load More" button. Pehli click pe data load karo,
 *   agar user double-click kare, toh ignore karo.
 *
 * - Lodash mein iske liye { leading: true, trailing: false } jaisa option hota hai.
 * - Humein ek boolean flag add karna hoga: `immediate`.
 */

/* Basic behavior (trailing): type -> wait -> execute */
/* Immediate behavior (leading): execute -> wait (ignore subsequent calls) */


In [ ]:
/**
 * Edge Case 3 Fix Karna (Immediate/Leading Edge + Cancel)
 */

/**
 * Interviewer: Agar turant execution chahiye toh?
 *
 * You: Ek 'immediate' flag pass karunga.
 * - Agar immediate true hai, toh setTimeout ke andar function nahi chalega.
 * - Function turant chalega, aur timer agle calls ko block karega.
 * - Jab timer khatam hoga, tab wapas next call allowed hogi.
 *
 * - Cancel method bhi add kar deta hoon: taaki pending timer ko manually rok sakein.
 */

/* Step 3: Immediate flag + Cancel method */
function debounce(func, wait, immediate = false) {
    let timeoutId;

    const debounced = function () {
        const context = this;
        const args = arguments;

        clearTimeout(timeoutId);

        if (immediate) {
            /* Agar immediate true hai:
             * - Pehli call pe timeoutId null hoga, toh turant call karo.
             * - Baaki calls pe timeoutId set hoga, toh ignore karo.
             */
            const callNow = !timeoutId;
            
            /* Timer set karo taaki wait time tak next call block rahe.
             * Wait time ke baad timeoutId wapas null ho jayega. */
            timeoutId = setTimeout(function() {
                timeoutId = null;
            }, wait);

            if (callNow) {
                func.apply(context, args);
            }
        } else {
            /* Normal behavior (trailing edge) */
            timeoutId = setTimeout(function () {
                func.apply(context, args);
            }, wait);
        }
    };

    /* Cancel method add karo */
    debounced.cancel = function() {
        clearTimeout(timeoutId);
        timeoutId = null;
    };

    return debounced;
};

/* Test 1: Immediate execution */
function logClick() {
    console.log('Button clicked!');
}

const immediateClick = debounce(logClick, 500, true); // immediate = true

immediateClick(); // ✅ "Button clicked!" (turant)
immediateClick(); // ❌ Ignore (500ms ke andar hai)
immediateClick(); // ❌ Ignore

setTimeout(() => {
    immediateClick(); // ✅ "Button clicked!" (500ms baad, wapas allowed)
}, 600);

/* Test 2: Cancel method */
const normalSearch = debounce(logClick, 500);
normalSearch(); // Timer set
normalSearch.cancel(); // ✅ Timer clear, function kabhi nahi chalega


In [ ]:
/**
 * Debounce vs Throttle — Detail Mein (Hinglish)
 */

/**
 * Debounce:
 * - "Ruk ja, agar thodi der tak kuch nahi karta, tabhi chala."
 * - Action ko tab tak delay karta hai jab tak user stop na ho jaye.
 * - Har naye call pe timer RESET hota hai.
 *
 * Example: Search box typing.
 * User: J -> a -> v -> a -> (stop)
 * API call: -------------------------> (only after stop)
 *
 * Throttle:
 * - " Ek baar chala, ab agle X ms tak wait kar, chahe user kitni bhi bar kare."
 * - Action ko fixed time intervals pe chalata hai.
 * - Naye call timer reset nahi karte, bas ignore ho jate hain.
 *
 * Example: Window scroll event.
 * User: Scroll -> Scroll -> Scroll -> Scroll
 * Action: Run --------> Run --------> Run (every X ms)
 */

/* Demo: Real-world search implementation */
const searchInput = document.getElementById('search');

function fetchData(query) {
    console.log('Fetching data for: ' + query);
}

/* Debounced version: wait 300ms after typing stops */
const debouncedFetch = debounce(fetchData, 300);

/* Simulating typing */
searchInput.addEventListener('input', function(e) {
    debouncedFetch(e.target.value);
});

/**
 * Typing "JS":
 * J -> timer reset -> S -> timer reset -> (stop)
 * 300ms baad: "Fetching data for: JS" (sirf ek baar API call hui)
 */


In [ ]:
/**
 * Final Code:
 * 1. clearTimeout — naye call pe purana timer clear karo
 * 2. setTimeout — wait time ke baad function chalao
 * 3. this & arguments — context aur args preserve karo
 * 4. immediate flag — turant execution support
 * 5. cancel method — pending timer ko roko
 *
 * Debounce = "Ruk ja, agar user thodi der tak kuch nahi karta, tabhi chala"
 */

function debounce(func, wait, immediate = false) {
  let timeoutId;

  const debounced = function () {
    const context = this;
    const args = arguments;

    clearTimeout(timeoutId);

    if (immediate) {
      const callNow = !timeoutId;
      timeoutId = setTimeout(function () {
        timeoutId = null;
      }, wait);

      if (callNow) {
        func.apply(context, args);
      }
    } else {
      timeoutId = setTimeout(function () {
        func.apply(context, args);
      }, wait);
    }
  };

  debounced.cancel = function () {
    clearTimeout(timeoutId);
    timeoutId = null;
  };

  return debounced;
}

/**
 * Summary:
 * a. Debounce ka funda simple hai: jab user koi action repeatedly kare (jaise typing),
 *    toh har action pe function ko call karne ke bajaye, wait karo. Agar user thodi 
 *    der (wait time) tak us action ko repeat karna band kar de, tabhi main function 
 *    chalao. Har naye action pe purana wait timer reset ho jata hai.
 * b. this aur arguments ka funda: wrapper function ke andar this aur arguments capture
 *    kar lo, aur setTimeout ke callback mein func.apply(context, args) use karke 
 *    original function ko sahi context mein call karo.
 * c. Immediate (Leading Edge) ka funda: agar immediate = true ho, toh pehli call pe 
 *    function turant chalao, aur wait time tak aane wali baaki calls ko ignore karo. 
 *    Wait time ke baad wapas next pehli call allowed hogi.
 * d. Debounce vs Throttle ka difference: Debounce tab tak function nahi chalata jab 
 *    tak user stop na ho jaye (timer reset hota hai). Throttle function ko fixed time
 *    intervals pe chalata hai (timer reset nahi hota, bas ignore ho jate hain).
 * e. Real World Use Cases:
 *    Search box autocomplete, window resize events, form validation on typing.
*/

In [ ]:
/**
 * Throttle Polyfill (Basic Version)
 */

/**
 * Interviewer: Throttle function ka polyfill likh sakte ho?
 *
 * You: Haan, pehle samajhte hain Throttle kaise kaam karta hai:
 * - Throttle ek function ko ek fix time ke andar SIRF ek baar chalne deta hai.
 * - Jab user bahut tezi se koi action kare (jaise scroll karna ya button click karna),
 *   toh throttle function ko "limit" kar deta hai.
 * - Function turant chalta hai (pehli call pe), aur agle X ms tak koi call nahi manta.
 *
 * Simple words mein: "Ek baar chala diya, ab agle limit time tak ignore karo."
 *
 * Real-world example:
 * - Window scroll event. User scroll karta rehta hai.
 * - Har pixel pe function chalana expensive hai.
 * - Throttle lagate hain: function har 500ms mein sirf ek baar chalega,
 *   chahe user kitni bhi baar scroll kare.
 *
 * Important points:
 * 1. Throttle ek NAYA function return karta hai (wrapper).
 * 2. Isme ek 'wait' time hota hai (e.g., 500ms).
 * 3. Pehli call pe function turant chalta hai, phir wait time tak block ho jata hai.
 *
 * Syntax: const throttledFn = throttle(actualFunc, waitTime)
 * 
 * Note: startups m sirf ye likho, agar edge case handle karna bole sirf tabhi aage
 *       badho.
 */

/* Step 1: Basic version — sirf flag check karna */
function throttle(func, wait) {
    let isThrottled = false;

    /* Naya function return karo */
    return function () {
        /* Agar block mode mein nahi hai, toh function chala do */
        if (!isThrottled) {
            func(); // Asli function ko call karo
            isThrottled = true; // Block mode on karo

            /* Wait time baad block mode off karo */
            setTimeout(function () {
                isThrottled = false;
            }, wait);
        }
    };
};

/* Test 1: Simple scroll simulation */
function logScroll() {
    console.log('Scrolled!');
}

const throttledScroll = throttle(logScroll, 500);

/* Agar user tezi se scroll kare */
throttledScroll(); // ✅ "Scrolled!" (turant chala)
throttledScroll(); // ❌ Ignore (500ms block)
throttledScroll(); // ❌ Ignore (500ms block)

/* User 500ms baad dobara scroll kare */
setTimeout(() => {
    throttledScroll(); // ✅ "Scrolled!" (block hata, wapas chala)
}, 600);

/**
 * Step-by-step flow:
 *
 * Call 1: isThrottled = false -> func() chala, isThrottled = true -> timer set (500ms)
 * Call 2: isThrottled = true -> Ignore
 * Call 3: isThrottled = true -> Ignore
 * (500ms baad) : isThrottled = false (wapis ready)
 * Call 4: isThrottled = false -> func() chala, isThrottled = true -> timer set (500ms)
 */


In [ ]:
/**
 * Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Basic polyfill simple cases mein kaam karta hai,
 * laakin 2 situations mein break ho jata hai.
 * Ek-ek karke dekhte hain.
 */

/**
 * FAILURE 1 — 'this' context lost ho raha hai
 * - Agar hum object method ko throttle karein:
 *     const obj = { name: 'Aslam', greet() { console.log(this.name) } }
 *     const throttledGreet = throttle(obj.greet, 100);
 *     throttledGreet(); // undefined print hoga ❌
 *
 * - Kyunki jab func() call hua, toh uska 'this' object nahi, balki global object (window) ban gaya.
 * - Humein 'this' ko preserve karna hoga.
 */

const obj = {
    name: 'Aslam',
    greet: function() {
        console.log('Hello, ' + this.name);
    }
};

const throttledGreet = throttle(obj.greet, 100);
throttledGreet(); // ❌ "Hello, undefined"

/**
 * FAILURE 2 — Arguments pass nahi ho rahe
 * - Agar hum function ko arguments pass karein:
 *     function sum(a, b) { console.log(a + b) }
 *     const throttledSum = throttle(sum, 100);
 *     throttledSum(2, 3); // NaN print hoga ❌
 *
 * - Kyunki func() bina arguments ke call hua.
 * - Humein arguments bhi preserve karna honge.
 */

function sum(a, b) {
    console.log('Sum is: ' + (a + b));
}

const throttledSum = throttle(sum, 100);
throttledSum(2, 3); // ❌ "Sum is: NaN"


In [ ]:
/**
 * Edge Cases 1 & 2 Fix Karna (this context + arguments)
 */

/**
 * Interviewer: 'this' aur arguments kaise preserve karoge?
 *
 * You: Return hone wale function ke andar mujhe 'this' aur 'arguments'
 *   milte hain (kyunki woh normal function hai).
 * - Main inko closure mein capture karunga.
 * - Jab func() call karunga, tab main func.apply(this, arguments) use karunga.
 *   Isse original function ko sahi context aur arguments mil jayenge.
 */

/* Step 2: this context + arguments fix */
function throttle(func, wait) {
    let isThrottled = false;

    /* Arrow function use mat karna, warna 'arguments' nahi milega */
    return function () {
        if (!isThrottled) {
            /* Asli function ko sahi 'this' aur args ke saath call karo */
            func.apply(this, arguments);
            isThrottled = true;

            setTimeout(function () {
                isThrottled = false;
            }, wait);
        }
    };
};

/* Test 1: 'this' context ab sahi kaam karega */
const obj2 = {
    name: 'Aslam',
    greet: function() {
        console.log('Hello, ' + this.name);
    }
};

const throttledGreet2 = throttle(obj2.greet, 100);
throttledGreet2(); // ✅ "Hello, Aslam"

/* Test 2: Arguments ab sahi pass honge */
function sum2(a, b) {
    console.log('Sum is: ' + (a + b));
}

const throttledSum2 = throttle(sum2, 100);
throttledSum2(2, 3); // ✅ "Sa ki ch is: 5"


In [ ]:
/**
 * Abhi Bhi Kya Fail Ho Raha Hai, Aur Kyun?
 */

/**
 * Ab tak 'this' aur arguments fix ho gaye.
 * Ab ek advanced feature bacha hai jo production libraries (jaise lodash)
 * provide karti hain: "Trailing Call" (Pichli call ko preserve karna).
 */

/**
 * FAILURE — Trailing Call (Last call ka data loss)
 * - Current behavior: Function turant chalta hai (leading edge), phir block ho jata hai.
 * - Lekin agar user block period ke darmiyan (wait time ke andar) dobara call kare,
 *   toh hum us call ka data lose kar dete hain.
 * - Example: Button click karke page top pe jana. Agar user wait time ke andar dobara
 *   click kare, toh hum us event ko ignore karte hain. Lodash mein iske liye
 *   { trailing: true } option hota hai.
 *
 * - Lodash mein iske liye { leading: true, trailing: true } jaisa option hota hai.
 * - Humein block period ke andar aane wali calls ka "last call" preserve karna hoga,
 *   aur wait time khatam hone pe us "last call" ko execute karna hoga.
 */


In [ ]:
/**
 * Edge Case 3 Fix Karna (Trailing Call + Cancel)
 */

/**
 * Interviewer: Agar trailing call chahiye toh?
 *
 * You: Block period ke andar aane wali calls ko ignore mat karo.
 *   Uske arguments aur context ko save kar lo.
 *   Jab wait time khatam ho, toh agar koi saved call hai, toh usko execute kar do.
 *   Yeh ek recursive process ban jayega.
 *
 * - Cancel method bhi add karo: taaki pending timer aur trailing call ko manual clear kar sakein.
 */

/* Step 3: Trailing call + Cancel method */
function throttle(func, wait) {
    let isThrottled = false;
    let lastArgs = null;   /* Trailing call ke arguments save karne ke liye */
    let lastThis = null;   /* Trailing call ka context save karne ke liye */

    const throttled = function () {
        if (isThrottled) {
            /* Agar block mode mein hai, toh args aur context save kar lo (trailing) */
            lastArgs = arguments;
            lastThis = this;
            return;
        }

        /* Agar block mode mein nahi hai, toh function chala do (leading) */
        func.apply(this, arguments);
        isThrottled = true;

        setTimeout(function () {
            /* Wait time khatam hua, block mode off karo */
            isThrottled = false;

            /* Agar block period ke dauran koi call aayi thi (trailing call), toh usko execute karo */
            if (lastArgs) {
                /* Apne aap ko hi call karo saved args aur context ke saath */
                throttled.apply(lastThis, lastArgs);
                /* Saved data clear kar do */
                lastArgs = null;
                lastThis = null;
            }
        }, wait);
    };

    /* Cancel method add karo */
    throttled. cancel = function () {
        clearTimeout(timeoutId); // Note: yahan timeoutId define nahi hua, yeh demo hai
        isThrottled = false;
        lastArgs = null;
        lastThis = null;
    };

    return throttled;
};

/* Test 1: Leading + Trailing call */
function logClick(message) {
    console.log('Clicked: ' + message);
}

const myThrottle = throttle(logClick, 500);

myThrottle('First');  // ✅ "Clicked: First" (Leading - turant)
myThrottle('Second'); // ❌ Ignore (Block), but args saved (Trailing)
myThrottle('Third');  // ❌ Ignore (Block), last args overwrite ('Third' saved)

/* 500ms baad: "Clicked: Third" (Trailing - last saved call execute hua) */


In [ ]:
/**
 * Debounce vs Throttle — Detail Mein (Hinglish)
 */

/**
 * Debounce:
 * - "Ruk ja, agar thodi der tak kuch nahi karta, tabhi chala."
 * - Action ko tab tak delay karta hai jab tak user stop na ho jaye.
 * - Har naye call pe timer RESET hota hai.
 *
 * Example: Search box typing.
 * User: J -> a -> v -> a -> (stop)
 * API call: -------------------------> (only after stop)
 *
 * Throttle:
 * - "Ek baar chala, ab agle X ms tak wait kar, chahe user kitni bhi baar kare."
 * - Action ko fixed time intervals pe chalata hai.
 * - Naye call timer reset nahi karte, bas ignore ho jate hain (ya trailing ke liye save hote hain).
 *
 * Example: Window scroll event.
 * User: Scroll -> Scroll -> Scroll -> Scroll
 * Action: Run --------> Run --------> Run (every X ms)
 */

/* Demo: Real-world scroll implementation */
const handleScroll = debounce(fetchData, 300);

/* Simulating typing */
searchInput.addEventListener('input', function(e) {
    debouncedFetch(e.target.value);
});

/**
 * Typing "JS":
 * J -> timer reset -> S -> typon reset -> (stop)
 * 30ms baad: "Fetching data for: JS" (sirf ek baar API call hui)
 */


In [ ]:
/**
 * Final Code:
 * 1. isThrottled flag — block mode on/off karo
 * 2. leading edge — pehli call pe turant execute karo
 * 3. trailing call — block period ke andar aane wali last call ko save karo
 *  4. recursive call — trailing call ko execute karne ke liye function ko khud call karo
 * 5. cancel method — pending timer aur trailing call ko clear karo
 *
 * Throttle = "Ek baar chala diya, ab agle limit time tak ignore karo"
 */

function throttle(func, wait) {
    let isThrottled = false;
    let lastArgs = null;
    lastThis = null; // Bug: should be `let lastThis = null;`
    let timeoutId = null; // Added for cancel method

    const throttled = function () {
        if (isThrottled) {
            lastArgs = arguments;
            lastThis = this;
            return;
        }

        func.apply(this, arguments);
        isThrottled = true;

        timeoutId = setTimeout(function () {
            isThrottled = false;

            if (lastArgs) {
                throttled.apply(lastThis, lastArgs);
                lastArgs = null;
                lastThis = null;
                timeoutId = null; // Added for cancel method
            }
        }, wait);
    };

    throttled.cancel = function () {
        clearTimeout(timeoutId);
        isThrottled = false;
        lastArgs = null;
        lastThis = null; // Bug: should be `lastThis = null;`
    };

    return throttled;
}


In [ ]:
/**
 * Other Polyfills:
 * 1. EventEmitter
 * 2. LRU Cache
 * 3. groupBy
*/

In [ ]:
/**
 * Advanced Browser APIs:
 * 1. Proxy
 * 2. Reflect
 * 3. MutationObserver
 * 4. IntersectionObserver
 * 5. requestAnimationFrame
 * 6. Atomics
 * 7. AbortController
*/